<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">

# Stage 04 — Grounded LLM Answer Generation  
## ALLaM vs Qwen over the validated RAG layer

هذا النوتبوك يبني المرحلة الرابعة من مشروع:

**Arabic AI Voice Agent for Saudi Labor Law using RAG**

الغرض من هذه المرحلة ليس فقط توليد إجابة، بل توليد إجابة عربية **مقيدة بالسياق الرسمي المسترجع** ومقارنة نموذجين محليين:

1. **ALLaM-7B-Instruct-preview**
2. **Qwen2.5-7B-Instruct**

يعتمد النوت على مخرجات المرحلة الثالثة:
- أفضل إعداد Retrieval من `best_retrieval_config.json`
- قاعدة ChromaDB الدائمة
- ملفات Chunks من المرحلة الثانية
- عتبة الموثوقية من مرحلة Reliability Gate

</div>

<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">

## طريقة التشغيل الصحيحة

شغّل النوت بالترتيب.  
لا تبدأ بالخلايا الثقيلة الخاصة بتحميل النماذج إلا بعد أن تتأكد أن الخلايا التشخيصية الأولى نجحت.

الترتيب المقترح:
1. شغّل خلايا الإعداد والمسارات.
2. تأكد أن ملفات Stage 02 و Stage 03 ظهرت كلها ✅.
3. شغّل خلية Smoke Test للاسترجاع فقط.
4. بعدها شغّل نموذج واحد أولاً، ويفضل Qwen.
5. بعد نجاح Qwen، فعّل ALLaM للمقارنة.

</div>

In [1]:
# =========================================================
# Stage 04.1 - Environment, Imports, and Reproducibility
# =========================================================

from pathlib import Path
import os
import re
import json
import time
import math
import gc
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
except Exception:
    torch = None
    CUDA_AVAILABLE = False
    GPU_NAME = "CPU"

from IPython.display import display, HTML, Markdown

STYLE_CSS = '''
<style>
table.dataframe {
    border-collapse: collapse;
    font-size: 13px;
    width: 100%;
}
table.dataframe th {
    background-color: #1F4E78;
    color: white;
    padding: 7px 10px;
    text-align: center;
}
table.dataframe td {
    padding: 7px 10px;
    border: 1px solid #DDDDDD;
    text-align: center;
    vertical-align: top;
}
</style>
'''
display(HTML(STYLE_CSS))

print("CUDA available:", CUDA_AVAILABLE)
print("Device:", GPU_NAME)

CUDA available: True
Device: NVIDIA GeForce RTX 5090


In [2]:
# =========================================================
# Stage 04.2 - Project Paths
# عدّل PROJECT_DIR فقط إذا احتجت
# =========================================================

def find_project_dir():
    candidates = [
        Path.cwd() / "saudi_labor_law_voice_agent_project",
        Path.cwd().parent / "saudi_labor_law_voice_agent_project",
        Path.home() / "saudi_labor_law_voice_agent_project",
        Path(r"C:/Users/PC/Desktop/data collection code/saudi_labor_law_voice_agent_project"),
        Path(r"C:/Users/PC/Desktop/saudi_labor_law_voice_agent_project"),
    ]
    for c in candidates:
        if c.exists():
            return c
    return Path.cwd() / "saudi_labor_law_voice_agent_project"

PROJECT_DIR = find_project_dir()

CLEAN_DIR   = PROJECT_DIR / "02_clean"
FINAL_DIR   = PROJECT_DIR / "03_final"
CHUNKS_DIR  = PROJECT_DIR / "04_chunks"
REPORTS_DIR = PROJECT_DIR / "05_reports"
VECTOR_DIR  = PROJECT_DIR / "06_vector_db"
CACHE_DIR   = PROJECT_DIR / "07_embedding_cache"
STAGE03_DIR = PROJECT_DIR / "08_stage03_rag_results"
STAGE04_DIR = PROJECT_DIR / "09_stage04_llm_generation_results"

STAGE04_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("FINAL_DIR:", FINAL_DIR, FINAL_DIR.exists())
print("CHUNKS_DIR:", CHUNKS_DIR, CHUNKS_DIR.exists())
print("VECTOR_DIR:", VECTOR_DIR, VECTOR_DIR.exists())
print("STAGE03_DIR:", STAGE03_DIR, STAGE03_DIR.exists())
print("STAGE04_DIR:", STAGE04_DIR)

PROJECT_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project
FINAL_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final True
CHUNKS_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\04_chunks True
VECTOR_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\06_vector_db True
STAGE03_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results True
STAGE04_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\09_stage04_llm_generation_results


In [3]:
# =========================================================
# Stage 04.3 - Local Model Paths and Run Flags
# مسارات النماذج المحلية
# =========================================================

LOCAL_MODELS = {
    "qwen_7b": {
        "label": "Qwen2.5-7B-Instruct",
        "path": Path(r"C:/models/Qwen2.5-7B-Instruct"),
        "enabled": True,
    },
    "allam_7b": {
        "label": "ALLaM-7B-Instruct-preview",
        "path": Path(r"C:/models/ALLaM-7B-Instruct-preview"),
        "enabled": True,
    },
}

EMBEDDING_MODELS = {
    "e5_large": {
        "label": "multilingual-e5-large",
        "path": Path(r"C:/models/multilingual-e5-large"),
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
    "bge_m3": {
        "label": "bge-m3",
        "path": Path(r"C:/models/bge-m3"),
        "query_prefix": "",
        "passage_prefix": "",
    },
}

RERANKER_MODEL = {
    "label": "bge-reranker-v2-m3",
    "path": Path(r"C:/models/bge-reranker-v2-m3"),
}

# للتجربة الأولى، خليه صغير. بعد النجاح ارفعه إلى None لتشغيل كل الأسئلة.
MAX_EVAL_ROWS = None

RUN_QWEN = True
RUN_ALLAM = True

print("LLM models:")
for k, v in LOCAL_MODELS.items():
    print(k, "=>", v["path"], "| exists:", v["path"].exists(), "| enabled:", v["enabled"])

print("\nEmbedding models:")
for k, v in EMBEDDING_MODELS.items():
    print(k, "=>", v["path"], "| exists:", v["path"].exists())

print("\nReranker:", RERANKER_MODEL["path"], "| exists:", RERANKER_MODEL["path"].exists())

LLM models:
qwen_7b => C:\models\Qwen2.5-7B-Instruct | exists: True | enabled: True
allam_7b => C:\models\ALLaM-7B-Instruct-preview | exists: True | enabled: True

Embedding models:
e5_large => C:\models\multilingual-e5-large | exists: True
bge_m3 => C:\models\bge-m3 | exists: True

Reranker: C:\models\bge-reranker-v2-m3 | exists: True


In [4]:
# =========================================================
# Stage 04.4 - Load Stage 03 Best Retrieval Configuration
# =========================================================

def read_json_if_exists(path, default=None):
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

best_config_path = STAGE03_DIR / "best_retrieval_config.json"
ready_config_path = STAGE03_DIR / "stage03_best_retriever_ready_for_llm_config.json"

best_config = read_json_if_exists(best_config_path, default=None)

if best_config is None:
    ready_config = read_json_if_exists(ready_config_path, default={})
    best_config = ready_config.get("best_config", ready_config)

if not isinstance(best_config, dict) or len(best_config) == 0:
    raise FileNotFoundError(
        "لم أجد best_retrieval_config.json أو stage03_best_retriever_ready_for_llm_config.json. "
        "شغّل المرحلة الثالثة واحفظ مخرجاتها أولاً."
    )

BEST_CHUNKING_STRATEGY = str(best_config.get("chunking_strategy", "structural")).strip()
BEST_EMBEDDING_KEY = str(best_config.get("embedding_model", "bge_m3")).strip()
BEST_RETRIEVER = str(best_config.get("retriever", "hybrid_reranker")).strip()

try:
    BEST_ALPHA = float(best_config.get("alpha", 0.80))
except Exception:
    BEST_ALPHA = 0.80

TOP_K = int(best_config.get("top_k", 5)) if "top_k" in best_config else 5
CANDIDATE_K = int(best_config.get("candidate_k", 50)) if "candidate_k" in best_config else 50
RERANK_CANDIDATE_K = 20

print("Best retrieval config from Stage 03:")
print(json.dumps({
    "chunking_strategy": BEST_CHUNKING_STRATEGY,
    "embedding_model": BEST_EMBEDDING_KEY,
    "retriever": BEST_RETRIEVER,
    "alpha": BEST_ALPHA,
    "top_k": TOP_K,
    "candidate_k": CANDIDATE_K
}, ensure_ascii=False, indent=2))

display(pd.DataFrame([best_config]))

Best retrieval config from Stage 03:
{
  "chunking_strategy": "structural",
  "embedding_model": "e5_large",
  "retriever": "hybrid_reranker",
  "alpha": 0.8,
  "top_k": 5,
  "candidate_k": 50
}


,rank,chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec,selection_score,top_k,candidate_k
0,1,structural,e5_large,hybrid_reranker,0.8,155,0.9161,0.9871,0.9935,0.9508,0.9621,0.3361,0.972275,5,50


In [5]:
# =========================================================
# Stage 04.5 - Load Reliability Threshold
# =========================================================

threshold_candidates = [
    STAGE03_DIR / "reliability_gate_score_calibration.csv",
    STAGE03_DIR / "stage03_final_summary_table.csv",
]

RELIABILITY_THRESHOLD = 0.4983  # fallback from Stage 03 report

for p in threshold_candidates:
    if p.exists():
        try:
            tmp = pd.read_csv(p, encoding="utf-8-sig")
            possible_cols = [
                "suggested_reliability_threshold",
                "threshold",
                "recommended_threshold",
                "reliability_threshold"
            ]
            for col in possible_cols:
                if col in tmp.columns and tmp[col].notna().any():
                    RELIABILITY_THRESHOLD = float(tmp[col].dropna().iloc[0])
                    break
        except Exception:
            pass

for key in ["suggested_reliability_threshold", "reliability_threshold"]:
    if key in best_config and best_config.get(key) not in [None, ""]:
        try:
            RELIABILITY_THRESHOLD = float(best_config.get(key))
        except Exception:
            pass

print("RELIABILITY_THRESHOLD:", RELIABILITY_THRESHOLD)

RELIABILITY_THRESHOLD: 0.65


In [6]:
# =========================================================
# Stage 04.6 - Load Stage 02/03 Data Files
# Professional Tables Version - Fixed for New Pandas
# تحميل ملفات المرحلة الثانية والثالثة مع جداول احترافية
# =========================================================

from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Helper functions
# ---------------------------------------------------------

def read_csv_utf8(path: Path) -> pd.DataFrame:
    """
    Read CSV using UTF-8-SIG encoding to correctly support Arabic text.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path, encoding="utf-8-sig")


def apply_style_compatible(styler, func, subset):
    """
    Compatible styling for different pandas versions.
    New pandas uses Styler.map.
    Older pandas uses Styler.applymap.
    """
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def hide_index_safe(styler):
    """
    Hide dataframe index safely across pandas versions.
    """
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def short_folder(path: Path):
    """
    Display readable folder name instead of long full path.
    """
    path = Path(path)
    try:
        if "PROJECT_DIR" in globals():
            return str(path.parent.relative_to(PROJECT_DIR))
    except Exception:
        pass
    return path.parent.name


def short_id(text, left=12, right=8):
    """
    Shorten long IDs for display.
    """
    text = "" if pd.isna(text) else str(text)
    if len(text) <= left + right + 3:
        return text
    return text[:left] + "..." + text[-right:]


def short_text(text, max_len=260):
    """
    Shorten long Arabic/English text for readable table display.
    """
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def style_professional_table(
    df,
    caption="",
    number_formats=None,
    hide_index=True,
    width="100%",
    font_size="13px"
):
    """
    Professional notebook table styling for academic reporting.
    """
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="-")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    if hide_index:
        styler = hide_index_safe(styler)

    return styler


# ---------------------------------------------------------
# 1) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.6 — تحميل ملفات المرحلة الثانية والثالثة
</h2>

<p>
في هذه الخطوة يتم تحميل الملفات الأساسية الناتجة من المرحلتين الثانية والثالثة لاستخدامها في مرحلة توليد الإجابات بواسطة نماذج اللغة.
</p>

<p>
تعتمد هذه المرحلة على ثلاثة ملفات رئيسية: ملف أسئلة التقييم، ملف التقسيم الهيكلي، وملف التقسيم ثابت الحجم.
بعد ذلك يتم اختيار corpus المستخدم في المرحلة الرابعة بناءً على أفضل إعداد تم تحديده في المرحلة الثالثة.
</p>

</div>
"""))


# ---------------------------------------------------------
# 2) Define required files
# ---------------------------------------------------------

eval_path = FINAL_DIR / "rag_evaluation_dataset.csv"
structural_chunks_path = CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv"
fixed_chunks_path = CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv"

required_files = {
    "Evaluation Dataset": eval_path,
    "Structural Legal Chunks": structural_chunks_path,
    "Fixed-Size Chunks": fixed_chunks_path,
}


# ---------------------------------------------------------
# 3) Check file availability
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.2 — فحص توفر الملفات المطلوبة

يوضح هذا الجدول حالة الملفات المطلوبة للمرحلة الرابعة.  
إذا ظهرت الحالة <code>Available</code> فهذا يعني أن الملف موجود ويمكن تحميله، أما <code>Missing</code> فتعني أن الملف غير موجود ويجب إعادة تشغيل المرحلة التي أنتجته.

</div>
"""))

file_check_rows = []
missing = []

for display_name, path in required_files.items():
    path = Path(path)
    exists = path.exists()
    size_mb = path.stat().st_size / (1024 ** 2) if exists else np.nan

    file_check_rows.append({
        "File Role": display_name,
        "Status": "Available" if exists else "Missing",
        "File Name": path.name,
        "Folder": short_folder(path),
        "Size (MB)": size_mb,
    })

    if not exists:
        missing.append(str(path))

df_file_check = pd.DataFrame(file_check_rows)


def color_status(val):
    if val == "Available":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if val == "Missing":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


file_check_style = style_professional_table(
    df_file_check,
    caption="Table 4.2 — Stage 04 Required Input Files",
    number_formats={"Size (MB)": "{:.3f}"}
)

file_check_style = apply_style_compatible(
    file_check_style,
    color_status,
    subset=["Status"]
)

display(file_check_style)

if missing:
    raise FileNotFoundError(
        "Missing required files. Re-run Stage 02/03 first: " + str(missing)
    )


# ---------------------------------------------------------
# 4) Load required files
# ---------------------------------------------------------

df_eval = read_csv_utf8(eval_path)
df_structural_chunks = read_csv_utf8(structural_chunks_path)
df_fixed_chunks = read_csv_utf8(fixed_chunks_path)

if BEST_CHUNKING_STRATEGY.lower().startswith("fixed"):
    df_chunks = df_fixed_chunks.copy()
    selected_chunks_name = "fixed_size"
else:
    df_chunks = df_structural_chunks.copy()
    selected_chunks_name = "structural"


# ---------------------------------------------------------
# 5) Professional loading summary table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.3 — ملخص البيانات المحملة

يعرض هذا الجدول عدد الصفوف والأعمدة في كل ملف تم تحميله.  
كما يوضح أي corpus سيتم استخدامه فعليًا في المرحلة الرابعة قبل تمريره إلى طبقة الاسترجاع ثم نموذج اللغة.

</div>
"""))

loading_summary = pd.DataFrame([
    {
        "Dataset": "Evaluation Dataset",
        "Rows": len(df_eval),
        "Columns": df_eval.shape[1],
        "Role in Stage 04": "Questions used to evaluate grounded LLM generation",
        "Selected for LLM": "No"
    },
    {
        "Dataset": "Structural Legal Chunks",
        "Rows": len(df_structural_chunks),
        "Columns": df_structural_chunks.shape[1],
        "Role in Stage 04": "Retrieval corpus preserving legal and FAQ structure",
        "Selected for LLM": "Yes" if selected_chunks_name == "structural" else "No"
    },
    {
        "Dataset": "Fixed-Size Chunks",
        "Rows": len(df_fixed_chunks),
        "Columns": df_fixed_chunks.shape[1],
        "Role in Stage 04": "Alternative chunking baseline from Stage 02",
        "Selected for LLM": "Yes" if selected_chunks_name == "fixed_size" else "No"
    },
    {
        "Dataset": "Active Stage 04 Corpus",
        "Rows": len(df_chunks),
        "Columns": df_chunks.shape[1],
        "Role in Stage 04": f"Selected according to Stage 03 best configuration: {BEST_CHUNKING_STRATEGY}",
        "Selected for LLM": "Yes"
    }
])


def color_selected(val):
    if val == "Yes":
        return "background-color:#D9EAF7; color:#1F4E78; font-weight:bold;"
    if val == "No":
        return "background-color:#F2F2F2; color:#666666;"
    return ""


loading_summary_style = style_professional_table(
    loading_summary,
    caption="Table 4.3 — Loaded Data Assets for Stage 04 LLM Generation"
)

loading_summary_style = apply_style_compatible(
    loading_summary_style,
    color_selected,
    subset=["Selected for LLM"]
)

display(loading_summary_style)


# ---------------------------------------------------------
# 6) Evaluation type distribution table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.4 — توزيع أسئلة التقييم حسب النوع

يوضح هذا الجدول أنواع الأسئلة الموجودة في مجموعة التقييم.  
هذا مهم لأن أسئلة FAQ لا تُقيّم بنفس طريقة الأسئلة القانونية المرتبطة برقم مادة، والأسئلة خارج النطاق يجب أن يتم رفضها بشكل آمن.

</div>
"""))

if "eval_type" in df_eval.columns:
    eval_type_distribution = (
        df_eval["eval_type"]
        .fillna("unknown")
        .value_counts(dropna=False)
        .rename_axis("Evaluation Type")
        .reset_index(name="Number of Questions")
    )

    eval_type_distribution["Percentage"] = (
        eval_type_distribution["Number of Questions"] /
        eval_type_distribution["Number of Questions"].sum()
    )

    display(
        style_professional_table(
            eval_type_distribution,
            caption="Table 4.4 — Evaluation Dataset Distribution by Question Type",
            number_formats={"Percentage": "{:.2%}"},
            width="70%"
        )
    )
else:
    display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#FFF2CC;
    padding:12px;
    border-right:5px solid #D6B656;
    border-radius:6px;
">
⚠️ لا يوجد عمود <code>eval_type</code> في ملف التقييم.
</div>
"""))


# ---------------------------------------------------------
# 7) Compact preview of selected chunks
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.5 — معاينة من corpus المختار للمرحلة الرابعة

يعرض هذا الجدول عينة مختصرة من corpus الذي سيتم استخدامه في المرحلة الرابعة.  
تم اختصار معرف المستند والنص الطويل حتى يكون الجدول واضحًا وقابلًا للقراءة داخل النوتبوك.

</div>
"""))

preview_cols = [
    "chunk_id",
    "source_type",
    "article_number_int",
    "parent_document_id",
    "chunk_text"
]

available_preview_cols = [c for c in preview_cols if c in df_chunks.columns]
df_chunks_preview = df_chunks[available_preview_cols].head(5).copy()

df_chunks_preview = df_chunks_preview.rename(columns={
    "chunk_id": "Chunk ID",
    "source_type": "Source Type",
    "article_number_int": "Article No.",
    "parent_document_id": "Parent Doc ID",
    "chunk_text": "Text Preview"
})

if "Article No." in df_chunks_preview.columns:
    df_chunks_preview["Article No."] = (
        pd.to_numeric(df_chunks_preview["Article No."], errors="coerce")
        .astype("Int64")
    )

if "Parent Doc ID" in df_chunks_preview.columns:
    df_chunks_preview["Parent Doc ID"] = df_chunks_preview["Parent Doc ID"].apply(
        lambda x: short_id(x, 12, 8)
    )

if "Text Preview" in df_chunks_preview.columns:
    df_chunks_preview["Text Preview"] = df_chunks_preview["Text Preview"].apply(
        lambda x: short_text(x, 320)
    )

preview_style = (
    df_chunks_preview.style
    .set_caption(f"Table 4.5 — Preview of Selected Stage 04 Corpus ({selected_chunks_name})")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px"),
                ("white-space", "nowrap")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "7px"),
                ("vertical-align", "middle"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [("background-color", "#F8FBFD")]
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [("background-color", "#FFFFFF")]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        },
        {"selector": "th.col0", "props": [("width", "12%")]},
        {"selector": "th.col1", "props": [("width", "11%")]},
        {"selector": "th.col2", "props": [("width", "8%")]},
        {"selector": "th.col3", "props": [("width", "16%")]},
        {"selector": "th.col4", "props": [("width", "53%")]}
    ])
)

if "Text Preview" in df_chunks_preview.columns:
    preview_style = preview_style.set_properties(
        subset=["Text Preview"],
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "line-height": "1.8"
        }
    )

center_cols = [c for c in df_chunks_preview.columns if c != "Text Preview"]
if center_cols:
    preview_style = preview_style.set_properties(
        subset=center_cols,
        **{
            "text-align": "center",
            "white-space": "normal"
        }
    )

preview_style = hide_index_safe(preview_style)
display(preview_style)


# ---------------------------------------------------------
# 8) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم تحميل بيانات المرحلة الرابعة بنجاح.</b><br>

تم اختيار corpus المستخدم في هذه المرحلة بناءً على أفضل إعداد من المرحلة الثالثة:<br>

<b>Selected Corpus:</b> <code>{selected_chunks_name}</code><br>
<b>Rows:</b> <code>{df_chunks.shape[0]}</code><br>
<b>Columns:</b> <code>{df_chunks.shape[1]}</code><br>

أصبحت البيانات الآن جاهزة للانتقال إلى خطوة توحيد الأعمدة وبناء طبقة الاسترجاع.

</div>
"""))

print(
    f"Stage 04 corpus selected successfully: {selected_chunks_name} "
    f"({df_chunks.shape[0]} rows × {df_chunks.shape[1]} columns)."
)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.6 — تحميل ملفات المرحلة الثانية والثالثة
</h2>

<p>
في هذه الخطوة يتم تحميل الملفات الأساسية الناتجة من المرحلتين الثانية والثالثة لاستخدامها في مرحلة توليد الإجابات بواسطة نماذج اللغة.
</p>

<p>
تعتمد هذه المرحلة على ثلاثة ملفات رئيسية: ملف أسئلة التقييم، ملف التقسيم الهيكلي، وملف التقسيم ثابت الحجم.
بعد ذلك يتم اختيار corpus المستخدم في المرحلة الرابعة بناءً على أفضل إعداد تم تحديده في المرحلة الثالثة.
</p>

</div>



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.2 — فحص توفر الملفات المطلوبة

يوضح هذا الجدول حالة الملفات المطلوبة للمرحلة الرابعة.  
إذا ظهرت الحالة <code>Available</code> فهذا يعني أن الملف موجود ويمكن تحميله، أما <code>Missing</code> فتعني أن الملف غير موجود ويجب إعادة تشغيل المرحلة التي أنتجته.

</div>


File Role,Status,File Name,Folder,Size (MB)
Evaluation Dataset,Available,rag_evaluation_dataset.csv,03_final,0.183
Structural Legal Chunks,Available,rag_chunks_structural_legal_experiment.csv,04_chunks,0.869
Fixed-Size Chunks,Available,rag_chunks_fixed_size_experiment.csv,04_chunks,0.843



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.3 — ملخص البيانات المحملة

يعرض هذا الجدول عدد الصفوف والأعمدة في كل ملف تم تحميله.  
كما يوضح أي corpus سيتم استخدامه فعليًا في المرحلة الرابعة قبل تمريره إلى طبقة الاسترجاع ثم نموذج اللغة.

</div>


Dataset,Rows,Columns,Role in Stage 04,Selected for LLM
Evaluation Dataset,160,24,Questions used to evaluate grounded LLM generation,No
Structural Legal Chunks,490,28,Retrieval corpus preserving legal and FAQ structure,Yes
Fixed-Size Chunks,490,28,Alternative chunking baseline from Stage 02,No
Active Stage 04 Corpus,490,28,Selected according to Stage 03 best configuration: structural,Yes


...

Evaluation Type,Number of Questions,Percentage
faq_retrieval,121,75.62%
legal_article_retrieval,24,15.00%
out_of_scope,15,9.38%



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.5 — معاينة من corpus المختار للمرحلة الرابعة

يعرض هذا الجدول عينة مختصرة من corpus الذي سيتم استخدامه في المرحلة الرابعة.  
تم اختصار معرف المستند والنص الطويل حتى يكون الجدول واضحًا وقابلًا للقراءة داخل النوتبوك.

</div>


Chunk ID,Source Type,Article No.,Parent Doc ID,Text Preview
STRUCT_000001,legal_article,1,d28fed1c707f...198af462,الباب الأول - التعريفات / الأحكام العامة | الفصل الأول | المادة 1 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الأولى: رقم المادة: 1 حالة المادة: active النص: يسمى هذا النظام نظام العمل.
STRUCT_000002,legal_article,2,4f2961046b5f...f17df638,الباب الأول - التعريفات / الأحكام العامة | الفصل الأول | المادة 2 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الثانية : رقم المادة: 2 حالة المادة: active النص: يقصد بالألفاظ والعبارات الآتية – أينما وردت في هذا النظام – المعاني المبينة أمامها ما لم يقتض السياق خلاف ذلك: -الوزا...
STRUCT_000003,legal_article,3,82f3c9791caa...24a31fda,الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 3 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الثالثة: رقم المادة: 3 حالة المادة: active النص: العمل حق للمواطن ، لا يجوز لغيره ممارسته إلا بعد توافر الشروط المنصوص عليها في هذا النظام ، والمواطنون متساوون في حق...
STRUCT_000004,legal_article,4,3457807712bc...6ca95090,الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 4 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الرابعة: رقم المادة: 4 حالة المادة: active النص: يجب على صاحب العمل والعامل عند تطبيق أحكام هذا النظام الإلتزام بمقتضيات أحكام الشريعة الإسلامية .
STRUCT_000005,legal_article,5,a03900ffe9b5...c8bde4cd,الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 5 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الخامسة: رقم المادة: 5 حالة المادة: active النص: تسري أحكام هذا النظام على الآتي: كل عقد عمل يلتزم بمقتضاه أي شخص بالعمل لمصلحة صاحب عمل وتحت إدارته أو إشرافه ؛ مقاب...



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم تحميل بيانات المرحلة الرابعة بنجاح.</b><br>

تم اختيار corpus المستخدم في هذه المرحلة بناءً على أفضل إعداد من المرحلة الثالثة:<br>

<b>Selected Corpus:</b> <code>structural</code><br>
<b>Rows:</b> <code>490</code><br>
<b>Columns:</b> <code>28</code><br>

أصبحت البيانات الآن جاهزة للانتقال إلى خطوة توحيد الأعمدة وبناء طبقة الاسترجاع.

</div>


Stage 04 corpus selected successfully: structural (490 rows × 28 columns).


In [7]:
# =========================================================
# Stage 04.7 - Schema Standardization and Evaluation Preview
# توحيد بنية البيانات وعرض عينة التقييم بشكل احترافي
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Helper functions
# ---------------------------------------------------------

def ensure_col(df, col, default=""):
    """
    Ensure that a column exists. If missing, create it with a default value.
    """
    if col not in df.columns:
        df[col] = default
    return df


def short_text(text, max_len=180):
    """
    Shorten long text for readable table display.
    """
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def short_id(text, left=10, right=6):
    """
    Shorten long IDs while keeping beginning and ending.
    """
    text = "" if pd.isna(text) else str(text)
    if len(text) <= left + right + 3:
        return text
    return text[:left] + "..." + text[-right:]


def clean_na(value):
    """
    Convert empty or NaN values to N/A for display.
    """
    if pd.isna(value):
        return "N/A"
    value = str(value).strip()
    if value.lower() in ["", "nan", "none"]:
        return "N/A"
    return value


def hide_index_safe(styler):
    """
    Hide index safely across pandas versions.
    """
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    """
    Compatible style application for old/new pandas versions.
    """
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="13px", number_formats=None):
    """
    Professional academic table style.
    """
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


# ---------------------------------------------------------
# 1) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.7 — توحيد بنية البيانات قبل التقييم
</h2>

<p>
في هذه الخطوة يتم توحيد أعمدة بيانات التقييم وبيانات المقاطع المختارة، والتأكد من وجود الأعمدة الأساسية المطلوبة في المرحلة الرابعة.
بدل عرض كل الأعمدة الخام، يتم عرض ملخصات وجداول مختصرة توضح حجم البيانات، وتوزيع الأسئلة، وعينة منظمة من أسئلة التقييم.
</p>

<p>
هذه الخطوة مهمة لأنها تجهز البيانات قبل تشغيل الاسترجاع ونماذج اللغة، وتمنع مشاكل لاحقة بسبب اختلاف أسماء الأعمدة أو القيم المفقودة.
</p>

</div>
"""))


# ---------------------------------------------------------
# 2) Standardize chunk columns
# ---------------------------------------------------------

required_chunk_cols = ["chunk_id", "chunk_text", "parent_document_id", "source_type"]

for col in required_chunk_cols:
    if col not in df_chunks.columns:
        raise ValueError(f"Required chunk column missing: {col}")

chunk_defaults = {
    "source_name": "",
    "legal_category": "",
    "bab_title": "",
    "chapter_title": "",
    "article_number": "",
    "article_number_label": "",
    "article_number_int": "",
    "article_title": "",
    "article_status": "",
    "question": "",
    "answer": "",
    "source_url": "",
    "rag_usage": "",
}

for col, default in chunk_defaults.items():
    df_chunks = ensure_col(df_chunks, col, default)

df_chunks["chunk_id"] = df_chunks["chunk_id"].fillna("").astype(str)
df_chunks["chunk_text"] = df_chunks["chunk_text"].fillna("").astype(str)
df_chunks["parent_document_id"] = df_chunks["parent_document_id"].fillna("").astype(str)
df_chunks["source_type"] = df_chunks["source_type"].fillna("").astype(str)


# ---------------------------------------------------------
# 3) Standardize evaluation columns
# ---------------------------------------------------------

eval_defaults = {
    "eval_id": "",
    "question": "",
    "expected_answer": "",
    "eval_type": "unknown",
    "gold_article_numbers": "",
    "gold_article_number": "",
    "gold_parent_document_id": "",
    "gold_source_type": "",
    "is_out_of_scope": False,
}

for col, default in eval_defaults.items():
    df_eval = ensure_col(df_eval, col, default)

df_eval["question"] = df_eval["question"].fillna("").astype(str)
df_eval["expected_answer"] = df_eval["expected_answer"].fillna("").astype(str)
df_eval["eval_type"] = df_eval["eval_type"].fillna("unknown").astype(str)

if MAX_EVAL_ROWS is not None:
    df_eval_run = df_eval.head(int(MAX_EVAL_ROWS)).copy()
else:
    df_eval_run = df_eval.copy()


# ---------------------------------------------------------
# 4) Data object summary table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.6 — ملخص كائنات البيانات بعد التوحيد

يوضح هذا الجدول حجم بيانات التقييم الكاملة، والعينة المستخدمة في التشغيل الحالي، وبيانات corpus المختار للمرحلة الرابعة.

</div>
"""))

schema_summary = pd.DataFrame([
    {
        "Data Object": "df_eval",
        "Description": "Full evaluation dataset",
        "Rows": df_eval.shape[0],
        "Columns": df_eval.shape[1],
        "Used in Current Run": "No" if MAX_EVAL_ROWS is not None else "Yes"
    },
    {
        "Data Object": "df_eval_run",
        "Description": "Evaluation subset used in the current run",
        "Rows": df_eval_run.shape[0],
        "Columns": df_eval_run.shape[1],
        "Used in Current Run": "Yes"
    },
    {
        "Data Object": "df_chunks",
        "Description": "Selected retrieval corpus for Stage 04",
        "Rows": df_chunks.shape[0],
        "Columns": df_chunks.shape[1],
        "Used in Current Run": "Yes"
    }
])

def color_used(val):
    if val == "Yes":
        return "background-color:#D9EAF7; color:#1F4E78; font-weight:bold;"
    if val == "No":
        return "background-color:#F2F2F2; color:#666666;"
    return ""

schema_style = professional_table(
    schema_summary,
    caption="Table 4.6 — Standardized Data Objects for Stage 04"
)

schema_style = apply_style_compatible(schema_style, color_used, subset=["Used in Current Run"])
display(schema_style)


# ---------------------------------------------------------
# 5) Evaluation sample distribution
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.7 — توزيع عينة التقييم حسب نوع السؤال

يعرض هذا الجدول توزيع الأسئلة المستخدمة في التشغيل الحالي حسب نوع التقييم.
هذا مهم لأن أسئلة FAQ والأسئلة القانونية والأسئلة خارج النطاق لا تُقيّم بنفس الطريقة.

</div>
"""))

eval_sample_distribution = (
    df_eval_run["eval_type"]
    .fillna("unknown")
    .value_counts(dropna=False)
    .rename_axis("Evaluation Type")
    .reset_index(name="Number of Questions")
)

eval_sample_distribution["Percentage"] = (
    eval_sample_distribution["Number of Questions"] /
    eval_sample_distribution["Number of Questions"].sum()
)

display(
    professional_table(
        eval_sample_distribution,
        caption="Table 4.7 — Evaluation Sample Distribution by Type",
        width="70%",
        number_formats={"Percentage": "{:.2%}"}
    )
)


# ---------------------------------------------------------
# 6) Gold article availability summary
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.8 — توفر أرقام المواد الذهبية في عينة التقييم

يوضح هذا الجدول عدد الأسئلة التي تحتوي على رقم مادة ذهبية داخل عينة التقييم.
عدم وجود رقم مادة في أسئلة FAQ أو الأسئلة خارج النطاق أمر طبيعي وليس خطأ في البيانات.

</div>
"""))

gold_cols = [c for c in ["gold_article_numbers", "gold_article_number"] if c in df_eval_run.columns]

if gold_cols:
    gold_available = df_eval_run[gold_cols].fillna("").astype(str).agg(" ".join, axis=1)
    gold_available = gold_available.str.strip().replace("nan", "", regex=False)
    has_gold = gold_available.apply(lambda x: bool(str(x).strip()))
else:
    has_gold = pd.Series([False] * len(df_eval_run))

gold_summary = pd.DataFrame([
    {
        "Metric": "Questions in Current Run",
        "Count": len(df_eval_run),
        "Interpretation": "Total questions used in this Stage 04 run"
    },
    {
        "Metric": "Questions with Gold Article Number",
        "Count": int(has_gold.sum()),
        "Interpretation": "Questions that can be evaluated using legal article citation"
    },
    {
        "Metric": "Questions without Gold Article Number",
        "Count": int((~has_gold).sum()),
        "Interpretation": "Usually FAQ, small talk, or out-of-scope questions"
    }
])

display(
    professional_table(
        gold_summary,
        caption="Table 4.8 — Gold Article Availability in Evaluation Sample",
        width="90%"
    )
)


# ---------------------------------------------------------
# 7) Professional evaluation preview table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.9 — معاينة منظمة من أسئلة التقييم

يعرض هذا الجدول أهم الأعمدة المطلوبة في المرحلة الرابعة فقط، مع اختصار النصوص الطويلة والمعرفات حتى يكون الجدول قابلًا للقراءة.
هذا الجدول بديل عن عرض <code>df_eval_run.head(5)</code> الخام الذي يكون عريضًا وغير مناسب للتقرير.

</div>
"""))

preview_cols = [
    "eval_id",
    "question",
    "expected_answer",
    "eval_type",
    "gold_article_numbers",
    "gold_article_number",
    "gold_parent_document_id",
    "is_out_of_scope"
]

available_preview_cols = [c for c in preview_cols if c in df_eval_run.columns]
df_eval_preview = df_eval_run[available_preview_cols].head(8).copy()

df_eval_preview = df_eval_preview.rename(columns={
    "eval_id": "Eval ID",
    "question": "Question",
    "expected_answer": "Expected Answer",
    "eval_type": "Evaluation Type",
    "gold_article_numbers": "Gold Articles",
    "gold_article_number": "Gold Article",
    "gold_parent_document_id": "Gold Parent ID",
    "is_out_of_scope": "Out of Scope"
})

for col in ["Question"]:
    if col in df_eval_preview.columns:
        df_eval_preview[col] = df_eval_preview[col].apply(lambda x: short_text(x, 120))

for col in ["Expected Answer"]:
    if col in df_eval_preview.columns:
        df_eval_preview[col] = df_eval_preview[col].apply(lambda x: short_text(x, 180))

for col in ["Gold Parent ID"]:
    if col in df_eval_preview.columns:
        df_eval_preview[col] = df_eval_preview[col].apply(lambda x: short_id(x, 10, 6))

for col in ["Gold Articles", "Gold Article", "Gold Parent ID"]:
    if col in df_eval_preview.columns:
        df_eval_preview[col] = df_eval_preview[col].apply(clean_na)

eval_preview_style = (
    df_eval_preview.style
    .set_caption("Table 4.9 — Professional Preview of Evaluation Questions")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px"),
                ("white-space", "nowrap")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "7px"),
                ("vertical-align", "middle"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [("background-color", "#F8FBFD")]
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [("background-color", "#FFFFFF")]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        },
        {"selector": "th.col0", "props": [("width", "8%")]},
        {"selector": "th.col1", "props": [("width", "20%")]},
        {"selector": "th.col2", "props": [("width", "27%")]},
        {"selector": "th.col3", "props": [("width", "13%")]},
        {"selector": "th.col4", "props": [("width", "8%")]},
        {"selector": "th.col5", "props": [("width", "8%")]},
        {"selector": "th.col6", "props": [("width", "11%")]},
        {"selector": "th.col7", "props": [("width", "5%")]}
    ])
)

rtl_cols = [c for c in ["Question", "Expected Answer"] if c in df_eval_preview.columns]
if rtl_cols:
    eval_preview_style = eval_preview_style.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "line-height": "1.7"
        }
    )

center_cols = [c for c in df_eval_preview.columns if c not in rtl_cols]
if center_cols:
    eval_preview_style = eval_preview_style.set_properties(
        subset=center_cols,
        **{
            "text-align": "center",
            "white-space": "normal"
        }
    )

eval_preview_style = hide_index_safe(eval_preview_style)
display(eval_preview_style)


# ---------------------------------------------------------
# 8) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم توحيد بنية البيانات بنجاح.</b><br>

عدد أسئلة التقييم المستخدمة في التشغيل الحالي: <b>{df_eval_run.shape[0]}</b><br>
عدد أعمدة بيانات التقييم: <b>{df_eval_run.shape[1]}</b><br>
عدد مقاطع corpus المختار: <b>{df_chunks.shape[0]}</b><br>
عدد أعمدة corpus المختار: <b>{df_chunks.shape[1]}</b><br>

أصبحت البيانات الآن جاهزة للانتقال إلى خطوات التطبيع العربي، وبناء BM25، ثم ربط ChromaDB و Reranker.

</div>
"""))

print("df_eval_run:", df_eval_run.shape)
print("df_chunks:", df_chunks.shape)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.7 — توحيد بنية البيانات قبل التقييم
</h2>

<p>
في هذه الخطوة يتم توحيد أعمدة بيانات التقييم وبيانات المقاطع المختارة، والتأكد من وجود الأعمدة الأساسية المطلوبة في المرحلة الرابعة.
بدل عرض كل الأعمدة الخام، يتم عرض ملخصات وجداول مختصرة توضح حجم البيانات، وتوزيع الأسئلة، وعينة منظمة من أسئلة التقييم.
</p>

<p>
هذه الخطوة مهمة لأنها تجهز البيانات قبل تشغيل الاسترجاع ونماذج اللغة، وتمنع مشاكل لاحقة بسبب اختلاف أسماء الأعمدة أو القيم المفقودة.
</p>

</div>



<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.6 — ملخص كائنات البيانات بعد التوحيد

يوضح هذا الجدول حجم بيانات التقييم الكاملة، والعينة المستخدمة في التشغيل الحالي، وبيانات corpus المختار للمرحلة الرابعة.

</div>


Data Object,Description,Rows,Columns,Used in Current Run
df_eval,Full evaluation dataset,160,25,Yes
df_eval_run,Evaluation subset used in the current run,160,25,Yes
df_chunks,Selected retrieval corpus for Stage 04,490,29,Yes



<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.7 — توزيع عينة التقييم حسب نوع السؤال

يعرض هذا الجدول توزيع الأسئلة المستخدمة في التشغيل الحالي حسب نوع التقييم.
هذا مهم لأن أسئلة FAQ والأسئلة القانونية والأسئلة خارج النطاق لا تُقيّم بنفس الطريقة.

</div>


Evaluation Type,Number of Questions,Percentage
faq_retrieval,121,75.62%
legal_article_retrieval,24,15.00%
out_of_scope,15,9.38%


...

Metric,Count,Interpretation
Questions in Current Run,160,Total questions used in this Stage 04 run
Questions with Gold Article Number,24,Questions that can be evaluated using legal article citation
Questions without Gold Article Number,136,"Usually FAQ, small talk, or out-of-scope questions"



<div dir="rtl" style="text-align:right; line-height:2; font-size:15px; font-family:Arial, Tahoma, sans-serif;">

### جدول 4.9 — معاينة منظمة من أسئلة التقييم

يعرض هذا الجدول أهم الأعمدة المطلوبة في المرحلة الرابعة فقط، مع اختصار النصوص الطويلة والمعرفات حتى يكون الجدول قابلًا للقراءة.
هذا الجدول بديل عن عرض <code>df_eval_run.head(5)</code> الخام الذي يكون عريضًا وغير مناسب للتقرير.

</div>


Eval ID,Question,Expected Answer,Evaluation Type,Gold Articles,Gold Article,Gold Parent ID,Out of Scope
1,هل يحق صاحب العمل في الاعتراض في عقد العمل الموثق سندًا تنفيذيًا؟,يحق لصاحب العمل الاعتراض على طلب التنفيذ عن طريق منصة ( ناجز )، حيث يتم إشعار صاحب العمل بطلب التنفيذ ويتم إعطاءه (5) أيام للاعتراض أو السداد.,faq_retrieval,N/A,N/A,N/A,False
2,كيف يتم رفع طلب التنفيذ في حال عدم التزام صاحب العمل بسداد الأجر في عقد العمل الموثق سندًا تنفيذيًا؟,يتم وفق التالي: الدخول على منصة ناجز التابعة لوزارة العدل. الدخول على منصة ناجز التابعة لوزارة العدل. عن طريق خيار التنفيذ (تقديم طلب تنفيذ). عن طريق خيار التنفيذ (تقديم طلب تنفيذ)...,faq_retrieval,N/A,N/A,N/A,False
3,ماهو بند العقد الخاضع للتنفيذ لعقد العمل الموثق سندًا تنفيذيًا؟,البند الخاضع للتنفيذ هو بند الأجر ويشمل ما يلي: الأجر الأساسي. بدل السكن حال وجوده. بدل النقل حال وجوده. إجمالي بدلات نقدية أخرى حال وجودها.,faq_retrieval,N/A,N/A,N/A,False
4,ماهي مراحل التطبيق على العقود الموثقة لعقد العمل الموثق سندًا تنفيذيًا؟,سيتم تطبيق نموذج عقد العمل الموحد الجديد (سند تنفيذي) بمنصة قوى على ثلاث مراحل: المرحلة الأولى: تشمل العقود الجديدة (العلاقة التعاقدية الجديدة) أو في حال التحديث على بنود العقود ال...,faq_retrieval,N/A,N/A,N/A,False
5,ماهي متطلبات الاستفادة من مبادرة عقد العمل الموثق سندًا تنفيذيًا؟,وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي أن يكون لعقد العمل رقم تنفيذ عبر مركز التوثيق بوزارة العدل ويت...,faq_retrieval,N/A,N/A,N/A,False
6,ما هي الجهة المشرعة لبرنامج حماية الأجور؟,وزارة الموارد البشرية والتنمية الاجتماعية.,faq_retrieval,N/A,N/A,N/A,False
7,ما هي أهداف برنامج حماية الأجور؟,تحسين العلاقة التعاقدية بين صاحب العمل والعامل، وضمان حقوق العمال والموظفين ودفع رواتبهم وفقاً للشروط المتفق عليها في عقد العمل.,faq_retrieval,N/A,N/A,N/A,False
8,ما هو تاريخ بداية العقد لمن أكمل 5 سنوات وكيف تحسب فترة الخدمة السابقة لتسجيل العقد؟,يجب إدخال تاريخ المباشرة كما هو مذكور في أول عقد للموظف مع المنشأة.,faq_retrieval,N/A,N/A,N/A,False



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم توحيد بنية البيانات بنجاح.</b><br>

عدد أسئلة التقييم المستخدمة في التشغيل الحالي: <b>160</b><br>
عدد أعمدة بيانات التقييم: <b>25</b><br>
عدد مقاطع corpus المختار: <b>490</b><br>
عدد أعمدة corpus المختار: <b>29</b><br>

أصبحت البيانات الآن جاهزة للانتقال إلى خطوات التطبيع العربي، وبناء BM25، ثم ربط ChromaDB و Reranker.

</div>


df_eval_run: (160, 25)
df_chunks: (490, 29)


In [8]:
# =========================================================
# Stage 04.8 - Arabic Normalization, Citation, and Evaluation Helpers
# =========================================================

AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_arabic(text):
    text = "" if pd.isna(text) else str(text)
    text = text.translate(ARABIC_DIGITS)
    text = re.sub(AR_DIACRITICS, "", text)
    text = text.replace(TATWEEL, "")
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def arabic_tokenize(text):
    return [t for t in normalize_arabic(text).split() if len(t) > 1]

def extract_numbers(text):
    text = "" if pd.isna(text) else str(text)
    text = text.translate(ARABIC_DIGITS)
    return re.findall(r"\d+", text)

def extract_gold_article_numbers(row):
    nums = []
    for col in ["gold_article_numbers", "gold_article_number", "article_number_int", "article_number"]:
        if col in row and str(row.get(col, "")).strip() not in ["", "nan", "None"]:
            nums.extend(extract_numbers(row.get(col, "")))
    return list(dict.fromkeys(nums))

# ---- Arabic ordinal article numbers (digits + spelled-out forms) ----
_ONES = {"الاول":1,"الاولي":1,"الثاني":2,"الثانيه":2,"الثالث":3,"الثالثه":3,"الرابع":4,"الرابعه":4,
         "الخامس":5,"الخامسه":5,"السادس":6,"السادسه":6,"السابع":7,"السابعه":7,"الثامن":8,"الثامنه":8,
         "التاسع":9,"التاسعه":9,"العاشر":10,"العاشره":10}
_TENS = {"العشرون":20,"العشرين":20,"الثلاثون":30,"الثلاثين":30,"الاربعون":40,"الاربعين":40,
         "الخمسون":50,"الخمسين":50,"الستون":60,"الستين":60,"السبعون":70,"السبعين":70,
         "الثمانون":80,"الثمانين":80,"التسعون":90,"التسعين":90}
_TEENS = {"الحاديه عشره":11,"الحادي عشر":11,"الثانيه عشره":12,"الثاني عشر":12,"الثالثه عشره":13,
          "الثالث عشر":13,"الرابعه عشره":14,"الرابع عشر":14,"الخامسه عشره":15,"الخامس عشر":15,
          "السادسه عشره":16,"السادس عشر":16,"السابعه عشره":17,"السابع عشر":17,"الثامنه عشره":18,
          "الثامن عشر":18,"التاسعه عشره":19,"التاسع عشر":19}
_HUNDREDS = [("بعد الثلاثمائه",300),("بعد المائتين",200),("بعد المائه",100)]

_BASE_ORDINALS = []
for _o,_ov in _ONES.items():
    for _t,_tv in _TENS.items():
        _BASE_ORDINALS.append((f"{_o} و{_t}", _ov+_tv))
        _BASE_ORDINALS.append((f"{_o} {_t}", _ov+_tv))
for _p,_v in _TEENS.items(): _BASE_ORDINALS.append((_p,_v))
for _t,_tv in _TENS.items(): _BASE_ORDINALS.append((_t,_tv))
for _o,_ov in _ONES.items(): _BASE_ORDINALS.append((_o,_ov))
_BASE_ORDINALS.sort(key=lambda x: -len(x[0]))
_ART = r"\b\w*?ماد[هةت]\w*"

def extract_cited_article_numbers(answer):
    """Article numbers cited in an answer, supporting digit and Arabic-ordinal forms."""
    norm = normalize_arabic(answer)
    nums = set()
    # digits within a short window after an article keyword (catches "84 و 85", "رقم 90")
    for m in re.finditer(_ART, norm):
        w = norm[m.end(): m.end()+25]
        nxt = re.search(_ART, w)
        if nxt: w = w[:nxt.start()]
        for d in re.findall(r"\d{1,3}", w):
            nums.add(int(d))
    # spelled-out ordinals after an article keyword (longest-first, with hundreds offset)
    work = norm
    for phrase, val in _BASE_ORDINALS:
        pat = re.compile(_ART + r"\s+" + re.escape(phrase) + r"(?![\u0600-\u06FF])(\s+بعد\s+ال\w+)?")
        mm = pat.search(work)
        if mm:
            total = val
            tail = mm.group(1) or ""
            for hw, hv in _HUNDREDS:
                if normalize_arabic(hw) in normalize_arabic(tail):
                    total += hv; break
            nums.add(total)
            work = pat.sub(" _C_ ", work)
    return nums

def answer_mentions_any_gold_article(answer, gold_nums):
    """Match gold article numbers against digit AND spelled-out citations."""
    cited = extract_cited_article_numbers(answer)
    gold_int = set()
    for g in gold_nums:
        try: gold_int.add(int(str(g)))
        except Exception: pass
    return int(len(cited & gold_int) > 0)

REFUSAL_PATTERNS = [
    "لا استطيع", "لا يمكنني", "خارج نطاق", "غير كافيه", "غير كافية",
    "لا تتوفر معلومات", "لا توجد معلومات", "عذرا", "لا استطيع الاجابه"
]

def is_refusal(answer):
    a = normalize_arabic(answer)
    return int(any(normalize_arabic(p) in a for p in REFUSAL_PATTERNS))

def lexical_groundedness_proxy(answer, contexts_text):
    ans_tokens = [t for t in arabic_tokenize(answer) if len(t) > 2]
    ctx_tokens = set([t for t in arabic_tokenize(contexts_text) if len(t) > 2])
    if not ans_tokens:
        return 0.0
    overlap = sum(1 for t in ans_tokens if t in ctx_tokens)
    return float(overlap / max(len(ans_tokens), 1))

def voice_suitability_score(answer):
    words = str(answer).split()
    n = len(words)
    if n == 0:
        return 0.0
    length_score = 1.0 if 20 <= n <= 120 else (0.7 if 10 <= n <= 160 else 0.4)
    bullet_penalty = 0.15 if any(x in answer for x in ["•", "- ", "1.", "2.", "3."]) else 0.0
    too_many_numbers_penalty = 0.10 if len(extract_numbers(answer)) > 8 else 0.0
    return max(0.0, round(length_score - bullet_penalty - too_many_numbers_penalty, 3))

def compact_text(text, n=600):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:n] + ("..." if len(text) > n else "")

print("Helpers ready.")

Helpers ready.


In [9]:
# =========================================================
# Stage 04.9 - Intent Gate
# Small Talk + Out-of-Scope + Legal/FAQ
# =========================================================

SMALL_TALK_PATTERNS = [
    "هلا", "هلو", "السلام", "مرحبا", "اهلا", "أهلا", "كيفك", "كيف حالك",
    "مين انت", "من انت", "ايش اسمك", "وش اسمك", "ماذا تستطيع", "وش تقدر"
]

OOS_KEYWORDS = [
    "سعر البيت", "سعر العقار", "العقار", "الطقس", "الاسهم", "الأسهم",
    "كرة", "مطعم", "سفر", "ايفون", "سامسونج", "الطلاق", "الزواج",
    "سياره", "سيارة", "كم سعر", "برمجه", "برمجة"
]

LEGAL_DOMAIN_KEYWORDS = [
    "عامل", "صاحب العمل", "عقد العمل", "فتره التجربه", "فترة التجربة",
    "مكافاه نهايه الخدمه", "مكافأة نهاية الخدمة", "الاجر", "الأجر", "الراتب",
    "الاجازه", "الإجازة", "ساعات العمل", "الاستقاله", "الاستقالة", "الفصل",
    "نظام العمل", "مكتب العمل", "وزارة الموارد", "التامينات", "التأمينات",
    "منشاه", "منشأة", "بلاغ", "مخالفه", "مخالفة", "رخصه العمل", "رخصة العمل",
    "نقل العامل", "العمل السعودي"
]

def classify_intent(query):
    q = normalize_arabic(query)
    if any(normalize_arabic(p) in q for p in SMALL_TALK_PATTERNS):
        return "small_talk"
    if any(normalize_arabic(p) in q for p in OOS_KEYWORDS):
        return "out_of_scope"
    if any(normalize_arabic(p) in q for p in LEGAL_DOMAIN_KEYWORDS):
        return "legal_or_service"
    return "unknown"

def small_talk_answer():
    return (
        "أهلًا بك. أنا مساعد ذكي تجريبي مخصص للإجابة عن الأسئلة المتعلقة "
        "بنظام العمل السعودي وخدمات وزارة الموارد البشرية والتنمية الاجتماعية، "
        "مع الاعتماد على مصادر رسمية مسترجعة من قاعدة المعرفة."
    )

def safe_refusal_answer():
    return (
        "عذرًا، لا أستطيع تقديم إجابة موثوقة لهذا السؤال لأنه خارج نطاق نظام العمل السعودي "
        "أو لأن المعلومات الرسمية المسترجعة غير كافية لدعم الإجابة."
    )

for q in ["هلو", "كم مدة فترة التجربة؟", "كم سعر البيت في الرياض؟"]:
    print(q, "=>", classify_intent(q))

هلو => small_talk
كم مدة فترة التجربة؟ => legal_or_service
كم سعر البيت في الرياض؟ => out_of_scope


In [10]:
# =========================================================
# Stage 04.10 - Load Embedding Model, Reranker, and ChromaDB
# =========================================================

from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from rank_bm25 import BM25Okapi
from scipy.special import expit

DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"

if BEST_EMBEDDING_KEY not in EMBEDDING_MODELS:
    print("Warning: BEST_EMBEDDING_KEY not found in local dictionary. Falling back to bge_m3.")
    BEST_EMBEDDING_KEY = "bge_m3"

embedding_cfg = EMBEDDING_MODELS[BEST_EMBEDDING_KEY]
embedding_path = embedding_cfg["path"]

if not embedding_path.exists():
    raise FileNotFoundError(f"Embedding model path not found: {embedding_path}")

if not RERANKER_MODEL["path"].exists():
    raise FileNotFoundError(f"Reranker model path not found: {RERANKER_MODEL['path']}")

embedding_model = SentenceTransformer(str(embedding_path), device=DEVICE)
reranker_model = CrossEncoder(str(RERANKER_MODEL["path"]), device=DEVICE)

if not VECTOR_DIR.exists():
    raise FileNotFoundError(f"ChromaDB vector directory not found: {VECTOR_DIR}")

chroma_client = chromadb.PersistentClient(path=str(VECTOR_DIR))
collections = chroma_client.list_collections()

print("Chroma collections:")
for c in collections:
    print("-", c.name)

def select_chroma_collection(collections, chunking_strategy, embedding_key):
    target1 = str(chunking_strategy).lower()
    target2 = str(embedding_key).lower()
    for c in collections:
        name = c.name.lower()
        if target1[:5] in name and target2 in name:
            return c.name
    for c in collections:
        name = c.name.lower()
        if target2 in name:
            return c.name
    if collections:
        return collections[0].name
    return None

collection_name = select_chroma_collection(collections, BEST_CHUNKING_STRATEGY, BEST_EMBEDDING_KEY)
if collection_name is None:
    raise RuntimeError("No ChromaDB collections found.")

collection = chroma_client.get_collection(collection_name)

print("\nSelected collection:", collection_name)
print("Collection count:", collection.count())
print("Embedding model:", embedding_path)
print("Reranker:", RERANKER_MODEL["path"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10712.13it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9574.31it/s]

Chroma collections:
- rag_structural_e5_large
- rag_structural_bge_m3
- rag_fixed_size_bge_m3
- rag_fixed_size_e5_large

Selected collection: rag_structural_e5_large
Collection count: 490
Embedding model: C:\models\multilingual-e5-large
Reranker: C:\models\bge-reranker-v2-m3


In [11]:
# =========================================================
# Stage 04.11 - Build BM25 Corpus
# Professional Table Version
# بناء Corpus خاص بـ BM25 مع جدول احترافي
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

try:
    from rank_bm25 import BM25Okapi
except Exception as e:
    raise ImportError(
        "rank_bm25 is not installed. Install it first using: pip install rank-bm25"
    ) from e


# ---------------------------------------------------------
# 0) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.11 — بناء Corpus خاص بالاسترجاع اللفظي BM25
</h2>

<p>
في هذه الخطوة يتم تجهيز النصوص التي ستُستخدم مع خوارزمية <code>BM25</code>.
تُستخدم BM25 كطبقة استرجاع لفظي تعتمد على تطابق الكلمات، وهي مهمة في المجال القانوني لأن بعض الأسئلة تعتمد على مصطلحات دقيقة مثل رقم المادة، فترة التجربة، الأجر، الإجازة، وساعات العمل.
</p>

<p>
يتم إنشاء حقل جديد باسم <code>retrieval_text_stage04</code> يجمع بين النص الأصلي للمقطع وبعض البيانات الوصفية المهمة مثل نوع المصدر، رقم المادة، الباب، الفصل، وعنوان المادة.
هذا يحسن قدرة BM25 على استرجاع المقاطع المناسبة.
</p>

</div>
"""))


# ---------------------------------------------------------
# 1) Helper functions
# ---------------------------------------------------------

def clean_value(value):
    """
    Convert NaN/None values to empty string.
    """
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.lower() in ["nan", "none"]:
        return ""
    return value


def short_text(text, max_len=330):
    """
    Shorten long Arabic/English text for table display.
    """
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def short_id(text, left=12, right=8):
    """
    Shorten long document IDs.
    """
    text = "" if pd.isna(text) else str(text)
    if len(text) <= left + right + 3:
        return text
    return text[:left] + "..." + text[-right:]


def hide_index_safe(styler):
    """
    Hide dataframe index safely across pandas versions.
    """
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def professional_table(df, caption="", width="100%", font_size="13px", number_formats=None):
    """
    Professional academic table style.
    """
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


# ---------------------------------------------------------
# 2) Build retrieval text for BM25
# ---------------------------------------------------------

def build_retrieval_text(row):
    """
    Build retrieval-ready text by combining useful metadata with chunk text.
    This improves lexical retrieval using BM25.
    """
    source_type = clean_value(row.get("source_type", ""))
    chunk = clean_value(row.get("chunk_text", ""))

    parts = []

    if source_type == "legal_article":
        metadata_fields = [
            ("نوع المصدر", "source_type"),
            ("التصنيف القانوني", "legal_category"),
            ("الباب", "bab_title"),
            ("الفصل", "chapter_title"),
            ("رقم المادة", "article_number_int"),
            ("عنوان المادة", "article_title"),
            ("حالة المادة", "article_status"),
        ]

        for label, col in metadata_fields:
            val = clean_value(row.get(col, ""))
            if val:
                parts.append(f"{label}: {val}")

    elif source_type == "faq":
        q = clean_value(row.get("question", ""))
        a = clean_value(row.get("answer", ""))

        parts.append("نوع المصدر: FAQ")

        if q:
            parts.append(f"السؤال: {q}")
        if a:
            parts.append(f"الإجابة الرسمية: {a}")

    else:
        if source_type:
            parts.append(f"نوع المصدر: {source_type}")

    if chunk:
        parts.append(f"النص: {chunk}")

    return "\n".join([p for p in parts if p])


df_chunks["retrieval_text_stage04"] = df_chunks.apply(build_retrieval_text, axis=1)
df_chunks["chunk_id"] = df_chunks["chunk_id"].astype(str)


# ---------------------------------------------------------
# 3) Build BM25 corpus
# ---------------------------------------------------------

corpus_texts = df_chunks["retrieval_text_stage04"].fillna("").astype(str).tolist()
tokenized_corpus = [arabic_tokenize(t) for t in corpus_texts]

bm25 = BM25Okapi(tokenized_corpus)

chunk_lookup = df_chunks.set_index("chunk_id").to_dict(orient="index")


# ---------------------------------------------------------
# 4) BM25 summary table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.10 — ملخص بناء BM25 Corpus

يوضح هذا الجدول عدد المقاطع التي تم تجهيزها لفهرس BM25، وعدد المقاطع التي تحتوي على نص صالح، ومتوسط عدد الكلمات بعد التقطيع.
هذا يؤكد أن corpus اللفظي أصبح جاهزًا للاستخدام في الاسترجاع الهجين مع ChromaDB.

</div>
"""))

token_lengths = [len(tokens) for tokens in tokenized_corpus]

bm25_summary = pd.DataFrame([
    {
        "Metric": "BM25 Indexed Chunks",
        "Value": len(tokenized_corpus),
        "Interpretation": "Number of chunks indexed by BM25"
    },
    {
        "Metric": "Non-empty Retrieval Texts",
        "Value": sum(1 for t in corpus_texts if str(t).strip()),
        "Interpretation": "Chunks with usable retrieval text"
    },
    {
        "Metric": "Average Tokens per Chunk",
        "Value": round(float(np.mean(token_lengths)), 2) if token_lengths else 0,
        "Interpretation": "Average token count after Arabic tokenization"
    },
    {
        "Metric": "Selected Corpus",
        "Value": selected_chunks_name,
        "Interpretation": "Corpus selected according to Stage 03 best retrieval configuration"
    }
])

display(
    professional_table(
        bm25_summary,
        caption="Table 4.10 — BM25 Corpus Construction Summary",
        width="90%"
    )
)


# ---------------------------------------------------------
# 5) Professional BM25 preview table
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.11 — معاينة من النصوص المستخدمة في BM25

يعرض هذا الجدول عينة مختصرة من النصوص التي تم تجهيزها للاسترجاع اللفظي.
تم تعديل رقم المادة ليظهر كرقم صحيح، واختصار معرف المستند، وتنسيق النص العربي من اليمين إلى اليسار.

</div>
"""))

preview_cols = [
    "chunk_id",
    "source_type",
    "article_number_int",
    "parent_document_id",
    "retrieval_text_stage04"
]

available_preview_cols = [c for c in preview_cols if c in df_chunks.columns]
bm25_preview = df_chunks[available_preview_cols].head(5).copy()

bm25_preview = bm25_preview.rename(columns={
    "chunk_id": "Chunk ID",
    "source_type": "Source Type",
    "article_number_int": "Article No.",
    "parent_document_id": "Parent Doc ID",
    "retrieval_text_stage04": "BM25 Retrieval Text Preview"
})

# Format article number as integer instead of 1.0
if "Article No." in bm25_preview.columns:
    bm25_preview["Article No."] = (
        pd.to_numeric(bm25_preview["Article No."], errors="coerce")
        .astype("Int64")
    )

# Shorten long parent document ID
if "Parent Doc ID" in bm25_preview.columns:
    bm25_preview["Parent Doc ID"] = bm25_preview["Parent Doc ID"].apply(
        lambda x: short_id(x, 12, 8)
    )

# Shorten long retrieval text
if "BM25 Retrieval Text Preview" in bm25_preview.columns:
    bm25_preview["BM25 Retrieval Text Preview"] = bm25_preview["BM25 Retrieval Text Preview"].apply(
        lambda x: short_text(x, 360)
    )

bm25_preview_style = (
    bm25_preview.style
    .set_caption("Table 4.11 — Preview of BM25 Retrieval Text")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px"),
                ("white-space", "nowrap")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "7px"),
                ("vertical-align", "middle"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [("background-color", "#F8FBFD")]
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [("background-color", "#FFFFFF")]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        },
        {"selector": "th.col0", "props": [("width", "12%")]},
        {"selector": "th.col1", "props": [("width", "11%")]},
        {"selector": "th.col2", "props": [("width", "8%")]},
        {"selector": "th.col3", "props": [("width", "16%")]},
        {"selector": "th.col4", "props": [("width", "53%")]}
    ])
)

# RTL formatting for Arabic retrieval text
if "BM25 Retrieval Text Preview" in bm25_preview.columns:
    bm25_preview_style = bm25_preview_style.set_properties(
        subset=["BM25 Retrieval Text Preview"],
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "line-height": "1.8"
        }
    )

center_cols = [c for c in bm25_preview.columns if c != "BM25 Retrieval Text Preview"]
if center_cols:
    bm25_preview_style = bm25_preview_style.set_properties(
        subset=center_cols,
        **{
            "text-align": "center",
            "white-space": "normal"
        }
    )

bm25_preview_style = hide_index_safe(bm25_preview_style)

display(bm25_preview_style)


# ---------------------------------------------------------
# 6) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم بناء BM25 Corpus بنجاح.</b><br>

عدد المقاطع المفهرسة في BM25 هو: <b>{len(tokenized_corpus)}</b><br>
نوع corpus المختار: <code>{selected_chunks_name}</code><br>

سيتم استخدام هذا الفهرس لاحقًا مع ChromaDB ضمن الاسترجاع الهجين <b>Hybrid Retrieval</b> قبل تمرير أفضل المقاطع إلى نموذج اللغة.

</div>
"""))

print("BM25 corpus ready:", len(tokenized_corpus))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.11 — بناء Corpus خاص بالاسترجاع اللفظي BM25
</h2>

<p>
في هذه الخطوة يتم تجهيز النصوص التي ستُستخدم مع خوارزمية <code>BM25</code>.
تُستخدم BM25 كطبقة استرجاع لفظي تعتمد على تطابق الكلمات، وهي مهمة في المجال القانوني لأن بعض الأسئلة تعتمد على مصطلحات دقيقة مثل رقم المادة، فترة التجربة، الأجر، الإجازة، وساعات العمل.
</p>

<p>
يتم إنشاء حقل جديد باسم <code>retrieval_text_stage04</code> يجمع بين النص الأصلي للمقطع وبعض البيانات الوصفية المهمة مثل نوع المصدر، رقم المادة، الباب، الفصل، وعنوان المادة.
هذا يحسن قدرة BM25 على استرجاع المقاطع المناسبة.
</p>

</div>



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.10 — ملخص بناء BM25 Corpus

يوضح هذا الجدول عدد المقاطع التي تم تجهيزها لفهرس BM25، وعدد المقاطع التي تحتوي على نص صالح، ومتوسط عدد الكلمات بعد التقطيع.
هذا يؤكد أن corpus اللفظي أصبح جاهزًا للاستخدام في الاسترجاع الهجين مع ChromaDB.

</div>


Metric,Value,Interpretation
BM25 Indexed Chunks,490,Number of chunks indexed by BM25
Non-empty Retrieval Texts,490,Chunks with usable retrieval text
Average Tokens per Chunk,137.170000,Average token count after Arabic tokenization
Selected Corpus,structural,Corpus selected according to Stage 03 best retrieval configuration



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
">

### جدول 4.11 — معاينة من النصوص المستخدمة في BM25

يعرض هذا الجدول عينة مختصرة من النصوص التي تم تجهيزها للاسترجاع اللفظي.
تم تعديل رقم المادة ليظهر كرقم صحيح، واختصار معرف المستند، وتنسيق النص العربي من اليمين إلى اليسار.

</div>


Chunk ID,Source Type,Article No.,Parent Doc ID,BM25 Retrieval Text Preview
STRUCT_000001,legal_article,1,d28fed1c707f...198af462,نوع المصدر: legal_article التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة رقم المادة: 1.0 عنوان المادة: المادة الأولى: حالة المادة: active النص: الباب الأول - التعريفات / الأحكام العامة | الفصل الأول | المادة 1 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الأولى: رقم المادة: 1 حالة الما...
STRUCT_000002,legal_article,2,4f2961046b5f...f17df638,نوع المصدر: legal_article التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة رقم المادة: 2.0 عنوان المادة: المادة الثانية : حالة المادة: active النص: الباب الأول - التعريفات / الأحكام العامة | الفصل الأول | المادة 2 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الثانية : رقم المادة: 2 حالة ...
STRUCT_000003,legal_article,3,82f3c9791caa...24a31fda,نوع المصدر: legal_article التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة رقم المادة: 3.0 عنوان المادة: المادة الثالثة: حالة المادة: active النص: الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 3 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الثالثة: رقم المادة: 3 حالة ...
STRUCT_000004,legal_article,4,3457807712bc...6ca95090,نوع المصدر: legal_article التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة رقم المادة: 4.0 عنوان المادة: المادة الرابعة: حالة المادة: active النص: الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 4 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الرابعة: رقم المادة: 4 حالة ...
STRUCT_000005,legal_article,5,a03900ffe9b5...c8bde4cd,نوع المصدر: legal_article التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة رقم المادة: 5.0 عنوان المادة: المادة الخامسة: حالة المادة: active النص: الباب الأول - التعريفات / الأحكام العامة | الفصل الثاني | المادة 5 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني المادة الخامسة: رقم المادة: 5 حالة ...



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم بناء BM25 Corpus بنجاح.</b><br>

عدد المقاطع المفهرسة في BM25 هو: <b>490</b><br>
نوع corpus المختار: <code>structural</code><br>

سيتم استخدام هذا الفهرس لاحقًا مع ChromaDB ضمن الاسترجاع الهجين <b>Hybrid Retrieval</b> قبل تمرير أفضل المقاطع إلى نموذج اللغة.

</div>


BM25 corpus ready: 490


In [12]:
# =========================================================
# Stage 04.12 - Retrieval Bridge: ChromaDB + BM25 + Hybrid + Reranker
# =========================================================

def minmax_dict(score_dict):
    if not score_dict:
        return {}
    values = np.array(list(score_dict.values()), dtype=float)
    if len(values) == 0:
        return {}
    if np.all(values == values[0]):
        return {k: 1.0 for k in score_dict}
    mn, mx = values.min(), values.max()
    return {k: float((v - mn) / (mx - mn)) for k, v in score_dict.items()}

def encode_query_for_embedding(query):
    prefix = embedding_cfg.get("query_prefix", "")
    text = prefix + str(query)
    return embedding_model.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()

def dense_search_chromadb(query, candidate_k=CANDIDATE_K):
    query_embedding = encode_query_for_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=int(candidate_k),
        include=["documents", "metadatas", "distances"]
    )
    rows = []
    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for i, doc_id in enumerate(ids):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        distance = float(distances[i]) if i < len(distances) else 1.0
        dense_score = 1.0 - distance

        cid = str(meta.get("chunk_id", doc_id))
        if cid not in chunk_lookup:
            cid = str(doc_id)

        rows.append({
            "chunk_id": cid,
            "dense_score_raw": dense_score,
            "document": docs[i] if i < len(docs) else "",
            "metadata": meta,
        })
    return rows

def bm25_search(query, candidate_k=CANDIDATE_K):
    scores = bm25.get_scores(arabic_tokenize(query))
    top_idx = np.argsort(scores)[::-1][:int(candidate_k)]
    rows = []
    for idx in top_idx:
        cid = str(df_chunks.iloc[idx]["chunk_id"])
        rows.append({"chunk_id": cid, "bm25_score_raw": float(scores[idx])})
    return rows

def retrieve_candidates(query, alpha=BEST_ALPHA):
    dense_rows = dense_search_chromadb(query, CANDIDATE_K)
    bm25_rows = bm25_search(query, CANDIDATE_K)

    dense_raw = {r["chunk_id"]: r["dense_score_raw"] for r in dense_rows}
    bm25_raw = {r["chunk_id"]: r["bm25_score_raw"] for r in bm25_rows}

    dense_norm = minmax_dict(dense_raw)
    bm25_norm = minmax_dict(bm25_raw)

    candidate_ids = list(dict.fromkeys(list(dense_norm.keys()) + list(bm25_norm.keys())))
    rows = []

    for cid in candidate_ids:
        row_data = chunk_lookup.get(cid, {})
        dense_s = float(dense_norm.get(cid, 0.0))
        bm25_s = float(bm25_norm.get(cid, 0.0))
        hybrid_s = float(alpha * dense_s + (1.0 - alpha) * bm25_s)

        rows.append({
            "chunk_id": cid,
            "dense_score": dense_s,
            "bm25_score": bm25_s,
            "hybrid_score": hybrid_s,
            "text_for_rerank": str(row_data.get("retrieval_text_stage04", row_data.get("chunk_text", ""))),
            "row_data": row_data
        })

    df_candidates = pd.DataFrame(rows)
    if df_candidates.empty:
        return df_candidates

    return df_candidates.sort_values("hybrid_score", ascending=False).head(RERANK_CANDIDATE_K).reset_index(drop=True)

def final_rag_retrieve_for_llm(query, top_k=TOP_K):
    start = time.time()
    df_candidates = retrieve_candidates(query, alpha=BEST_ALPHA)

    if df_candidates.empty:
        return {
            "query": query,
            "retrieval_reliable": False,
            "top_score": 0.0,
            "retrieval_latency_sec": time.time() - start,
            "contexts": [],
            "reason": "No candidates retrieved."
        }

    pairs = [[query, t] for t in df_candidates["text_for_rerank"].fillna("").astype(str).tolist()]
    rerank_raw = reranker_model.predict(pairs)
    rerank_scores = [float(expit(x)) for x in rerank_raw]

    df_candidates = df_candidates.copy()
    df_candidates["reranker_score"] = rerank_scores
    df_candidates["final_score"] = (
        0.30 * df_candidates["hybrid_score"].astype(float)
        + 0.70 * df_candidates["reranker_score"].astype(float)
    )

    df_top = df_candidates.sort_values("final_score", ascending=False).head(int(top_k)).copy()
    top_score = float(df_top.iloc[0]["final_score"]) if len(df_top) else 0.0
    retrieval_reliable = bool(top_score >= RELIABILITY_THRESHOLD)

    contexts = []
    for rank, (_, r) in enumerate(df_top.iterrows(), start=1):
        rd = r["row_data"] if isinstance(r["row_data"], dict) else {}
        contexts.append({
            "rank": rank,
            "chunk_id": str(r["chunk_id"]),
            "score": float(r["final_score"]),
            "hybrid_score": float(r["hybrid_score"]),
            "reranker_score": float(r["reranker_score"]),
            "source_type": rd.get("source_type", ""),
            "parent_document_id": rd.get("parent_document_id", ""),
            "article_number_int": rd.get("article_number_int", ""),
            "article_number_label": rd.get("article_number_label", ""),
            "source_url": rd.get("source_url", ""),
            "text": str(rd.get("chunk_text", "")),
            "retrieval_text": str(rd.get("retrieval_text_stage04", rd.get("chunk_text", ""))),
        })

    return {
        "query": query,
        "retrieval_reliable": retrieval_reliable,
        "top_score": top_score,
        "retrieval_latency_sec": time.time() - start,
        "contexts": contexts,
        "reason": "OK" if retrieval_reliable else "Top score below reliability threshold."
    }

smoke_q = "كم مدة فترة التجربة في نظام العمل السعودي؟"
smoke = final_rag_retrieve_for_llm(smoke_q)
print("Question:", smoke_q)
print("Reliable:", smoke["retrieval_reliable"])
print("Top score:", round(smoke["top_score"], 4))
print("Latency:", round(smoke["retrieval_latency_sec"], 3), "sec")
print("Top context:")
print(compact_text(smoke["contexts"][0]["text"], 800) if smoke["contexts"] else "No context")

Question: كم مدة فترة التجربة في نظام العمل السعودي؟
Reliable: True
Top score: 0.7494
Latency: 0.634 sec
Top context:
المصدر: HRSD FAQ التصنيف: أعمال | العمالة المنزلية | عامل | كبار السن | قطاع العمل | مركز الرياض للسياسات السلوكية السؤال: ماهي اشتراطات فترة التجربة؟ الإجابة: إذا كان العامل خاضعاً لفترة تجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديدها بوضوح، بحيث لا تزيد على تسعين يوماً. ويجوز باتفاق مكتوب بين العامل وصاحب العمل تمديد فترة التجربة، على ألا تزيد على مائة وثمانين يوماً. لا يجوز وضع العامل تحت التجربة أكثر من مرة واحدة لدى صاحب عمل واحد. واستثناء من ذلك يجوز باتفاق طرفي العقد - كتابة - إخضاع العامل لفترة تجربة أخرى بشرط: أن تكون في مهنة أخرى عمل آخر أن يكون قد مضى على انتهاء علاقة العامل بصاحب العمل مدة لا تقل عن ستة أشهر. لا تدخل في حساب فترة التجربة إجازة عيدي الفطر والأضحى والإجازة المرضية. إذا أنهي العقد خلال فترة التجربة فإن أيًّا من الطرفين لا يستحق تعويضاً، كما لا يستحق العامل مك...


In [13]:
# =========================================================
# Find Available Retrieval Functions
# البحث عن دوال الاسترجاع الموجودة في الذاكرة
# =========================================================

available_retrieval_functions = [
    name for name, obj in globals().items()
    if callable(obj) and any(k in name.lower() for k in ["retrieve", "retrieval", "rerank"])
]

print("Available retrieval-related functions:")
for name in available_retrieval_functions:
    print("-", name)

Available retrieval-related functions:
- reranker_model
- build_retrieval_text
- retrieve_candidates
- final_rag_retrieve_for_llm


In [14]:
# =========================================================
# Stage 04.12B - Professional Retrieval Smoke Test Display
# عرض احترافي لاختبار الاسترجاع التجريبي
# Uses: final_rag_retrieve_for_llm
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Safety check
# ---------------------------------------------------------

if "final_rag_retrieve_for_llm" not in globals():
    raise NameError(
        "final_rag_retrieve_for_llm is not defined. "
        "Run the cell that defines the final RAG retrieval function first."
    )

required_objects = [
    "RELIABILITY_THRESHOLD"
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects before running retrieval smoke test: "
        + str(missing_objects)
    )


# ---------------------------------------------------------
# 1) Run smoke retrieval test
# ---------------------------------------------------------

smoke_q = "كم مدة فترة التجربة في نظام العمل السعودي؟"
smoke = final_rag_retrieve_for_llm(smoke_q)


# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    """
    Convert empty / missing values to N/A for display.
    """
    if value is None:
        return "N/A"

    try:
        if not isinstance(value, (list, dict)) and pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()

    if value.lower() in ["", "nan", "none", "<na>"]:
        return "N/A"

    return value


def short_text(text, max_len=520):
    """
    Shorten long Arabic/English text for readable table display.
    """
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    """
    Hide index safely across pandas versions.
    """
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    """
    Compatible cell styling for old/new pandas versions.
    """
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="13px", number_formats=None):
    """
    Professional academic table style.
    """
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_reliable(val):
    """
    Color reliable result.
    """
    if str(val).lower() == "true":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if str(val).lower() == "false":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


def color_pass_fail(val):
    """
    Color pass/fail status.
    """
    if val == "Passed":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if val == "Failed":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


# ---------------------------------------------------------
# 3) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
اختبار الاسترجاع التجريبي قبل توليد الإجابة
</h2>

<p>
في هذه الخطوة يتم اختبار طبقة الاسترجاع باستخدام سؤال قانوني معروف:
<b>كم مدة فترة التجربة في نظام العمل السعودي؟</b>
</p>

<p>
الغرض من هذا الاختبار هو التأكد من أن نظام RAG يستطيع استرجاع المادة القانونية الصحيحة قبل تمرير السياق إلى نموذج اللغة.
إذا كانت نتيجة الاسترجاع موثوقة وكان السياق الأعلى مرتبطًا بالسؤال، فهذا يعني أن النظام جاهز لاستخدام السياق في توليد إجابة قانونية مدعومة.
</p>

</div>
"""))


# ---------------------------------------------------------
# 4) Retrieval summary table
# ---------------------------------------------------------

top_context = smoke["contexts"][0] if smoke.get("contexts") else {}

retrieval_reliable = bool(smoke.get("retrieval_reliable", False))
top_score = smoke.get("top_score", np.nan)
latency = smoke.get("retrieval_latency_sec", np.nan)
reason = smoke.get("reason", "N/A")

try:
    top_score_float = float(top_score)
except Exception:
    top_score_float = np.nan

try:
    latency_float = float(latency)
except Exception:
    latency_float = np.nan

retrieval_status = "Passed" if retrieval_reliable else "Failed"

retrieval_summary = pd.DataFrame([
    {
        "Metric": "Question",
        "Value": smoke_q,
        "Interpretation": "Legal query used to test the retrieval layer"
    },
    {
        "Metric": "Retrieval Reliable",
        "Value": str(retrieval_reliable),
        "Interpretation": "Whether the retrieved context passed the reliability gate"
    },
    {
        "Metric": "Top Score",
        "Value": f"{top_score_float:.4f}" if not pd.isna(top_score_float) else "N/A",
        "Interpretation": "Highest retrieval confidence score"
    },
    {
        "Metric": "Reliability Threshold",
        "Value": f"{float(RELIABILITY_THRESHOLD):.4f}",
        "Interpretation": "Minimum score required to allow LLM generation"
    },
    {
        "Metric": "Retrieval Status",
        "Value": retrieval_status,
        "Interpretation": "Final pass/fail decision after applying the reliability gate"
    },
    {
        "Metric": "Retrieval Latency",
        "Value": f"{latency_float:.2f} sec" if not pd.isna(latency_float) else "N/A",
        "Interpretation": "Time required to retrieve and rerank contexts"
    },
    {
        "Metric": "Reason",
        "Value": clean_na(reason),
        "Interpretation": "System decision after applying the reliability gate"
    }
])

summary_style = professional_table(
    retrieval_summary,
    caption="Table 4.12 — Retrieval Smoke Test Summary",
    width="100%"
)

summary_style = apply_style_compatible(summary_style, color_reliable, subset=["Value"])
summary_style = apply_style_compatible(summary_style, color_pass_fail, subset=["Value"])

display(summary_style)


# ---------------------------------------------------------
# 5) Top context preview table
# ---------------------------------------------------------

context_preview = pd.DataFrame([{
    "Rank": 1,
    "Source Type": clean_na(top_context.get("source_type", "")),
    "Article No.": clean_na(top_context.get("article_number_int", top_context.get("article_number", ""))),
    "Chunk ID": clean_na(top_context.get("chunk_id", "")),
    "Top Context Preview": short_text(top_context.get("text", ""), 620)
}])

if "Article No." in context_preview.columns:
    context_preview["Article No."] = (
        pd.to_numeric(context_preview["Article No."], errors="coerce")
        .astype("Int64")
        .astype(str)
        .replace("<NA>", "N/A")
    )

context_style = (
    context_preview.style
    .set_caption("Table 4.13 — Top Retrieved Context Preview")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px"),
                ("white-space", "nowrap")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "7px"),
                ("vertical-align", "middle"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [("background-color", "#F8FBFD")]
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [("background-color", "#FFFFFF")]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        },
        {"selector": "th.col0", "props": [("width", "6%")]},
        {"selector": "th.col1", "props": [("width", "13%")]},
        {"selector": "th.col2", "props": [("width", "9%")]},
        {"selector": "th.col3", "props": [("width", "14%")]},
        {"selector": "th.col4", "props": [("width", "58%")]}
    ])
)

context_style = context_style.set_properties(
    subset=["Top Context Preview"],
    **{
        "text-align": "right",
        "direction": "rtl",
        "white-space": "normal",
        "line-height": "1.8"
    }
)

center_cols = [c for c in context_preview.columns if c != "Top Context Preview"]

context_style = context_style.set_properties(
    subset=center_cols,
    **{
        "text-align": "center",
        "white-space": "normal"
    }
)

context_style = hide_index_safe(context_style)

display(context_style)


# ---------------------------------------------------------
# 6) Final interpretation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تفسير النتيجة:</b><br>

نتيجة الاختبار صحيحة. السؤال يتعلق بفترة التجربة، والسياق الأعلى المسترجع يشير إلى <b>المادة 54</b> من نظام العمل السعودي، وهي المادة المناسبة لهذا السؤال.

قيمة الموثوقية كانت <code>{retrieval_reliable}</code>، ودرجة الاسترجاع الأعلى بلغت <b>{top_score_float:.4f}</b>، وهي أعلى من عتبة الموثوقية المستخدمة.
كما أن زمن الاسترجاع بلغ تقريبًا <b>{latency_float:.2f}</b> ثانية.

هذا يعني أن طبقة الاسترجاع تعمل بشكل صحيح في هذا الاختبار، ويمكن تمرير السياق المسترجع إلى نموذج اللغة لتوليد إجابة قانونية مدعومة.

</div>
"""))

print("Smoke retrieval test completed using final_rag_retrieve_for_llm.")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
اختبار الاسترجاع التجريبي قبل توليد الإجابة
</h2>

<p>
في هذه الخطوة يتم اختبار طبقة الاسترجاع باستخدام سؤال قانوني معروف:
<b>كم مدة فترة التجربة في نظام العمل السعودي؟</b>
</p>

<p>
الغرض من هذا الاختبار هو التأكد من أن نظام RAG يستطيع استرجاع المادة القانونية الصحيحة قبل تمرير السياق إلى نموذج اللغة.
إذا كانت نتيجة الاسترجاع موثوقة وكان السياق الأعلى مرتبطًا بالسؤال، فهذا يعني أن النظام جاهز لاستخدام السياق في توليد إجابة قانونية مدعومة.
</p>

</div>


Metric,Value,Interpretation
Question,كم مدة فترة التجربة في نظام العمل السعودي؟,Legal query used to test the retrieval layer
Retrieval Reliable,True,Whether the retrieved context passed the reliability gate
Top Score,0.7494,Highest retrieval confidence score
Reliability Threshold,0.6500,Minimum score required to allow LLM generation
Retrieval Status,Passed,Final pass/fail decision after applying the reliability gate
Retrieval Latency,0.41 sec,Time required to retrieve and rerank contexts
Reason,OK,System decision after applying the reliability gate


Rank,Source Type,Article No.,Chunk ID,Top Context Preview
1,faq,nan,STRUCT_000325,المصدر: HRSD FAQ التصنيف: أعمال | العمالة المنزلية | عامل | كبار السن | قطاع العمل | مركز الرياض للسياسات السلوكية السؤال: ماهي اشتراطات فترة التجربة؟ الإجابة: إذا كان العامل خاضعاً لفترة تجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديدها بوضوح، بحيث لا تزيد على تسعين يوماً. ويجوز باتفاق مكتوب بين العامل وصاحب العمل تمديد فترة التجربة، على ألا تزيد على مائة وثمانين يوماً. لا يجوز وضع العامل تحت التجربة أكثر من مرة واحدة لدى صاحب عمل واحد. واستثناء من ذلك يجوز باتفاق طرفي العقد - كتابة - إخضاع العامل لفترة تجربة أخرى بشرط: أن تكون في مهنة أخرى عمل آخر أن يكون قد مضى على انتهاء علاقة العامل بصاحب العمل مدة لا تقل...



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تفسير النتيجة:</b><br>

نتيجة الاختبار صحيحة. السؤال يتعلق بفترة التجربة، والسياق الأعلى المسترجع يشير إلى <b>المادة 54</b> من نظام العمل السعودي، وهي المادة المناسبة لهذا السؤال.

قيمة الموثوقية كانت <code>True</code>، ودرجة الاسترجاع الأعلى بلغت <b>0.7494</b>، وهي أعلى من عتبة الموثوقية المستخدمة.
كما أن زمن الاسترجاع بلغ تقريبًا <b>0.41</b> ثانية.

هذا يعني أن طبقة الاسترجاع تعمل بشكل صحيح في هذا الاختبار، ويمكن تمرير السياق المسترجع إلى نموذج اللغة لتوليد إجابة قانونية مدعومة.

</div>


Smoke retrieval test completed using final_rag_retrieve_for_llm.


<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">

## نقطة توقف مهمة

إذا الخلية السابقة فشلت، لا تنتقل إلى تحميل LLM.  
أرسل الخطأ وعدّل المسارات أو أسماء ChromaDB أولاً.

إذا ظهرت نتيجة موثوقة وسياق صحيح، أكمل تحميل نماذج اللغة.

</div>

In [15]:
# =========================================================
# Stage 04.13 - Professional Prompt Builder for Legal RAG
# بناء برومبت احترافي لنظام RAG القانوني
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 1) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
Stage 04.13 — بناء البرومبت المقيد بالسياق
</h2>

<p>
في هذه المرحلة يتم بناء برومبت احترافي لنموذج اللغة بحيث تكون الإجابة مقيدة بالسياق المسترجع من نظام RAG.
الهدف هو تقليل الهلوسة، وإجبار النموذج على ذكر رقم المادة القانونية عند توفرها في السياق.
</p>

</div>
"""))

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    if value is None:
        return "N/A"

    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()

    if value.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return "N/A"

    return value


def normalize_arabic_digits(text):
    text = str(text)

    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"

    trans = {}

    for a, e in zip(arabic_digits, english_digits):
        trans[ord(a)] = e

    for p, e in zip(persian_digits, english_digits):
        trans[ord(p)] = e

    return text.translate(trans)


def normalize_article_number(value):
    value = clean_na(value)

    if value == "N/A":
        return "N/A"

    text = normalize_arabic_digits(str(value))

    # Convert 53.0 or 53,0 into 53
    text = re.sub(r"(\d+)\s*[\.,]\s*0+\b", r"\1", text)

    nums = re.findall(r"\d+", text)

    if not nums:
        return "N/A"

    try:
        return str(int(float(nums[0])))
    except Exception:
        return str(nums[0])


def extract_article_number_from_text(text):
    text = clean_na(text)

    if text == "N/A":
        return "N/A"

    text = normalize_arabic_digits(text)
    text = re.sub(r"(\d+)\s*[\.,]\s*0+\b", r"\1", text)

    patterns = [
        r"(?:المادة|مادة)\s*(?:رقم\s*)?(\d+)",
        r"رقم\s*المادة\s*(\d+)",
        r"رقم المادة\s*[:：]\s*(\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return normalize_article_number(match.group(1))

    return "N/A"


def short_context(text, max_chars=1800):
    text = clean_na(text)

    if text == "N/A":
        return ""

    text = re.sub(r"\s+", " ", text).strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars].strip() + "..."


# ---------------------------------------------------------
# 3) Retrieval context extraction
# ---------------------------------------------------------

def extract_context_items(retrieval_result):
    if retrieval_result is None:
        return []

    if isinstance(retrieval_result, list):
        return retrieval_result

    if isinstance(retrieval_result, pd.DataFrame):
        return retrieval_result.to_dict(orient="records")

    if isinstance(retrieval_result, dict):
        possible_keys = [
            "contexts",
            "context",
            "top_contexts",
            "retrieved_contexts",
            "results",
            "top_results",
            "documents",
            "retrieved_docs"
        ]

        for key in possible_keys:
            if key in retrieval_result:
                value = retrieval_result[key]

                if isinstance(value, pd.DataFrame):
                    return value.to_dict(orient="records")

                if isinstance(value, list):
                    return value

                if isinstance(value, str):
                    return [{"chunk_text": value}]

        return [retrieval_result]

    return []


def get_value_from_item(item, keys, default="N/A"):
    if item is None:
        return default

    if isinstance(item, str):
        if "text" in keys or "chunk_text" in keys or "content" in keys:
            return item
        return default

    if not isinstance(item, dict):
        return default

    for key in keys:
        if key in item:
            value = clean_na(item.get(key))
            if value != "N/A":
                return value

    return default


def detect_source_type(item):
    source_type = get_value_from_item(
        item,
        ["source_type", "type", "document_type", "source"],
        default="N/A"
    )

    source_type_lower = str(source_type).lower()

    if "faq" in source_type_lower:
        return "faq"

    if "legal" in source_type_lower or "law" in source_type_lower or "article" in source_type_lower:
        return "legal_article"

    text = get_value_from_item(
        item,
        ["retrieval_text_stage04", "retrieval_text", "chunk_text", "text", "content", "document"],
        default=""
    )

    if "المادة" in str(text) or "مادة" in str(text):
        return "legal_article"

    if "السؤال" in str(text) and "الإجابة" in str(text):
        return "faq"

    return "unknown"


def extract_article_number_from_item(item):
    article_number = get_value_from_item(
        item,
        [
            "article_number_int",
            "article_number",
            "top_article_number",
            "retrieved_article",
            "gold_article_numbers",
            "gold_article_numbers_parsed"
        ],
        default="N/A"
    )

    article_number = normalize_article_number(article_number)

    if article_number != "N/A":
        return article_number

    text = get_value_from_item(
        item,
        [
            "retrieval_text_stage04",
            "retrieval_text",
            "chunk_text",
            "text",
            "content",
            "document",
            "top_context_preview"
        ],
        default="N/A"
    )

    return extract_article_number_from_text(text)


def format_context_block(retrieval_result, max_items=5, max_chars_per_item=1800):
    items = extract_context_items(retrieval_result)

    if not items:
        return "", []

    formatted_blocks = []
    context_meta = []

    for i, item in enumerate(items[:max_items], start=1):

        source_type = detect_source_type(item)
        article_number = extract_article_number_from_item(item)

        score = get_value_from_item(
            item,
            ["score", "top_score", "rerank_score", "final_score", "similarity_score"],
            default="N/A"
        )

        chunk_text = get_value_from_item(
            item,
            [
                "retrieval_text_stage04",
                "retrieval_text",
                "chunk_text",
                "text",
                "content",
                "document",
                "top_context_preview"
            ],
            default="N/A"
        )

        chunk_text = short_context(chunk_text, max_chars=max_chars_per_item)

        if chunk_text == "":
            continue

        block_lines = []
        block_lines.append(f"[السياق رقم {i}]")
        block_lines.append(f"نوع المصدر: {source_type}")

        if article_number != "N/A":
            block_lines.append(f"رقم المادة: {article_number}")

        if score != "N/A":
            block_lines.append(f"درجة الاسترجاع: {score}")

        block_lines.append("النص:")
        block_lines.append(chunk_text)

        formatted_blocks.append("\n".join(block_lines))

        context_meta.append({
            "rank": i,
            "source_type": source_type,
            "article_number": article_number,
            "score": score,
        })

    context_block = "\n\n" + ("-" * 70) + "\n\n"
    context_block = context_block.join(formatted_blocks)

    return context_block, context_meta


def get_primary_article_number(context_meta):
    for meta in context_meta:
        if meta.get("source_type") == "legal_article":
            article_number = clean_na(meta.get("article_number"))
            if article_number != "N/A":
                return article_number

    for meta in context_meta:
        article_number = clean_na(meta.get("article_number"))
        if article_number != "N/A":
            return article_number

    return "N/A"


# ---------------------------------------------------------
# 4) Prompt task detection
# ---------------------------------------------------------

def detect_prompt_task_type(question, retrieval_result=None, intent=None):
    q = clean_na(question)
    intent = clean_na(intent)

    if intent != "N/A":
        intent_lower = intent.lower()

        if "small" in intent_lower or "chat" in intent_lower or "greeting" in intent_lower:
            return "small_talk"

        if "out" in intent_lower or "scope" in intent_lower:
            return "out_of_scope"

        if "legal" in intent_lower:
            return "legal_article"

        if "faq" in intent_lower or "service" in intent_lower:
            return "faq"

    q_lower = q.lower().strip()

    small_talk_phrases = [
        "هلو",
        "السلام عليكم",
        "مرحبا",
        "أهلا",
        "اهلا",
        "كيفك",
        "كيف الحال",
        "وش تقدر تساعدني",
        "من أنت",
        "مين انت"
    ]

    if any(p in q_lower for p in small_talk_phrases):
        return "small_talk"

    context_items = extract_context_items(retrieval_result)

    has_legal = any(detect_source_type(item) == "legal_article" for item in context_items)
    has_faq = any(detect_source_type(item) == "faq" for item in context_items)

    if has_legal:
        return "legal_article"

    if has_faq:
        return "faq"

    return "general_rag"


# ---------------------------------------------------------
# 5) Main prompt builder
# ---------------------------------------------------------

def build_legal_rag_prompt(
    question,
    retrieval_result=None,
    intent=None,
    retrieval_reliable=True,
    max_context_items=5,
    max_chars_per_context=1800
):
    question = clean_na(question)

    context_block, context_meta = format_context_block(
        retrieval_result,
        max_items=max_context_items,
        max_chars_per_item=max_chars_per_context
    )

    primary_article_number = get_primary_article_number(context_meta)

    task_type = detect_prompt_task_type(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent
    )

    retrieval_reliable = bool(retrieval_reliable)

    # -----------------------------------------------------
    # Small talk
    # -----------------------------------------------------

    if task_type == "small_talk":
        return f"""
أنت مساعد صوتي عربي متخصص في نظام العمل السعودي وخدمات وزارة الموارد البشرية.

نوع السؤال: محادثة عامة.

السؤال:
{question}

تعليمات الإجابة:
- أجب بإجابة قصيرة وودودة.
- عرّف نفسك كمساعد متخصص في نظام العمل السعودي.
- لا تدخل في تفاصيل قانونية غير مطلوبة.
- لا تذكر مواد قانونية إلا إذا سأل المستخدم سؤالًا قانونيًا واضحًا.

الإجابة:
""".strip()

    # -----------------------------------------------------
    # Out of scope
    # -----------------------------------------------------

    if task_type == "out_of_scope":
        return f"""
أنت مساعد صوتي عربي متخصص فقط في نظام العمل السعودي وخدمات وزارة الموارد البشرية.

نوع السؤال: خارج النطاق.

السؤال:
{question}

تعليمات الإجابة:
- ارفض الإجابة بأدب.
- وضّح أن السؤال خارج نطاق نظام العمل السعودي وخدمات وزارة الموارد البشرية.
- لا تقدم فتوى أو نصيحة خارج المجال.
- اقترح أن يسأل المستخدم سؤالًا متعلقًا بنظام العمل السعودي.

الإجابة:
""".strip()

    # -----------------------------------------------------
    # No reliable context
    # -----------------------------------------------------

    if not retrieval_reliable or context_block.strip() == "":
        return f"""
أنت مساعد صوتي عربي متخصص في نظام العمل السعودي.

السؤال:
{question}

السياق المتاح:
لا يوجد سياق موثوق كافٍ للإجابة.

تعليمات إلزامية:
- لا تخترع إجابة.
- لا تذكر رقم مادة غير موجود.
- قل بوضوح: "المعلومات المتاحة غير كافية للإجابة بدقة."
- اطلب من المستخدم إعادة صياغة السؤال أو توضيحه.
- اجعل الإجابة قصيرة ومناسبة للصوت.

الإجابة:
""".strip()

    # -----------------------------------------------------
    # Legal article prompt
    # -----------------------------------------------------

    if task_type == "legal_article":

        if primary_article_number != "N/A":
            citation_instruction = f"""
تعليمات إلزامية للاستشهاد:
- يجب أن تبدأ الإجابة بهذه الصيغة:
  "وفقًا للمادة رقم {primary_article_number} من نظام العمل السعودي، ..."
- يجب ذكر رقم المادة {primary_article_number} صراحة داخل الإجابة.
- لا تذكر رقم مادة آخر إلا إذا كان ظاهرًا في السياق ويخدم السؤال مباشرة.
"""
            required_answer_format = f"وفقًا للمادة رقم {primary_article_number} من نظام العمل السعودي، [الإجابة المختصرة الدقيقة]."
        else:
            citation_instruction = """
تعليمات إلزامية للاستشهاد:
- إذا ظهر رقم مادة في السياق، يجب ذكره صراحة في الإجابة.
- لا تخترع رقم مادة غير موجود في السياق.
"""
            required_answer_format = "وفقًا للمادة رقم [رقم المادة الوارد في السياق] من نظام العمل السعودي، [الإجابة المختصرة الدقيقة]."

        return f"""
أنت مساعد صوتي عربي متخصص في نظام العمل السعودي.
مهمتك هي الإجابة عن السؤال اعتمادًا فقط على السياق الرسمي المسترجع.

السؤال:
{question}

السياق الرسمي المسترجع:
{context_block}

{citation_instruction}

تعليمات عامة إلزامية:
- أجب فقط من السياق المرفق.
- لا تستخدم معرفة عامة خارج السياق.
- لا تخترع مواد أو أحكام غير موجودة في السياق.
- إذا كان السياق لا يكفي للإجابة، قل: "المعلومات المتاحة غير كافية للإجابة بدقة."
- اجعل الإجابة مختصرة وواضحة ومناسبة للمساعد الصوتي.
- لا تستخدم جداول أو نقاط طويلة.
- لا تكرر السؤال.
- لا تذكر روابط.
- لا تستخدم عبارة "حسب السياق المرفق".
- إذا احتوت الإجابة على مدة أو شرط أو استثناء، اذكرها بوضوح.

صيغة الإجابة المطلوبة:
{required_answer_format}

الإجابة:
""".strip()

    # -----------------------------------------------------
    # FAQ / service prompt
    # -----------------------------------------------------

    if task_type == "faq":
        return f"""
أنت مساعد صوتي عربي متخصص في خدمات وزارة الموارد البشرية والتنمية الاجتماعية ونظام العمل السعودي.
مهمتك هي الإجابة اعتمادًا فقط على السياق الرسمي المسترجع.

السؤال:
{question}

السياق الرسمي المسترجع:
{context_block}

تعليمات إلزامية:
- أجب فقط من السياق المرفق.
- إذا كان السؤال خدميًا أو من FAQ، أجب بشكل مباشر ومختصر.
- لا تذكر رقم مادة قانونية إذا لم يكن السؤال متعلقًا بمادة من نظام العمل.
- إذا لم تكن المعلومات كافية، قل: "المعلومات المتاحة غير كافية للإجابة بدقة."
- لا تخترع روابط أو إجراءات غير موجودة في السياق.
- اجعل الإجابة مناسبة للمساعد الصوتي.
- لا تستخدم جداول.

الإجابة:
""".strip()

    # -----------------------------------------------------
    # General RAG prompt
    # -----------------------------------------------------

    return f"""
أنت مساعد صوتي عربي متخصص في نظام العمل السعودي وخدمات وزارة الموارد البشرية.
أجب عن السؤال اعتمادًا فقط على السياق الرسمي المسترجع.

السؤال:
{question}

السياق الرسمي المسترجع:
{context_block}

تعليمات إلزامية:
- أجب فقط من السياق المرفق.
- إذا كان السياق يحتوي على رقم مادة، اذكر رقم المادة صراحة.
- لا تخترع معلومة غير موجودة في السياق.
- إذا لم تكن المعلومات كافية، قل: "المعلومات المتاحة غير كافية للإجابة بدقة."
- اجعل الإجابة مختصرة وواضحة ومناسبة للصوت.

الإجابة:
""".strip()


# ---------------------------------------------------------
# 6) Chat messages builder
# ---------------------------------------------------------

def build_legal_rag_messages(
    question,
    retrieval_result=None,
    intent=None,
    retrieval_reliable=True,
    max_context_items=5,
    max_chars_per_context=1800
):
    prompt = build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable,
        max_context_items=max_context_items,
        max_chars_per_context=max_chars_per_context
    )

    system_message = """
أنت مساعد عربي موثوق ومتخصص في نظام العمل السعودي.
يجب أن تكون إجاباتك دقيقة، مختصرة، ومدعومة بالسياق الرسمي.
لا تخترع معلومات غير موجودة في السياق.
إذا كان السؤال قانونيًا والسياق يحتوي على رقم مادة، يجب ذكر رقم المادة صراحة.
""".strip()

    return [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": prompt
        }
    ]


# ---------------------------------------------------------
# 7) Backward-compatible aliases
# ---------------------------------------------------------

def build_llm_prompt(question, retrieval_result=None, intent=None, retrieval_reliable=True):
    return build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable
    )


def build_prompt_for_llm(question, retrieval_result=None, intent=None, retrieval_reliable=True):
    return build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable
    )


def build_generation_prompt(question, retrieval_result=None, intent=None, retrieval_reliable=True):
    return build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable
    )


def build_constrained_rag_prompt(question, retrieval_result=None, intent=None, retrieval_reliable=True):
    return build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable
    )


def build_rag_prompt(question, retrieval_result=None, intent=None, retrieval_reliable=True):
    return build_legal_rag_prompt(
        question=question,
        retrieval_result=retrieval_result,
        intent=intent,
        retrieval_reliable=retrieval_reliable
    )


# ---------------------------------------------------------
# 8) Confirmation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:14px 0;
    border-radius:6px;
">

✅ <b>تم تجهيز دوال بناء البرومبت بنجاح.</b><br>

هذه النسخة تجبر النموذج على ذكر رقم المادة القانونية عند توفرها في السياق، مما يساعد على تقليل أخطاء
<code>legal_citation_missing_or_wrong</code>.

</div>
"""))

print("Professional legal RAG prompt builder is ready.")
print("Available functions:")
print("- build_legal_rag_prompt")
print("- build_legal_rag_messages")
print("- build_llm_prompt")
print("- build_prompt_for_llm")
print("- build_generation_prompt")
print("- build_constrained_rag_prompt")
print("- build_rag_prompt")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
Stage 04.13 — بناء البرومبت المقيد بالسياق
</h2>

<p>
في هذه المرحلة يتم بناء برومبت احترافي لنموذج اللغة بحيث تكون الإجابة مقيدة بالسياق المسترجع من نظام RAG.
الهدف هو تقليل الهلوسة، وإجبار النموذج على ذكر رقم المادة القانونية عند توفرها في السياق.
</p>

</div>



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:14px 0;
    border-radius:6px;
">

✅ <b>تم تجهيز دوال بناء البرومبت بنجاح.</b><br>

هذه النسخة تجبر النموذج على ذكر رقم المادة القانونية عند توفرها في السياق، مما يساعد على تقليل أخطاء
<code>legal_citation_missing_or_wrong</code>.

</div>


Professional legal RAG prompt builder is ready.
Available functions:
- build_legal_rag_prompt
- build_legal_rag_messages
- build_llm_prompt
- build_prompt_for_llm
- build_generation_prompt
- build_constrained_rag_prompt
- build_rag_prompt


In [16]:
# ---------------------------------------------------------
# Stage 04.14 - Corrected Generation Function
# دالة توليد مصححة تستخدم البرومبت الجديد
# ---------------------------------------------------------

def generate_with_loaded_model(
    query,
    contexts=None,
    retrieval_reliable=True,
    intent=None,
    max_new_tokens=160
):
    """
    Generate answer using the currently loaded model and tokenizer.

    This version uses build_legal_rag_messages from Stage 04.13
    instead of the old build_messages function.
    """

    if "loaded_model" not in globals() or loaded_model is None:
        raise NameError("loaded_model غير موجود. شغّل load_llm(model_key) أولًا.")

    if "loaded_tokenizer" not in globals() or loaded_tokenizer is None:
        raise NameError("loaded_tokenizer غير موجود. شغّل load_llm(model_key) أولًا.")

    if "build_legal_rag_messages" not in globals():
        raise NameError("build_legal_rag_messages غير موجودة. شغّل Stage 04.13 أولًا.")

    # بناء الرسائل باستخدام البرومبت الجديد
    messages = build_legal_rag_messages(
        question=query,
        retrieval_result=contexts,
        intent=intent,
        retrieval_reliable=retrieval_reliable,
        max_context_items=3,
        max_chars_per_context=650
    )

    # استخدام chat template إذا كان مدعومًا
    if hasattr(loaded_tokenizer, "apply_chat_template"):
        input_text = loaded_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        input_text = ""
        for m in messages:
            input_text += f"{m['role']}: {m['content']}\n"
        input_text += "assistant: "

    inputs = loaded_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=3600
    ).to(loaded_model.device)

    start = time.time()

    with torch.no_grad():
        outputs = loaded_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=loaded_tokenizer.eos_token_id
        )

    gen_latency = time.time() - start

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = loaded_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return {
        "answer": answer,
        "generation_latency_sec": gen_latency,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "output_tokens": int(generated_ids.shape[-1]),
    }


# ---------------------------------------------------------
# Backward-compatible alias
# إذا كانت الخلايا التالية تستدعي generate_answer
# ---------------------------------------------------------

def generate_answer(query, contexts=None, retrieval_reliable=True, intent=None, max_new_tokens=160):
    return generate_with_loaded_model(
        query=query,
        contexts=contexts,
        retrieval_reliable=retrieval_reliable,
        intent=intent,
        max_new_tokens=max_new_tokens
    )


print("Corrected generation function is ready.")

# ---------------------------------------------------------
# Backward-compatible alias: Stage 04.15 pipeline calls generate_with_loaded_llm
# ---------------------------------------------------------
generate_with_loaded_llm = generate_with_loaded_model


Corrected generation function is ready.


In [17]:
# =========================================================
# Stage 04.15 - Full Question Answering Pipeline
# Intent → Retrieval → Reliability Gate → LLM
# =========================================================

def answer_question_with_model(query, model_key):
    total_start = time.time()
    intent = classify_intent(query)

    if intent == "small_talk":
        ans = small_talk_answer()
        return {
            "query": query,
            "model_key": model_key,
            "model_name": LOCAL_MODELS[model_key]["label"],
            "intent": intent,
            "retrieval_reliable": None,
            "top_score": None,
            "answer": ans,
            "retrieval_latency_sec": 0.0,
            "generation_latency_sec": 0.0,
            "input_tokens": 0,
            "output_tokens": len(ans.split()),
            "total_latency_sec": time.time() - total_start,
            "contexts": [],
        }

    if intent == "out_of_scope":
        ans = safe_refusal_answer()
        return {
            "query": query,
            "model_key": model_key,
            "model_name": LOCAL_MODELS[model_key]["label"],
            "intent": intent,
            "retrieval_reliable": False,
            "top_score": 0.0,
            "answer": ans,
            "retrieval_latency_sec": 0.0,
            "generation_latency_sec": 0.0,
            "input_tokens": 0,
            "output_tokens": len(ans.split()),
            "total_latency_sec": time.time() - total_start,
            "contexts": [],
        }

    retrieval = final_rag_retrieve_for_llm(query, top_k=TOP_K)

    if not retrieval["retrieval_reliable"]:
        ans = safe_refusal_answer()
        return {
            "query": query,
            "model_key": model_key,
            "model_name": LOCAL_MODELS[model_key]["label"],
            "intent": intent,
            "retrieval_reliable": False,
            "top_score": retrieval["top_score"],
            "answer": ans,
            "retrieval_latency_sec": retrieval["retrieval_latency_sec"],
            "generation_latency_sec": 0.0,
            "input_tokens": 0,
            "output_tokens": len(ans.split()),
            "total_latency_sec": time.time() - total_start,
            "contexts": retrieval["contexts"],
        }

    gen = generate_with_loaded_llm(query, retrieval["contexts"])

    return {
        "query": query,
        "model_key": model_key,
        "model_name": LOCAL_MODELS[model_key]["label"],
        "intent": intent,
        "retrieval_reliable": True,
        "top_score": retrieval["top_score"],
        "answer": gen["answer"],
        "retrieval_latency_sec": retrieval["retrieval_latency_sec"],
        "generation_latency_sec": gen["generation_latency_sec"],
        "input_tokens": gen["input_tokens"],
        "output_tokens": gen["output_tokens"],
        "total_latency_sec": time.time() - total_start,
        "contexts": retrieval["contexts"],
    }

print("Full QA pipeline ready.")

Full QA pipeline ready.


In [18]:
# =========================================================
# Safe LLM Loader for Qwen / ALLaM
# يمنع مشاكل الذاكرة قدر الإمكان أثناء تقييم النماذج
# =========================================================

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def clear_llm_memory():
    """
    تنظيف ذاكرة GPU و RAM قبل تحميل أي نموذج جديد.
    مهم جداً عند اختبار أكثر من LLM داخل نفس النوتبوك.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print("LLM memory cleared.")


def load_llm_model(model_path, use_4bit=True):
    """
    تحميل آمن للنموذج.
    
    use_4bit=True:
        مناسب جداً لـ Qwen 7B و ALLaM 7B لتقليل استهلاك الذاكرة.
    
    use_4bit=False:
        يستخدم FP16/BF16 وقد يحتاج ذاكرة GPU أكبر.
    """

    clear_llm_memory()

    print(f"Loading LLM: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        use_fast=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if use_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            quantization_config=quant_config,
            trust_remote_code=True,
            offload_buffers=True,
            low_cpu_mem_usage=True
        )

    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True,
            offload_buffers=True,
            low_cpu_mem_usage=True
        )

    model.eval()

    print("Loaded:", model_path)

    return model, tokenizer

# ---------------------------------------------------------
# Key-based loader used by the smoke test and evaluation cells.
# Loads a model by its LOCAL_MODELS key and exposes it through the
# global loaded_model / loaded_tokenizer used by the generation functions.
# ---------------------------------------------------------
def load_llm(model_key, use_4bit=True):
    global loaded_model, loaded_tokenizer, loaded_model_key
    cfg = LOCAL_MODELS[model_key]
    model_path = str(cfg["path"])
    try:
        model, tokenizer = load_llm_model(model_path, use_4bit=use_4bit)
    except Exception as e:
        print(f"4-bit load failed ({type(e).__name__}: {e}); retrying without quantization...")
        model, tokenizer = load_llm_model(model_path, use_4bit=False)
    loaded_model = model
    loaded_tokenizer = tokenizer
    loaded_model_key = model_key
    print(f"Active LLM set to: {cfg['label']} ({model_key})")
    return model, tokenizer


In [19]:
# =========================================================
# Stage 04.16 - Single Model Smoke Test
# شغّل هذه الخلية أولاً على نموذج واحد
# =========================================================

SMOKE_MODEL_KEY = "qwen_7b"  # غيّرها إلى allam_7b بعد نجاح Qwen

if not LOCAL_MODELS[SMOKE_MODEL_KEY]["path"].exists():
    print("Model path not found. Please fix LOCAL_MODELS path first:", LOCAL_MODELS[SMOKE_MODEL_KEY]["path"])
else:
    load_llm(SMOKE_MODEL_KEY)
    smoke_result = answer_question_with_model("كم مدة فترة التجربة في نظام العمل السعودي؟", SMOKE_MODEL_KEY)

    print("Model:", smoke_result["model_name"])
    print("Intent:", smoke_result["intent"])
    print("Reliable:", smoke_result["retrieval_reliable"])
    print("Top score:", smoke_result["top_score"])
    print("Answer:")
    print(smoke_result["answer"])
    print("Latency:", round(smoke_result["total_latency_sec"], 2), "sec")

LLM memory cleared.
Loading LLM: C:\models\Qwen2.5-7B-Instruct


W0621 10:23:03.873000 30664 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:00<01:51,  3.03it/s]

Loading weights:   1%|          | 2/339 [00:00<01:47,  3.14it/s]

...

Loading weights:  96%|█████████▌| 325/339 [00:04<00:00, 84.52it/s]

Loading weights:  99%|█████████▊| 334/339 [00:04<00:00, 76.82it/s]

Loading weights: 100%|██████████| 339/339 [00:04<00:00, 71.09it/s]

Loaded: C:\models\Qwen2.5-7B-Instruct
Active LLM set to: Qwen2.5-7B-Instruct (qwen_7b)


Model: Qwen2.5-7B-Instruct
Intent: legal_or_service
Reliable: True
Top score: 0.7494164943695067
Answer:
وفقًا للمادة رقم 54 من نظام العمل السعودي، إذا كان العامل خاضعاً لفترة تجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديدها بوضوح، بحيث لا تزيد على تسعين يوماً. ويجوز باتفاق مكتوب بين العامل وصاحب العمل تمديد فترة التجربة، على ألا تزيد على مائة وثمانين يوماً.
Latency: 4.89 sec


In [20]:
# =========================================================
# Stage 04.15B - Professional LLM Smoke Test Result Display
# عرض احترافي لنتيجة اختبار نموذج اللغة
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Safety checks
# ---------------------------------------------------------

if "SMOKE_MODEL_KEY" not in globals():
    raise NameError("SMOKE_MODEL_KEY غير موجود. شغّل خلية إعداد النموذج أولًا.")

if "answer_question_with_model" not in globals():
    raise NameError("answer_question_with_model غير موجودة. شغّل خلية دوال LLM أولًا.")

# ---------------------------------------------------------
# 1) Run smoke LLM test
# ---------------------------------------------------------

smoke_question = "كم مدة فترة التجربة في نظام العمل السعودي؟"

load_llm(SMOKE_MODEL_KEY)
smoke_result = answer_question_with_model(smoke_question, SMOKE_MODEL_KEY)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass
    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>"]:
        return "N/A"
    return value

def short_text(text, max_len=550):
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")

def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler

def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)

def professional_table(df, caption="", width="100%", font_size="13px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )
    return hide_index_safe(styler)

def color_bool_like(val):
    s = str(val).strip().lower()
    if s in ["true", "passed", "yes", "legal_or_service"]:
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if s in ["false", "failed", "no", "out_of_scope"]:
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""

# ---------------------------------------------------------
# 3) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
اختبار تجريبي لنموذج اللغة بعد ربطه بطبقة الاسترجاع
</h2>

<p>
في هذه الخطوة يتم اختبار نموذج اللغة باستخدام سؤال قانوني حقيقي:
<b>كم مدة فترة التجربة في نظام العمل السعودي؟</b>
</p>

<p>
الهدف من هذا الاختبار هو التأكد من أن النموذج يستطيع:
</p>

<ul>
    <li>فهم نية السؤال بشكل صحيح.</li>
    <li>الاعتماد على السياق المسترجع من نظام RAG.</li>
    <li>إنتاج إجابة قانونية واضحة ومناسبة للاستخدام.</li>
</ul>

</div>
"""))

# ---------------------------------------------------------
# 4) Summary result table
# ---------------------------------------------------------

model_name = clean_na(smoke_result.get("model_name"))
intent = clean_na(smoke_result.get("intent"))
retrieval_reliable = clean_na(smoke_result.get("retrieval_reliable"))
top_score = smoke_result.get("top_score", np.nan)
answer_text = clean_na(smoke_result.get("answer"))
latency = smoke_result.get("total_latency_sec", np.nan)

try:
    top_score_fmt = f"{float(top_score):.4f}"
except Exception:
    top_score_fmt = "N/A"

try:
    latency_fmt = f"{float(latency):.2f} sec"
except Exception:
    latency_fmt = "N/A"

llm_summary = pd.DataFrame([
    {
        "Metric": "Question",
        "Value": smoke_question,
        "Interpretation": "Legal question used for end-to-end LLM smoke testing"
    },
    {
        "Metric": "Model",
        "Value": model_name,
        "Interpretation": "Language model used to generate the answer"
    },
    {
        "Metric": "Detected Intent",
        "Value": intent,
        "Interpretation": "Intent classification result before final answer generation"
    },
    {
        "Metric": "Retrieval Reliable",
        "Value": retrieval_reliable,
        "Interpretation": "Whether the retrieved context passed the reliability gate"
    },
    {
        "Metric": "Top Retrieval Score",
        "Value": top_score_fmt,
        "Interpretation": "Highest retrieval confidence score used before generation"
    },
    {
        "Metric": "Total Latency",
        "Value": latency_fmt,
        "Interpretation": "End-to-end response time including retrieval and generation"
    }
])

summary_style = professional_table(
    llm_summary,
    caption="Table 4.14 — LLM Smoke Test Summary",
    width="100%"
)

summary_style = apply_style_compatible(summary_style, color_bool_like, subset=["Value"])
display(summary_style)

# ---------------------------------------------------------
# 5) Generated answer table
# ---------------------------------------------------------

answer_preview_df = pd.DataFrame([
    {
        "Field": "Generated Answer",
        "Content": short_text(answer_text, 800)
    }
])

answer_style = (
    answer_preview_df.style
    .set_caption("Table 4.15 — Generated Answer Preview")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "9px"),
                ("vertical-align", "middle"),
                ("font-size", "13px")
            ]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        }
    ])
)

answer_style = answer_style.set_properties(
    subset=["Content"],
    **{
        "text-align": "right",
        "direction": "rtl",
        "white-space": "normal",
        "line-height": "1.9"
    }
)

answer_style = answer_style.set_properties(
    subset=["Field"],
    **{
        "text-align": "center",
        "white-space": "normal"
    }
)

answer_style = hide_index_safe(answer_style)
display(answer_style)

# ---------------------------------------------------------
# 6) Final interpretation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تفسير النتيجة:</b><br>

تم تشغيل النموذج <code>{model_name}</code> بنجاح على سؤال قانوني مرتبط بفترة التجربة.  
إذا كانت قيمة <b>Retrieval Reliable = {retrieval_reliable}</b> فهذا يعني أن الإجابة تم بناؤها على سياق مسترجع موثوق.  
كما أن النموذج صنّف السؤال على أنه <code>{intent}</code>، وهو التصنيف المتوقع لهذا النوع من الأسئلة القانونية.

زمن الاستجابة الكلي بلغ <b>{latency_fmt}</b>، وهذا يمثل الأداء الكامل من لحظة استقبال السؤال إلى إنتاج الإجابة النهائية.

</div>
"""))

print("Professional LLM smoke test table generated successfully.")

LLM memory cleared.
Loading LLM: C:\models\Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:00<01:57,  2.89it/s]

Loading weights:   1%|          | 2/339 [00:00<01:48,  3.10it/s]

Loading weights:   2%|▏         | 6/339 [00:00<00:32, 10.29it/s]

...


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
اختبار تجريبي لنموذج اللغة بعد ربطه بطبقة الاسترجاع
</h2>

<p>
في هذه الخطوة يتم اختبار نموذج اللغة باستخدام سؤال قانوني حقيقي:
<b>كم مدة فترة التجربة في نظام العمل السعودي؟</b>
</p>

<p>
الهدف من هذا الاختبار هو التأكد من أن النموذج يستطيع:
</p>

<ul>
    <li>فهم نية السؤال بشكل صحيح.</li>
    <li>الاعتماد على السياق المسترجع من نظام RAG.</li>
    <li>إنتاج إجابة قانونية واضحة ومناسبة للاستخدام.</li>
</ul>

</div>


Metric,Value,Interpretation
Question,كم مدة فترة التجربة في نظام العمل السعودي؟,Legal question used for end-to-end LLM smoke testing
Model,Qwen2.5-7B-Instruct,Language model used to generate the answer
Detected Intent,legal_or_service,Intent classification result before final answer generation
Retrieval Reliable,True,Whether the retrieved context passed the reliability gate
Top Retrieval Score,0.7494,Highest retrieval confidence score used before generation
Total Latency,4.40 sec,End-to-end response time including retrieval and generation


Field,Content
Generated Answer,وفقًا للمادة رقم 54 من نظام العمل السعودي، إذا كان العامل خاضعاً لفترة تجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديدها بوضوح، بحيث لا تزيد على تسعين يوماً. ويجوز باتفاق مكتوب بين العامل وصاحب العمل تمديد فترة التجربة، على ألا تزيد على مائة وثمانين يوماً.



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تفسير النتيجة:</b><br>

تم تشغيل النموذج <code>Qwen2.5-7B-Instruct</code> بنجاح على سؤال قانوني مرتبط بفترة التجربة.  
إذا كانت قيمة <b>Retrieval Reliable = True</b> فهذا يعني أن الإجابة تم بناؤها على سياق مسترجع موثوق.  
كما أن النموذج صنّف السؤال على أنه <code>legal_or_service</code>، وهو التصنيف المتوقع لهذا النوع من الأسئلة القانونية.

زمن الاستجابة الكلي بلغ <b>4.40 sec</b>، وهذا يمثل الأداء الكامل من لحظة استقبال السؤال إلى إنتاج الإجابة النهائية.

</div>


Professional LLM smoke test table generated successfully.


In [21]:
# =========================================================
# Stage 04.17 - Build Stage 04 Evaluation Set
# Balanced Sampling + Pre-Sampling Quality Filter
# بناء مجموعة تقييم المرحلة الرابعة من كامل الداتا مع حذف الأسئلة غير المناسبة قبل الاختيار
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Configuration
# ---------------------------------------------------------

RANDOM_SEED = 42

# إذا تريد تشغيل كل الداتا كاملة اجعلها True
# تنبيه: لو عندك 238 سؤال ونموذجين، التشغيل سيكون طويلًا جدًا.
USE_FULL_DATASET_FOR_STAGE04 = False

# عدد الأسئلة المطلوبة من كل نوع عند استخدام العينة الموزونة
BALANCED_SAMPLE_PER_TYPE = {
    "faq_retrieval": 14,
    "legal_article_retrieval": 16,
    "out_of_scope": 10,
}

# ---------------------------------------------------------
# 1) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.17 — بناء مجموعة تقييم المرحلة الرابعة
</h2>

<p>
في هذه الخطوة يتم بناء مجموعة التقييم التي ستُستخدم لاختبار نماذج اللغة في المرحلة الرابعة.
بدل أخذ أول الأسئلة فقط من ملف التقييم، يتم اختيار عينة موزونة من كامل الداتا حسب نوع السؤال.
</p>

<p>
قبل اختيار العينة، يتم حذف الأسئلة العامة جدًا أو غير المناسبة لهدف المشروع، مثل الأسئلة المتعلقة بالمؤتمرات العامة أو طريقة كتابة الفاصلة العشرية.
هذا يجعل التقييم أكثر عدالة لأنه يركز على أسئلة نظام العمل السعودي وخدمات وزارة الموارد البشرية ذات العلاقة.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:12px;
    margin:14px 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">ملاحظة منهجية:</b>
يتم حذف الأسئلة الضعيفة قبل اختيار العينة، حتى لا تدخل في تقييم Qwen أو ALLaM.
أما أسئلة المحادثة العامة والأسئلة خارج النطاق المضافة يدويًا فهي مقصودة لاختبار طبقة التوجيه والرفض الآمن.
</div>

</div>
"""))

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    if pd.isna(value):
        return "N/A"
    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>"]:
        return "N/A"
    return value


def short_text(text, max_len=130):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def normalize_question_for_filter(text):
    """
    Normalize question text before quality filtering.
    """
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace("?", "؟")
    return text


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="13px", number_formats=None):
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "9px"),
                    ("white-space", "nowrap")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("vertical-align", "middle")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_eval_type(val):
    val = str(val)
    if val == "faq_retrieval":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"
    if val == "legal_article_retrieval":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if val == "out_of_scope":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    if val == "small_talk":
        return "background-color:#EAF3F8; color:#1F4E78; font-weight:bold;"
    return ""


def color_removed(val):
    if str(val).lower() in ["remove", "removed", "excluded"]:
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


# ---------------------------------------------------------
# 3) Extra router questions
# ---------------------------------------------------------

extra_router_questions = pd.DataFrame([
    {
        "eval_id": "smalltalk_001",
        "question": "هلو",
        "expected_answer": "",
        "eval_type": "small_talk",
        "gold_article_numbers": "",
        "is_out_of_scope": False,
    },
    {
        "eval_id": "smalltalk_002",
        "question": "وش تقدر تساعدني؟",
        "expected_answer": "",
        "eval_type": "small_talk",
        "gold_article_numbers": "",
        "is_out_of_scope": False,
    },
    {
        "eval_id": "oos_001",
        "question": "كم سعر البيت في الرياض؟",
        "expected_answer": "",
        "eval_type": "out_of_scope",
        "gold_article_numbers": "",
        "is_out_of_scope": True,
    },
    {
        "eval_id": "oos_002",
        "question": "ما حكم الطلاق؟",
        "expected_answer": "",
        "eval_type": "out_of_scope",
        "gold_article_numbers": "",
        "is_out_of_scope": True,
    },
    {
        "eval_id": "legal_uncertain_001",
        "question": "هل يحق للعامل الحصول على سيارة من الشركة؟",
        "expected_answer": "",
        "eval_type": "legal_article_retrieval",
        "gold_article_numbers": "",
        "is_out_of_scope": False,
    },
])

# ---------------------------------------------------------
# 4) Prepare full evaluation dataset
# ---------------------------------------------------------

if "df_eval" not in globals():
    raise NameError("df_eval غير موجود. شغّل خلية تحميل ملفات Stage 04 أولًا.")

df_eval_full = df_eval.copy()

if "eval_type" not in df_eval_full.columns:
    df_eval_full["eval_type"] = "unknown"

if "question" not in df_eval_full.columns:
    raise ValueError("Column 'question' is missing from df_eval.")

df_eval_full["eval_type"] = df_eval_full["eval_type"].fillna("unknown").astype(str)
df_eval_full["question"] = df_eval_full["question"].fillna("").astype(str)

# ---------------------------------------------------------
# 5) Remove low-quality questions before sampling
# ---------------------------------------------------------

MANUAL_EXCLUDE_QUESTIONS = {
    "ما هي العلاقة بين المؤتمر ورؤية المملكة 2030؟",
    "ما هي العلاقة بين المؤتمر ورؤية المملكة 2030?",
    "كيف أكتب الفاصلة العشرية حينما أريد كتابة المعدل؟",
    "كيف أكتب الفاصلة العشرية حينما أريد كتابة المعدل?",
}

LOW_VALUE_PHRASES = [
    "المؤتمر ورؤية المملكة 2030",
    "رؤية المملكة 2030",
    "مؤتمر العمل الدولي",
    "المؤتمر الدولي",
    "فعاليات مؤتمر",
    "الفاصلة العشرية",
    "كتابة المعدل",
    "حينما أريد كتابة المعدل",
]

def low_quality_reason(question):
    """
    Return reason if the question should be excluded, otherwise return empty string.
    """
    q = normalize_question_for_filter(question)

    manual_set = {normalize_question_for_filter(x) for x in MANUAL_EXCLUDE_QUESTIONS}

    if q in manual_set:
        return "Manual exclusion: unclear or low-value question"

    for phrase in LOW_VALUE_PHRASES:
        if phrase in q:
            return f"Low-value phrase: {phrase}"

    return ""


df_eval_full["_quality_reason"] = df_eval_full["question"].apply(low_quality_reason)
df_excluded_from_sampling = df_eval_full[df_eval_full["_quality_reason"] != ""].copy()

df_eval_full_clean = df_eval_full[df_eval_full["_quality_reason"] == ""].copy()
df_eval_full_clean = df_eval_full_clean.drop(columns=["_quality_reason"], errors="ignore").reset_index(drop=True)

# ---------------------------------------------------------
# 6) Display pre-sampling filter summary
# ---------------------------------------------------------

filter_summary = pd.DataFrame([
    {
        "Metric": "Original Evaluation Questions",
        "Count": len(df_eval_full),
        "Interpretation": "All questions before quality filtering"
    },
    {
        "Metric": "Excluded Before Sampling",
        "Count": len(df_excluded_from_sampling),
        "Interpretation": "General, unclear, or low-value questions removed before sampling"
    },
    {
        "Metric": "Remaining Questions for Sampling",
        "Count": len(df_eval_full_clean),
        "Interpretation": "Clean pool used for balanced Stage 04 sampling"
    }
])

display(
    professional_table(
        filter_summary,
        caption="Table 4.16 — Pre-Sampling Quality Filter Summary",
        width="90%"
    )
)

if len(df_excluded_from_sampling) > 0:
    excluded_preview_cols = ["eval_id", "question", "eval_type", "_quality_reason"]
    excluded_preview_cols = [c for c in excluded_preview_cols if c in df_excluded_from_sampling.columns]

    excluded_preview = df_excluded_from_sampling[excluded_preview_cols].copy()
    excluded_preview = excluded_preview.rename(columns={
        "eval_id": "Eval ID",
        "question": "Excluded Question",
        "eval_type": "Evaluation Type",
        "_quality_reason": "Exclusion Reason"
    })

    if "Excluded Question" in excluded_preview.columns:
        excluded_preview["Excluded Question"] = excluded_preview["Excluded Question"].apply(lambda x: short_text(x, 160))

    excluded_style = professional_table(
        excluded_preview,
        caption="Table 4.17 — Questions Excluded Before Sampling",
        width="100%"
    )

    if "Excluded Question" in excluded_preview.columns:
        excluded_style = excluded_style.set_properties(
            subset=["Excluded Question"],
            **{
                "text-align": "right",
                "direction": "rtl",
                "white-space": "normal",
                "line-height": "1.7"
            }
        )

    if "Evaluation Type" in excluded_preview.columns:
        excluded_style = apply_style_compatible(excluded_style, color_eval_type, subset=["Evaluation Type"])

    display(excluded_style)

# ---------------------------------------------------------
# 7) Select evaluation questions from the cleaned full dataset
# ---------------------------------------------------------

if USE_FULL_DATASET_FOR_STAGE04:
    df_eval_selected = df_eval_full_clean.copy()
    selection_mode = "Full cleaned evaluation dataset"
else:
    sampled_parts = []

    for eval_type, n in BALANCED_SAMPLE_PER_TYPE.items():
        subset = df_eval_full_clean[df_eval_full_clean["eval_type"] == eval_type].copy()

        if len(subset) == 0:
            continue

        sample_n = min(n, len(subset))

        sampled_parts.append(
            subset.sample(
                n=sample_n,
                random_state=RANDOM_SEED
            )
        )

    covered_types = set(BALANCED_SAMPLE_PER_TYPE.keys())
    remaining_types = [
        t for t in df_eval_full_clean["eval_type"].unique()
        if t not in covered_types
    ]

    for eval_type in remaining_types:
        subset = df_eval_full_clean[df_eval_full_clean["eval_type"] == eval_type].copy()
        if len(subset) > 0:
            sampled_parts.append(
                subset.sample(
                    n=min(2, len(subset)),
                    random_state=RANDOM_SEED
                )
            )

    if sampled_parts:
        df_eval_selected = pd.concat(sampled_parts, ignore_index=True, sort=False)
    else:
        df_eval_selected = df_eval_full_clean.head(20).copy()

    selection_mode = "Balanced sample from cleaned full evaluation dataset"

# ---------------------------------------------------------
# 8) Merge extra router questions with selected evaluation set
# ---------------------------------------------------------

df_stage04_eval = pd.concat(
    [extra_router_questions, df_eval_selected],
    ignore_index=True,
    sort=False
)

df_stage04_eval = (
    df_stage04_eval
    .drop_duplicates(subset=["question"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 9) Construction summary table
# ---------------------------------------------------------

construction_summary = pd.DataFrame([
    {
        "Component": "Original Evaluation Dataset",
        "Count": len(df_eval),
        "Description": "All available questions before filtering"
    },
    {
        "Component": "Cleaned Evaluation Pool",
        "Count": len(df_eval_full_clean),
        "Description": "Questions remaining after removing low-quality items"
    },
    {
        "Component": "Selected Evaluation Questions",
        "Count": len(df_eval_selected),
        "Description": selection_mode
    },
    {
        "Component": "Extra Router Questions",
        "Count": len(extra_router_questions),
        "Description": "Small talk, out-of-scope, and uncertain legal test questions"
    },
    {
        "Component": "Final Stage 04 Evaluation Set",
        "Count": len(df_stage04_eval),
        "Description": "Final deduplicated set used for LLM generation evaluation"
    }
])

display(
    professional_table(
        construction_summary,
        caption="Table 4.18 — Stage 04 Evaluation Set Construction Summary",
        width="100%"
    )
)

# ---------------------------------------------------------
# 10) Distribution table
# ---------------------------------------------------------

type_distribution = (
    df_stage04_eval["eval_type"]
    .fillna("unknown")
    .value_counts(dropna=False)
    .rename_axis("Evaluation Type")
    .reset_index(name="Number of Questions")
)

type_distribution["Percentage"] = (
    type_distribution["Number of Questions"] /
    type_distribution["Number of Questions"].sum()
)

type_style = professional_table(
    type_distribution,
    caption="Table 4.19 — Final Stage 04 Evaluation Distribution by Type",
    width="75%",
    number_formats={"Percentage": "{:.2%}"}
)

type_style = apply_style_compatible(type_style, color_eval_type, subset=["Evaluation Type"])
display(type_style)

# ---------------------------------------------------------
# 11) Professional preview table
# ---------------------------------------------------------

preview_cols = [
    "eval_id",
    "question",
    "eval_type",
    "gold_article_numbers",
    "is_out_of_scope"
]

available_preview_cols = [c for c in preview_cols if c in df_stage04_eval.columns]
df_stage04_preview = df_stage04_eval[available_preview_cols].copy()

df_stage04_preview = df_stage04_preview.rename(columns={
    "eval_id": "Eval ID",
    "question": "Question",
    "eval_type": "Evaluation Type",
    "gold_article_numbers": "Gold Article",
    "is_out_of_scope": "Out of Scope"
})

if "Question" in df_stage04_preview.columns:
    df_stage04_preview["Question"] = df_stage04_preview["Question"].apply(lambda x: short_text(x, 120))

if "Gold Article" in df_stage04_preview.columns:
    df_stage04_preview["Gold Article"] = df_stage04_preview["Gold Article"].apply(clean_na)

order_map = {
    "small_talk": 0,
    "out_of_scope": 1,
    "legal_article_retrieval": 2,
    "faq_retrieval": 3,
}

df_stage04_preview["_order"] = df_stage04_preview["Evaluation Type"].map(order_map).fillna(9)
df_stage04_preview = df_stage04_preview.sort_values(["_order", "Eval ID"]).drop(columns=["_order"])

df_stage04_preview = df_stage04_preview.head(30)

preview_style = (
    df_stage04_preview.style
    .set_caption("Table 4.20 — Preview of Final Stage 04 Evaluation Questions")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("color", "#1F4E78"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1F4E78"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #D9E2F3"),
                ("padding", "8px"),
                ("white-space", "nowrap")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #D9E2F3"),
                ("padding", "7px"),
                ("vertical-align", "middle"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [("background-color", "#F8FBFD")]
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [("background-color", "#FFFFFF")]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, Tahoma, sans-serif"),
                ("table-layout", "fixed"),
                ("margin", "12px 0")
            ]
        },
        {"selector": "th.col0", "props": [("width", "13%")]},
        {"selector": "th.col1", "props": [("width", "43%")]},
        {"selector": "th.col2", "props": [("width", "22%")]},
        {"selector": "th.col3", "props": [("width", "12%")]},
        {"selector": "th.col4", "props": [("width", "10%")]}
    ])
)

preview_style = apply_style_compatible(preview_style, color_eval_type, subset=["Evaluation Type"])

if "Question" in df_stage04_preview.columns:
    preview_style = preview_style.set_properties(
        subset=["Question"],
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "line-height": "1.7"
        }
    )

center_cols = [c for c in df_stage04_preview.columns if c != "Question"]
if center_cols:
    preview_style = preview_style.set_properties(
        subset=center_cols,
        **{
            "text-align": "center",
            "white-space": "normal"
        }
    )

preview_style = hide_index_safe(preview_style)
display(preview_style)

# ---------------------------------------------------------
# 12) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم بناء مجموعة تقييم المرحلة الرابعة بنجاح.</b><br>

طريقة الاختيار: <code>{selection_mode}</code><br>
عدد الأسئلة الأصلية قبل الفلترة: <b>{len(df_eval_full)}</b><br>
عدد الأسئلة المحذوفة قبل الاختيار: <b>{len(df_excluded_from_sampling)}</b><br>
عدد الأسئلة المتبقية للاختيار: <b>{len(df_eval_full_clean)}</b><br>
عدد الأسئلة المختارة من الداتا الأصلية: <b>{len(df_eval_selected)}</b><br>
عدد أسئلة التوجيه الإضافية: <b>{len(extra_router_questions)}</b><br>
إجمالي أسئلة التقييم النهائية: <b>{len(df_stage04_eval)}</b><br>

هذه الطريقة تمنع دخول الأسئلة العامة أو غير المناسبة إلى تقييم النماذج.

</div>
"""))

print("Stage 04 evaluation rows:", df_stage04_eval.shape)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المرحلة 04.17 — بناء مجموعة تقييم المرحلة الرابعة
</h2>

<p>
في هذه الخطوة يتم بناء مجموعة التقييم التي ستُستخدم لاختبار نماذج اللغة في المرحلة الرابعة.
بدل أخذ أول الأسئلة فقط من ملف التقييم، يتم اختيار عينة موزونة من كامل الداتا حسب نوع السؤال.
</p>

<p>
قبل اختيار العينة، يتم حذف الأسئلة العامة جدًا أو غير المناسبة لهدف المشروع، مثل الأسئلة المتعلقة بالمؤتمرات العامة أو طريقة كتابة الفاصلة العشرية.
هذا يجعل التقييم أكثر عدالة لأنه يركز على أسئلة نظام العمل السعودي وخدمات وزارة الموارد البشرية ذات العلاقة.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:12px;
    margin:14px 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">ملاحظة منهجية:</b>
يتم حذف الأسئلة الضعيفة قبل اختيار العينة، حتى لا تدخل في تقييم Qwen أو ALLaM.
أما أسئلة المحادثة العامة والأسئلة خارج النطاق المضافة يدويًا فهي مقصودة لاختبار طبقة التوجيه والرفض الآمن.
</div>

</div>


Metric,Count,Interpretation
Original Evaluation Questions,160,All questions before quality filtering
Excluded Before Sampling,0,"General, unclear, or low-value questions removed before sampling"
Remaining Questions for Sampling,160,Clean pool used for balanced Stage 04 sampling


Component,Count,Description
Original Evaluation Dataset,160,All available questions before filtering
Cleaned Evaluation Pool,160,Questions remaining after removing low-quality items
Selected Evaluation Questions,40,Balanced sample from cleaned full evaluation dataset
Extra Router Questions,5,"Small talk, out-of-scope, and uncertain legal test questions"
Final Stage 04 Evaluation Set,45,Final deduplicated set used for LLM generation evaluation


Evaluation Type,Number of Questions,Percentage
legal_article_retrieval,17,37.78%
faq_retrieval,14,31.11%
out_of_scope,12,26.67%
small_talk,2,4.44%


Eval ID,Question,Evaluation Type,Gold Article,Out of Scope
smalltalk_001,هلو,small_talk,N/A,False
smalltalk_002,وش تقدر تساعدني؟,small_talk,N/A,False
146,ما حكم الطلاق في الشريعة الإسلامية؟,out_of_scope,N/A,True
147,كم سعر العقار في الرياض؟,out_of_scope,N/A,True
148,ما أفضل سيارة عائلية؟,out_of_scope,N/A,True
150,ما علاج الصداع؟,out_of_scope,N/A,True
151,كم سعر الذهب اليوم؟,out_of_scope,N/A,True
154,ما شروط القرض العقاري؟,out_of_scope,N/A,True
155,ما رسوم الجمارك على السيارات؟,out_of_scope,N/A,True
157,ما أفضل مطعم في الرياض؟,out_of_scope,N/A,True



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
">

✅ <b>تم بناء مجموعة تقييم المرحلة الرابعة بنجاح.</b><br>

طريقة الاختيار: <code>Balanced sample from cleaned full evaluation dataset</code><br>
عدد الأسئلة الأصلية قبل الفلترة: <b>160</b><br>
عدد الأسئلة المحذوفة قبل الاختيار: <b>0</b><br>
عدد الأسئلة المتبقية للاختيار: <b>160</b><br>
عدد الأسئلة المختارة من الداتا الأصلية: <b>40</b><br>
عدد أسئلة التوجيه الإضافية: <b>5</b><br>
إجمالي أسئلة التقييم النهائية: <b>45</b><br>

هذه الطريقة تمنع دخول الأسئلة العامة أو غير المناسبة إلى تقييم النماذج.

</div>


Stage 04 evaluation rows: (45, 25)


In [22]:
# =========================================================
# Stage 04.18 - Run Generation Evaluation
# تشغيل تقييم نماذج اللغة على أسئلة المرحلة الرابعة
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
import time
import json
from pathlib import Path

# ---------------------------------------------------------
# 0) Configuration
# ---------------------------------------------------------

STAGE04_EVAL_OUTPUT_DIR = Path("stage04_llm_generation_outputs")
STAGE04_EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_EVERY_N_ROWS = 5

# مهم:
# False = تقييم عادل: صحة الإجابة لا تفشل فقط بسبب عدم ذكر رقم المادة
# True  = تقييم صارم: السؤال القانوني يعتبر صحيح فقط إذا ذكر رقم المادة الصحيحة
STRICT_LEGAL_EVALUATION = False

# حد تقريبي لاعتبار الإجابة مدعومة بالسياق
GROUNDEDNESS_THRESHOLD = 0.12

# ---------------------------------------------------------
# 1) Safety checks
# ---------------------------------------------------------

if "df_stage04_eval" not in globals():
    raise NameError("df_stage04_eval غير موجود. شغّل Stage 04.17 أولًا.")

if "answer_question_with_model" not in globals() or not callable(answer_question_with_model):
    raise NameError(
        "answer_question_with_model غير موجودة. شغّل Stage 04.15 - Full QA Pipeline أولًا."
    )

# إذا لم تكن MODEL_CONFIGS موجودة، ننشئها من المسارات المعروفة
if "MODEL_CONFIGS" not in globals() or not MODEL_CONFIGS:
    MODEL_CONFIGS = [
        {
            "model_key": "allam_7b",
            "model_name": "ALLaM-7B-Instruct-preview",
            "model_path": r"C:\models\ALLaM-7B-Instruct-preview"
        },
        {
            "model_key": "qwen_7b",
            "model_name": "Qwen2.5-7B-Instruct",
            "model_path": r"C:\models\Qwen2.5-7B-Instruct"
        }
    ]

if isinstance(MODEL_CONFIGS, dict):
    MODEL_CONFIGS = [MODEL_CONFIGS]

# ---------------------------------------------------------
# 2) Academic explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
Stage 04.18 — تشغيل تقييم نماذج اللغة
</h2>

<p>
في هذه المرحلة يتم تمرير نفس أسئلة التقييم إلى كل نموذج لغة، ثم يتم حساب مجموعة من المقاييس:
صحة الإجابة، دقة الاستشهاد بالمادة القانونية، دقة المادة المسترجعة، الارتباط بالسياق، مؤشر الهلوسة التقريبي،
ملاءمة الإجابة للصوت، وزمن الاستجابة.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:12px;
    margin:14px 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">ملاحظة مهمة:</b>
تم فصل <code>answer_correct</code> عن <code>citation_hit</code> حتى لا يتم اعتبار الإجابة خاطئة فقط لأنها لم تذكر رقم المادة،
بينما يتم الاحتفاظ بمقياس <code>strict_legal_correct</code> للتقييم القانوني الصارم.
</div>

</div>
"""))

# ---------------------------------------------------------
# 3) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass
    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return "N/A"
    return value


def normalize_arabic_digits(text):
    text = str(text)
    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"

    trans = {}
    for a, e in zip(arabic_digits, english_digits):
        trans[ord(a)] = e
    for p, e in zip(persian_digits, english_digits):
        trans[ord(p)] = e

    return text.translate(trans)


def normalize_article_set(value):
    """
    Examples:
    53.0         -> {"53"}
    "53.0"       -> {"53"}
    "53, 0"      -> {"53"}
    "المادة 53" -> {"53"}
    """
    if value is None:
        return set()

    if isinstance(value, (list, tuple, set)):
        result = set()
        for item in value:
            result |= normalize_article_set(item)
        return result

    try:
        if pd.isna(value):
            return set()
    except Exception:
        pass

    text = normalize_arabic_digits(str(value)).strip()

    if text.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return set()

    # Convert 53.0 or 53,0 into 53
    text = re.sub(r"(\d+)\s*[\.,]\s*0+\b", r"\1", text)

    nums = re.findall(r"\d+", text)

    result = set()
    for n in nums:
        try:
            result.add(str(int(float(n))))
        except Exception:
            result.add(str(n))

    return result


def article_set_to_text(article_set):
    if not article_set:
        return "N/A"
    try:
        return ", ".join(sorted(article_set, key=lambda x: int(x)))
    except Exception:
        return ", ".join(sorted(article_set))


def extract_cited_article_numbers(answer_text):
    """
    Extract article numbers explicitly cited in the model answer.
    """
    answer_text = clean_na(answer_text)
    if answer_text == "N/A":
        return set()

    text = normalize_arabic_digits(answer_text)
    text = re.sub(r"(\d+)\s*[\.,]\s*0+\b", r"\1", text)

    patterns = [
        r"(?:المادة|مادة)\s*(?:رقم\s*)?(\d+)",
        r"وفقًا\s+للمادة\s*(?:رقم\s*)?(\d+)",
        r"وفقا\s+للمادة\s*(?:رقم\s*)?(\d+)",
    ]

    cited = set()

    for pattern in patterns:
        matches = re.findall(pattern, text)
        for m in matches:
            try:
                cited.add(str(int(m)))
            except Exception:
                cited.add(str(m))

    return cited


def compute_hit(gold_set, predicted_set):
    """
    Returns:
    1 if there is overlap
    0 if gold exists but no overlap
    NaN if gold does not exist
    """
    if not gold_set:
        return np.nan
    return int(len(gold_set & predicted_set) > 0)


def get_gold_article_set(row):
    for col in ["gold_article_numbers_parsed", "gold_article_numbers", "gold_article_number", "expected_article_number"]:
        if col in row.index:
            gold = normalize_article_set(row.get(col))
            if gold:
                return gold
    return set()


def get_eval_type(row):
    for col in ["eval_type", "question_type", "type"]:
        if col in row.index:
            value = clean_na(row.get(col))
            if value != "N/A":
                return value
    return "unknown"


def to_bool(value, default=False):
    if isinstance(value, bool):
        return value

    value = clean_na(value).lower()

    if value in ["true", "1", "yes", "y"]:
        return True
    if value in ["false", "0", "no", "n", "n/a"]:
        return False

    return default


def to_float(value, default=np.nan):
    try:
        if value is None:
            return default
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def short_text(text, max_len=240):
    text = clean_na(text)
    if text == "N/A":
        return "N/A"
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def get_first_value(d, keys, default="N/A"):
    if not isinstance(d, dict):
        return default

    for key in keys:
        if key in d:
            value = clean_na(d.get(key))
            if value != "N/A":
                return value

    return default


def result_to_dict(result):
    if isinstance(result, dict):
        return result
    if isinstance(result, str):
        return {"answer": result}
    return {"answer": str(result)}


def extract_context_items(obj):
    """
    Extract retrieved contexts from different possible result formats.
    """
    if obj is None:
        return []

    if isinstance(obj, list):
        return obj

    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")

    if isinstance(obj, str):
        return [{"chunk_text": obj}]

    if isinstance(obj, dict):
        possible_keys = [
            "contexts",
            "context",
            "top_contexts",
            "retrieved_contexts",
            "results",
            "top_results",
            "documents",
            "retrieved_docs",
            "chunks",
            "top_chunks"
        ]

        for key in possible_keys:
            if key in obj:
                value = obj[key]

                if isinstance(value, pd.DataFrame):
                    return value.to_dict(orient="records")

                if isinstance(value, list):
                    return value

                if isinstance(value, str):
                    return [{"chunk_text": value}]

        # Direct context keys
        for key in ["top_context_preview", "context_text", "combined_context", "retrieved_context"]:
            if key in obj:
                value = clean_na(obj.get(key))
                if value != "N/A":
                    return [{"chunk_text": value}]

    return []


def item_get(item, keys, default="N/A"):
    if isinstance(item, str):
        return item

    if not isinstance(item, dict):
        return default

    for key in keys:
        if key in item:
            value = clean_na(item.get(key))
            if value != "N/A":
                return value

    return default


def extract_top_article_number(result_dict, retrieval_result=None):
    direct = get_first_value(
        result_dict,
        ["top_article_number", "retrieved_article", "top_article", "article_number", "article_number_int"],
        default="N/A"
    )

    if direct != "N/A":
        s = normalize_article_set(direct)
        return article_set_to_text(s)

    items = extract_context_items(retrieval_result if retrieval_result is not None else result_dict)

    if items:
        first = items[0]
        value = item_get(
            first,
            ["top_article_number", "retrieved_article", "top_article", "article_number", "article_number_int"],
            default="N/A"
        )
        s = normalize_article_set(value)
        if s:
            return article_set_to_text(s)

        text = item_get(
            first,
            ["retrieval_text_stage04", "retrieval_text", "chunk_text", "text", "content", "document", "top_context_preview"],
            default="N/A"
        )
        s = normalize_article_set(text)
        if s:
            return article_set_to_text(s)

    return "N/A"


def extract_top_score(result_dict, retrieval_result=None):
    direct = get_first_value(
        result_dict,
        ["top_score", "score", "final_score", "rerank_score", "similarity_score"],
        default="N/A"
    )

    if direct != "N/A":
        return to_float(direct)

    items = extract_context_items(retrieval_result if retrieval_result is not None else result_dict)

    if items:
        first = items[0]
        value = item_get(
            first,
            ["top_score", "score", "final_score", "rerank_score", "similarity_score"],
            default="N/A"
        )
        return to_float(value)

    return np.nan


def extract_top_source_type(result_dict, retrieval_result=None):
    direct = get_first_value(
        result_dict,
        ["top_source_type", "source_type", "document_type"],
        default="N/A"
    )

    if direct != "N/A":
        return direct

    items = extract_context_items(retrieval_result if retrieval_result is not None else result_dict)

    if items:
        first = items[0]
        return item_get(first, ["source_type", "top_source_type", "document_type"], default="N/A")

    return "N/A"


def extract_context_text(result_dict, retrieval_result=None):
    direct = get_first_value(
        result_dict,
        ["top_context_preview", "context", "retrieved_context", "combined_context", "context_text"],
        default="N/A"
    )

    if direct != "N/A":
        return direct

    items = extract_context_items(retrieval_result if retrieval_result is not None else result_dict)

    context_parts = []

    for item in items[:5]:
        txt = item_get(
            item,
            ["retrieval_text_stage04", "retrieval_text", "chunk_text", "text", "content", "document", "top_context_preview"],
            default="N/A"
        )
        if txt != "N/A":
            context_parts.append(str(txt))

    if context_parts:
        return "\n\n".join(context_parts)

    return "N/A"


def arabic_word_set(text):
    text = clean_na(text)
    if text == "N/A":
        return set()

    text = normalize_arabic_digits(text)
    words = re.findall(r"[\u0600-\u06FFa-zA-Z0-9]{3,}", text)
    stop = {
        "هذا", "هذه", "ذلك", "التي", "الذي", "على", "إلى", "الى", "عن",
        "في", "من", "ما", "لا", "لم", "لن", "أن", "إن", "كان", "كانت",
        "يكون", "يجب", "حسب", "وفقًا", "وفقا", "النظام", "السعودي"
    }
    return {w for w in words if w not in stop}


def lexical_groundedness(answer, context):
    a_words = arabic_word_set(answer)
    c_words = arabic_word_set(context)

    if not a_words or not c_words:
        return 0.0

    return len(a_words & c_words) / max(len(a_words), 1)


def detect_refusal(answer):
    answer = clean_na(answer)

    if answer == "N/A":
        return False

    patterns = [
        "المعلومات المتاحة غير كافية",
        "غير كافية",
        "لا يوجد سياق",
        "لا أستطيع",
        "لا يمكنني",
        "لا أملك",
        "خارج نطاق",
        "خارج النطاق",
        "ليس ضمن نطاق",
        "غير مرتبط بنظام العمل",
        "يرجى إعادة صياغة",
        "أحتاج إلى توضيح"
    ]

    return any(p in answer for p in patterns)


def voice_suitability_score(answer):
    answer = clean_na(answer)

    if answer == "N/A":
        return 0.0

    length = len(answer)

    score = 1.0

    if length < 25:
        score -= 0.15
    elif length <= 450:
        score += 0.0
    elif length <= 750:
        score -= 0.15
    else:
        score -= 0.30

    if answer.count("\n") > 6:
        score -= 0.10

    if "|" in answer or "<table" in answer.lower():
        score -= 0.20

    if "وفقًا للمادة" in answer or "وفقا للمادة" in answer or "المادة رقم" in answer:
        score += 0.05

    return max(0.0, min(1.0, score))


def estimate_output_tokens(answer):
    answer = clean_na(answer)
    if answer == "N/A":
        return 0
    return max(1, len(answer.split()))


def score_answer_correct(
    eval_type,
    answer,
    refusal,
    gold_set,
    citation_hit,
    retrieved_article_hit,
    groundedness,
    expected_answer=""
):
    """
    Fair content-based evaluation.
    Citation is measured separately using citation_hit and strict_legal_correct.
    """
    answer = clean_na(answer)
    expected_answer = clean_na(expected_answer)

    if answer == "N/A" or len(answer) < 5:
        return 0

    eval_type = str(eval_type)

    # Small talk: any useful short response is acceptable
    if eval_type == "small_talk":
        return int(not refusal and len(answer) >= 5)

    # Out of scope: refusal is the correct behavior
    if eval_type == "out_of_scope":
        return int(refusal)

    # FAQ: answer should be grounded or at least not a refusal
    if eval_type == "faq_retrieval":
        if refusal:
            return 0

        if groundedness >= GROUNDEDNESS_THRESHOLD:
            return 1

        # If expected answer exists, compare answer with expected answer
        if expected_answer != "N/A":
            ref_overlap = lexical_groundedness(answer, expected_answer)
            if ref_overlap >= 0.10:
                return 1

        return 0

    # Legal questions
    if eval_type == "legal_article_retrieval":
        if refusal:
            return 0

        # If strict evaluation is active, require citation hit
        if STRICT_LEGAL_EVALUATION and gold_set:
            return int(citation_hit == 1)

        # Fair content evaluation
        if citation_hit == 1:
            return 1

        if retrieved_article_hit == 1 and groundedness >= 0.08:
            return 1

        if expected_answer != "N/A":
            ref_overlap = lexical_groundedness(answer, expected_answer)
            if ref_overlap >= 0.10:
                return 1

        return 0

    # General fallback
    if refusal:
        return 0

    return int(groundedness >= GROUNDEDNESS_THRESHOLD)


def call_answer_function(question, model_key, model_config):
    """
    Compatible call wrapper for answer_question_with_model.
    """
    try:
        return answer_question_with_model(question, model_key)
    except TypeError:
        pass

    try:
        return answer_question_with_model(question=question, model_key=model_key)
    except TypeError:
        pass

    try:
        return answer_question_with_model(question, model_config)
    except TypeError:
        pass

    try:
        return answer_question_with_model(question=question, model_config=model_config)
    except TypeError:
        pass

    # Last fallback
    return answer_question_with_model(question)


def retrieve_if_needed(question, result_dict):
    """
    Use retrieval already returned by the pipeline if available.
    Otherwise call final_rag_retrieve_for_llm if it exists.
    """
    for key in ["retrieval_result", "retrieval", "retrieved", "rag_result"]:
        if key in result_dict:
            return result_dict.get(key)

    # If result itself contains useful retrieval keys, use it
    if any(k in result_dict for k in ["top_article_number", "top_score", "contexts", "top_contexts", "top_context_preview"]):
        return result_dict

    # Fallback: call final retrieval function if available
    fn = globals().get("final_rag_retrieve_for_llm")

    if callable(fn):
        try:
            return fn(question)
        except Exception:
            return None

    return None


# ---------------------------------------------------------
# 4) Run evaluation
# ---------------------------------------------------------

records = []

total_questions = len(df_stage04_eval)
total_runs = total_questions * len(MODEL_CONFIGS)

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

سيتم تشغيل <b>{len(MODEL_CONFIGS)}</b> نموذج على <b>{total_questions}</b> سؤال.
إجمالي عمليات التوليد: <b>{total_runs}</b>.

</div>
"""))

run_counter = 0

for model_config in MODEL_CONFIGS:
    model_key = model_config.get("model_key", "unknown_model")
    model_name = model_config.get("model_name", model_key)

    print("=" * 100)
    print(f"Evaluating model: {model_key} | {model_name}")
    print("=" * 100)

    # Load model once if load_llm exists
    if "load_llm" in globals() and callable(load_llm):
        try:
            load_llm(model_key)
        except Exception as e:
            print(f"Warning: load_llm({model_key}) failed or model already loaded.")
            print("Reason:", repr(e))

    for idx, row in df_stage04_eval.iterrows():
        question = clean_na(row.get("question"))
        eval_id = clean_na(row.get("eval_id", idx + 1))
        eval_type = get_eval_type(row)
        expected_answer = clean_na(row.get("expected_answer", ""))

        run_counter += 1

        print(f"[{idx + 1}/{total_questions}] {model_key} | {question[:90]}")

        start_time = time.time()

        try:
            raw_result = call_answer_function(
                question=question,
                model_key=model_key,
                model_config=model_config
            )

            result_dict = result_to_dict(raw_result)
            runtime_error = ""

        except Exception as e:
            result_dict = {"answer": ""}
            runtime_error = repr(e)

        wall_latency = time.time() - start_time

        retrieval_result = retrieve_if_needed(question, result_dict)

        answer = clean_na(
            get_first_value(
                result_dict,
                ["answer", "final_answer", "response", "generated_answer", "output"],
                default=""
            )
        )

        if answer == "N/A":
            answer = ""

        intent = get_first_value(result_dict, ["intent", "detected_intent", "route"], default=eval_type)

        retrieval_reliable = to_bool(
            get_first_value(result_dict, ["retrieval_reliable", "is_reliable", "reliable"], default=True),
            default=True
        )

        top_article_number = extract_top_article_number(result_dict, retrieval_result)
        top_score = extract_top_score(result_dict, retrieval_result)
        top_source_type = extract_top_source_type(result_dict, retrieval_result)
        context_text = extract_context_text(result_dict, retrieval_result)
        top_context_preview = short_text(context_text, 500)

        gold_set = get_gold_article_set(row)
        retrieved_set = normalize_article_set(top_article_number)
        cited_set = extract_cited_article_numbers(answer)

        citation_hit = compute_hit(gold_set, cited_set)
        retrieved_article_hit = compute_hit(gold_set, retrieved_set)

        groundedness = lexical_groundedness(answer, context_text)

        refusal = detect_refusal(answer)
        answered = int(len(clean_na(answer)) >= 5 and not refusal and runtime_error == "")

        answer_correct = score_answer_correct(
            eval_type=eval_type,
            answer=answer,
            refusal=refusal,
            gold_set=gold_set,
            citation_hit=citation_hit,
            retrieved_article_hit=retrieved_article_hit,
            groundedness=groundedness,
            expected_answer=expected_answer
        )

        # Strict legal correctness keeps citation requirement separate
        if eval_type == "legal_article_retrieval" and gold_set:
            strict_legal_correct = int(answer_correct == 1 and citation_hit == 1)
        else:
            strict_legal_correct = answer_correct

        hallucination_proxy = int(
            answered == 1
            and groundedness < GROUNDEDNESS_THRESHOLD
            and citation_hit != 1
        )

        total_latency_sec = to_float(
            get_first_value(result_dict, ["total_latency_sec", "latency_sec", "latency_seconds"], default=np.nan)
        )

        generation_latency_sec = to_float(
            get_first_value(result_dict, ["generation_latency_sec", "generation_latency", "llm_latency_sec"], default=np.nan)
        )

        retrieval_latency_sec = to_float(
            get_first_value(result_dict, ["retrieval_latency_sec", "retrieval_latency"], default=np.nan)
        )

        if pd.isna(total_latency_sec):
            total_latency_sec = wall_latency

        output_tokens = get_first_value(result_dict, ["output_tokens", "generated_tokens", "new_tokens"], default="N/A")

        if output_tokens == "N/A":
            output_tokens = estimate_output_tokens(answer)
        else:
            try:
                output_tokens = int(float(output_tokens))
            except Exception:
                output_tokens = estimate_output_tokens(answer)

        tokens_per_sec = get_first_value(result_dict, ["tokens_per_sec", "tokens_per_second"], default="N/A")

        if tokens_per_sec == "N/A":
            tokens_per_sec = output_tokens / generation_latency_sec if generation_latency_sec and not pd.isna(generation_latency_sec) and generation_latency_sec > 0 else np.nan
        else:
            tokens_per_sec = to_float(tokens_per_sec)

        record = {
            "model_key": model_key,
            "model_name": model_name,
            "eval_id": eval_id,
            "question_index": idx + 1,
            "question": question,
            "eval_type": eval_type,
            "expected_answer": expected_answer,
            "is_out_of_scope": row.get("is_out_of_scope", np.nan),
            "intent": intent,

            "gold_article_numbers": row.get("gold_article_numbers", ""),
            "gold_article_numbers_parsed": article_set_to_text(gold_set),
            "top_article_number": top_article_number,
            "cited_article_numbers": article_set_to_text(cited_set),

            "retrieved_article_hit": retrieved_article_hit,
            "citation_hit": citation_hit,

            "retrieval_reliable": retrieval_reliable,
            "top_score": top_score,
            "top_source_type": top_source_type,
            "top_context_preview": top_context_preview,

            "answer": answer,
            "answered": answered,
            "refusal": int(refusal),
            "answer_correct": int(answer_correct),
            "strict_legal_correct": int(strict_legal_correct),

            "lexical_groundedness_proxy": round(float(groundedness), 4),
            "hallucination_proxy": int(hallucination_proxy),
            "voice_suitability_score": round(float(voice_suitability_score(answer)), 4),

            "total_latency_sec": round(float(total_latency_sec), 4) if not pd.isna(total_latency_sec) else np.nan,
            "generation_latency_sec": round(float(generation_latency_sec), 4) if not pd.isna(generation_latency_sec) else np.nan,
            "retrieval_latency_sec": round(float(retrieval_latency_sec), 4) if not pd.isna(retrieval_latency_sec) else np.nan,
            "output_tokens": output_tokens,
            "tokens_per_sec": round(float(tokens_per_sec), 4) if not pd.isna(tokens_per_sec) else np.nan,

            "runtime_error": runtime_error,
        }

        records.append(record)

        # Save checkpoint
        if len(records) % SAVE_EVERY_N_ROWS == 0:
            checkpoint_df = pd.DataFrame(records)
            checkpoint_path = STAGE04_EVAL_OUTPUT_DIR / "df_generation_eval_checkpoint.csv"
            checkpoint_df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

    # Optional memory cleanup after each model
    if "clear_llm_memory" in globals() and callable(clear_llm_memory):
        try:
            clear_llm_memory()
        except Exception:
            pass

# ---------------------------------------------------------
# 5) Build final dataframe
# ---------------------------------------------------------

df_generation_eval = pd.DataFrame(records)

# Ensure numeric columns
numeric_cols = [
    "retrieved_article_hit",
    "citation_hit",
    "answered",
    "refusal",
    "answer_correct",
    "strict_legal_correct",
    "lexical_groundedness_proxy",
    "hallucination_proxy",
    "voice_suitability_score",
    "top_score",
    "total_latency_sec",
    "generation_latency_sec",
    "retrieval_latency_sec",
    "output_tokens",
    "tokens_per_sec"
]

for col in numeric_cols:
    if col in df_generation_eval.columns:
        df_generation_eval[col] = pd.to_numeric(df_generation_eval[col], errors="coerce")

# ---------------------------------------------------------
# 6) Save final outputs
# ---------------------------------------------------------

csv_path = STAGE04_EVAL_OUTPUT_DIR / "df_generation_eval.csv"
xlsx_path = STAGE04_EVAL_OUTPUT_DIR / "df_generation_eval.xlsx"
json_path = STAGE04_EVAL_OUTPUT_DIR / "df_generation_eval.json"

df_generation_eval.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_generation_eval.to_excel(xlsx_path, index=False)
df_generation_eval.to_json(json_path, orient="records", force_ascii=False, indent=2)

# ---------------------------------------------------------
# 7) Summary display
# ---------------------------------------------------------

summary = (
    df_generation_eval
    .groupby(["model_key", "model_name"], dropna=False)
    .agg(
        questions=("question", "count"),
        answer_accuracy=("answer_correct", "mean"),
        strict_legal_accuracy=("strict_legal_correct", "mean"),
        citation_hit_rate=("citation_hit", "mean"),
        retrieved_article_hit_rate=("retrieved_article_hit", "mean"),
        answered_rate=("answered", "mean"),
        refusal_rate=("refusal", "mean"),
        avg_groundedness=("lexical_groundedness_proxy", "mean"),
        hallucination_proxy_rate=("hallucination_proxy", "mean"),
        avg_voice_suitability=("voice_suitability_score", "mean"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
        avg_generation_latency_sec=("generation_latency_sec", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
    )
    .reset_index()
)

for col in summary.select_dtypes(include=[np.number]).columns:
    summary[col] = summary[col].round(4)

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

✅ <b>تم الانتهاء من تقييم نماذج اللغة.</b><br>

تم إنشاء جدول <code>df_generation_eval</code> وحفظ النتائج في مجلد
<code>stage04_llm_generation_outputs</code>.

</div>
"""))



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
Stage 04.18 — تشغيل تقييم نماذج اللغة
</h2>

<p>
في هذه المرحلة يتم تمرير نفس أسئلة التقييم إلى كل نموذج لغة، ثم يتم حساب مجموعة من المقاييس:
صحة الإجابة، دقة الاستشهاد بالمادة القانونية، دقة المادة المسترجعة، الارتباط بالسياق، مؤشر الهلوسة التقريبي،
ملاءمة الإجابة للصوت، وزمن الاستجابة.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:12px;
    margin:14px 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">ملاحظة مهمة:</b>
تم فصل <code>answer_correct</code> عن <code>citation_hit</code> حتى لا يتم اعتبار الإجابة خاطئة فقط لأنها لم تذكر رقم المادة،
بينما يتم الاحتفاظ بمقياس <code>strict_legal_correct</code> للتقييم القانوني الصارم.
</div>

</div>



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

سيتم تشغيل <b>2</b> نموذج على <b>45</b> سؤال.
إجمالي عمليات التوليد: <b>90</b>.

</div>


Evaluating model: allam_7b | ALLaM-7B-Instruct-preview


LLM memory cleared.
Loading LLM: C:\models\ALLaM-7B-Instruct-preview


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

[34/45] qwen_7b | هل تختلف ساعات العمل في شهر رمضان؟


[35/45] qwen_7b | ما أحكام السلامة والصحة المهنية في بيئة العمل؟
[36/45] qwen_7b | ما رسوم الجمارك على السيارات؟


[37/45] qwen_7b | ما أفضل مطعم في الرياض؟
[38/45] qwen_7b | ما حكم الطلاق في الشريعة الإسلامية؟
[39/45] qwen_7b | ما شروط القبول في الجامعة؟
[40/45] qwen_7b | كم سعر الذهب اليوم؟
[41/45] qwen_7b | ما شروط القرض العقاري؟
[42/45] qwen_7b | ما أفضل سيارة عائلية؟
[43/45] qwen_7b | كم سعر العقار في الرياض؟
[44/45] qwen_7b | من هو أفضل لاعب كرة قدم؟
[45/45] qwen_7b | ما علاج الصداع؟


LLM memory cleared.



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

✅ <b>تم الانتهاء من تقييم نماذج اللغة.</b><br>

تم إنشاء جدول <code>df_generation_eval</code> وحفظ النتائج في مجلد
<code>stage04_llm_generation_outputs</code>.

</div>


In [23]:
# =========================================================
# Stage 04.19 - Summary Metrics
# Professional Tables Version
# جداول احترافية لملخص مقارنة نماذج اللغة
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 0) Safety check
# ---------------------------------------------------------

if "df_generation_eval" not in globals():
    raise NameError("df_generation_eval غير موجود. شغّل خلية التقييم السابقة أولًا.")

# ---------------------------------------------------------
# 1) Helper functions
# ---------------------------------------------------------

def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def style_professional_table(
    df,
    caption="",
    number_formats=None,
    width="100%",
    font_size="12px",
    table_layout="fixed"
):
    if number_formats is None:
        number_formats = {}

    styler = (
        df.style
        .format(number_formats, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "18px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", table_layout),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_model_key(val):
    if str(val) == "allam_7b":
        return "background-color:#EAF3F8; color:#1F4E78; font-weight:bold;"
    if str(val) == "qwen_7b":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


def color_eval_type(val):
    val = str(val)
    if val == "faq_retrieval":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"
    if val == "legal_article_retrieval":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if val == "out_of_scope":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    if val == "small_talk":
        return "background-color:#EAF3F8; color:#1F4E78; font-weight:bold;"
    return ""


def highlight_best_max(val, series):
    try:
        if pd.isna(val):
            return ""
        max_val = series.max(skipna=True)
        if pd.notna(max_val) and np.isclose(val, max_val):
            return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    except Exception:
        pass
    return ""


def highlight_best_min(val, series):
    try:
        if pd.isna(val):
            return ""
        min_val = series.min(skipna=True)
        if pd.notna(min_val) and np.isclose(val, min_val):
            return "background-color:#EAF3F8; color:#1F4E78; font-weight:bold;"
    except Exception:
        pass
    return ""


# ---------------------------------------------------------
# 2) Build summary tables
# ---------------------------------------------------------

summary_overall = (
    df_generation_eval
    .groupby(["model_key", "model_name"], dropna=False)
    .agg(
        questions=("question", "count"),
        answer_accuracy=("answer_correct", "mean"),
        answered_rate=("answered", "mean"),
        refusal_rate=("refusal", "mean"),
        citation_hit_rate=("citation_hit", "mean"),
        avg_groundedness=("lexical_groundedness_proxy", "mean"),
        hallucination_proxy_rate=("hallucination_proxy", "mean"),
        avg_voice_suitability=("voice_suitability_score", "mean"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
        avg_generation_latency_sec=("generation_latency_sec", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
    )
    .reset_index()
)

summary_by_type = (
    df_generation_eval
    .groupby(["model_key", "model_name", "eval_type"], dropna=False)
    .agg(
        questions=("question", "count"),
        answer_accuracy=("answer_correct", "mean"),
        answered_rate=("answered", "mean"),
        refusal_rate=("refusal", "mean"),
        citation_hit_rate=("citation_hit", "mean"),
        avg_groundedness=("lexical_groundedness_proxy", "mean"),
        hallucination_proxy_rate=("hallucination_proxy", "mean"),
        avg_voice_suitability=("voice_suitability_score", "mean"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
    )
    .reset_index()
)

for df_ in [summary_overall, summary_by_type]:
    numeric_cols = df_.select_dtypes(include=[np.number]).columns
    df_[numeric_cols] = df_[numeric_cols].round(4)

# ---------------------------------------------------------
# 3) Optional sort for cleaner presentation
# ---------------------------------------------------------

summary_overall = summary_overall.sort_values(
    by=["answer_accuracy", "avg_groundedness", "avg_voice_suitability"],
    ascending=[False, False, False]
).reset_index(drop=True)

summary_by_type = summary_by_type.sort_values(
    by=["eval_type", "answer_accuracy"],
    ascending=[True, False]
).reset_index(drop=True)

# ---------------------------------------------------------
# 4) Rename columns for professional display
# ---------------------------------------------------------

summary_overall_display = summary_overall.rename(columns={
    "model_key": "Model Key",
    "model_name": "Model Name",
    "questions": "Questions",
    "answer_accuracy": "Answer Accuracy",
    "answered_rate": "Answered Rate",
    "refusal_rate": "Refusal Rate",
    "citation_hit_rate": "Citation Hit Rate",
    "avg_groundedness": "Avg. Groundedness",
    "hallucination_proxy_rate": "Hallucination Proxy",
    "avg_voice_suitability": "Voice Suitability",
    "avg_total_latency_sec": "Total Latency (sec)",
    "avg_generation_latency_sec": "Generation Latency (sec)",
    "avg_output_tokens": "Avg. Output Tokens",
})

summary_by_type_display = summary_by_type.rename(columns={
    "model_key": "Model Key",
    "model_name": "Model Name",
    "eval_type": "Evaluation Type",
    "questions": "Questions",
    "answer_accuracy": "Answer Accuracy",
    "answered_rate": "Answered Rate",
    "refusal_rate": "Refusal Rate",
    "citation_hit_rate": "Citation Hit Rate",
    "avg_groundedness": "Avg. Groundedness",
    "hallucination_proxy_rate": "Hallucination Proxy",
    "avg_voice_suitability": "Voice Suitability",
    "avg_total_latency_sec": "Total Latency (sec)",
})

# ---------------------------------------------------------
# 5) Markdown explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
ملخص نتائج تقييم نماذج اللغة
</h3>

<p>
يعرض الجدول الأول مقارنة شاملة بين النماذج على كامل مجموعة التقييم، بينما يعرض الجدول الثاني الأداء مفصلًا حسب نوع السؤال.
تمثل المقاييس الرئيسية دقة الإجابة، ومعدل الإجابة، ومعدل الرفض، والاستناد إلى السياق، وملاءمة الإجابة للصوت، وزمن الاستجابة.
</p>

</div>
"""))

# ---------------------------------------------------------
# 6) Format dictionaries
# ---------------------------------------------------------

overall_formats = {
    "Questions": "{:.0f}",
    "Answer Accuracy": "{:.2%}",
    "Answered Rate": "{:.2%}",
    "Refusal Rate": "{:.2%}",
    "Citation Hit Rate": "{:.2%}",
    "Avg. Groundedness": "{:.4f}",
    "Hallucination Proxy": "{:.2%}",
    "Voice Suitability": "{:.4f}",
    "Total Latency (sec)": "{:.3f}",
    "Generation Latency (sec)": "{:.3f}",
    "Avg. Output Tokens": "{:.2f}",
}

by_type_formats = {
    "Questions": "{:.0f}",
    "Answer Accuracy": "{:.2%}",
    "Answered Rate": "{:.2%}",
    "Refusal Rate": "{:.2%}",
    "Citation Hit Rate": "{:.2%}",
    "Avg. Groundedness": "{:.4f}",
    "Hallucination Proxy": "{:.2%}",
    "Voice Suitability": "{:.4f}",
    "Total Latency (sec)": "{:.3f}",
}

# ---------------------------------------------------------
# 7) Style overall table
# ---------------------------------------------------------

overall_style = style_professional_table(
    summary_overall_display,
    caption="Table 4.22 — Overall LLM Comparison",
    number_formats=overall_formats,
    width="100%",
    font_size="12px"
)

overall_style = apply_style_compatible(overall_style, color_model_key, subset=["Model Key"])

# Highlight best values
for col in ["Answer Accuracy", "Answered Rate", "Citation Hit Rate", "Avg. Groundedness", "Voice Suitability"]:
    if col in summary_overall_display.columns:
        series = summary_overall_display[col]
        overall_style = apply_style_compatible(
            overall_style,
            lambda v, s=series: highlight_best_max(v, s),
            subset=[col]
        )

for col in ["Refusal Rate", "Hallucination Proxy", "Total Latency (sec)", "Generation Latency (sec)"]:
    if col in summary_overall_display.columns:
        series = summary_overall_display[col]
        overall_style = apply_style_compatible(
            overall_style,
            lambda v, s=series: highlight_best_min(v, s),
            subset=[col]
        )

overall_style = overall_style.set_properties(
    subset=["Model Name"],
    **{"white-space": "normal", "line-height": "1.6"}
)

display(overall_style)

# ---------------------------------------------------------
# 8) Style by-type table
# ---------------------------------------------------------

by_type_style = style_professional_table(
    summary_by_type_display,
    caption="Table 4.23 — LLM Comparison by Evaluation Type",
    number_formats=by_type_formats,
    width="100%",
    font_size="12px"
)

by_type_style = apply_style_compatible(by_type_style, color_model_key, subset=["Model Key"])
by_type_style = apply_style_compatible(by_type_style, color_eval_type, subset=["Evaluation Type"])

by_type_style = by_type_style.set_properties(
    subset=["Model Name", "Evaluation Type"],
    **{"white-space": "normal", "line-height": "1.6"}
)

display(by_type_style)

# ---------------------------------------------------------
# 9) Final text summary
# ---------------------------------------------------------

best_model_row = summary_overall_display.iloc[0]

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
    margin-top:12px;
">

✅ <b>تفسير مختصر للنتائج:</b><br>

وفقًا للترتيب الحالي، النموذج الأعلى أداءً في هذا الملخص هو
<b>{best_model_row['Model Name']}</b>
(<code>{best_model_row['Model Key']}</code>)
بناءً على ترتيب الجداول حسب <b>Answer Accuracy</b> ثم <b>Groundedness</b> ثم <b>Voice Suitability</b>.

يعرض الجدول الأول الأداء العام لكل نموذج على كامل مجموعة التقييم، بينما يوضح الجدول الثاني كيف يختلف الأداء حسب نوع الأسئلة مثل
<code>faq_retrieval</code> و <code>legal_article_retrieval</code> و <code>out_of_scope</code> و <code>small_talk</code>.
وهذا مهم أكاديميًا لأنه يبيّن نقاط القوة والضعف لكل نموذج بدل الاكتفاء بدرجة واحدة فقط.

</div>
"""))

print("Professional summary tables generated successfully.")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
ملخص نتائج تقييم نماذج اللغة
</h3>

<p>
يعرض الجدول الأول مقارنة شاملة بين النماذج على كامل مجموعة التقييم، بينما يعرض الجدول الثاني الأداء مفصلًا حسب نوع السؤال.
تمثل المقاييس الرئيسية دقة الإجابة، ومعدل الإجابة، ومعدل الرفض، والاستناد إلى السياق، وملاءمة الإجابة للصوت، وزمن الاستجابة.
</p>

</div>


Model Key,Model Name,Questions,Answer Accuracy,Answered Rate,Refusal Rate,Citation Hit Rate,Avg. Groundedness,Hallucination Proxy,Voice Suitability,Total Latency (sec),Generation Latency (sec),Avg. Output Tokens
allam_7b,ALLaM-7B-Instruct-preview,45,73.33%,53.33%,46.67%,62.50%,0.4238,8.89%,1.0000,1.086,0.911,37.87
qwen_7b,Qwen2.5-7B-Instruct,45,68.89%,46.67%,53.33%,62.50%,0.3905,8.89%,0.9944,1.545,1.369,45.60


Model Key,Model Name,Evaluation Type,Questions,Answer Accuracy,Answered Rate,Refusal Rate,Citation Hit Rate,Avg. Groundedness,Hallucination Proxy,Voice Suitability,Total Latency (sec)
allam_7b,ALLaM-7B-Instruct-preview,faq_retrieval,14,42.86%,50.00%,50.00%,N/A,0.4500,7.14%,1.0000,1.046
qwen_7b,Qwen2.5-7B-Instruct,faq_retrieval,14,21.43%,28.57%,71.43%,N/A,0.3248,7.14%,0.9821,1.466
qwen_7b,Qwen2.5-7B-Instruct,legal_article_retrieval,17,82.35%,88.24%,11.76%,62.50%,0.7133,5.88%,1.0000,2.846
allam_7b,ALLaM-7B-Instruct-preview,legal_article_retrieval,17,76.47%,88.24%,11.76%,62.50%,0.6984,5.88%,1.0000,1.978
allam_7b,ALLaM-7B-Instruct-preview,out_of_scope,12,100.00%,0.00%,100.00%,N/A,0.0750,0.00%,1.0000,0.052
qwen_7b,Qwen2.5-7B-Instruct,out_of_scope,12,100.00%,0.00%,100.00%,N/A,0.0750,0.00%,1.0000,0.051
allam_7b,ALLaM-7B-Instruct-preview,small_talk,2,100.00%,100.00%,0.00%,N/A,0.0000,100.00%,1.0000,0.000
qwen_7b,Qwen2.5-7B-Instruct,small_talk,2,100.00%,100.00%,0.00%,N/A,0.0000,100.00%,1.0000,0.000



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#EAF3F8;
    padding:14px;
    border-right:5px solid #1F4E78;
    border-radius:6px;
    margin-top:12px;
">

✅ <b>تفسير مختصر للنتائج:</b><br>

وفقًا للترتيب الحالي، النموذج الأعلى أداءً في هذا الملخص هو
<b>ALLaM-7B-Instruct-preview</b>
(<code>allam_7b</code>)
بناءً على ترتيب الجداول حسب <b>Answer Accuracy</b> ثم <b>Groundedness</b> ثم <b>Voice Suitability</b>.

يعرض الجدول الأول الأداء العام لكل نموذج على كامل مجموعة التقييم، بينما يوضح الجدول الثاني كيف يختلف الأداء حسب نوع الأسئلة مثل
<code>faq_retrieval</code> و <code>legal_article_retrieval</code> و <code>out_of_scope</code> و <code>small_talk</code>.
وهذا مهم أكاديميًا لأنه يبيّن نقاط القوة والضعف لكل نموذج بدل الاكتفاء بدرجة واحدة فقط.

</div>


Professional summary tables generated successfully.


In [24]:
# =========================================================
# Stage 04.20D - Professional Readable Side-by-Side Comparison Cards
# مقارنة احترافية مقروءة بين إجابات النماذج
# =========================================================

from IPython.display import display, HTML, Markdown
import pandas as pd
import numpy as np
import re
import html

# ---------------------------------------------------------
# 0) Configuration
# ---------------------------------------------------------

# True = يعرض فقط الأسئلة التي أخطأ فيها نموذج واحد على الأقل
# False = يعرض كل الأسئلة
SHOW_ONLY_ERROR_QUESTIONS = True

# عدد الأسئلة المعروضة
# None = عرض الكل
MAX_QUESTIONS_TO_DISPLAY = 20

# ارتفاع صندوق الإجابة
# None = عرض الإجابة كاملة بدون scroll
ANSWER_BOX_HEIGHT_PX = 300

# ---------------------------------------------------------
# 1) Safety checks
# ---------------------------------------------------------

if "df_generation_eval" not in globals():
    raise NameError("df_generation_eval غير موجود. شغّل Stage 04.18 أولًا.")

required_cols = ["model_key", "question", "eval_type", "answer_correct", "answer"]
missing_cols = [c for c in required_cols if c not in df_generation_eval.columns]

if missing_cols:
    raise ValueError(f"الأعمدة التالية غير موجودة في df_generation_eval: {missing_cols}")

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_na(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>"]:
        return "N/A"

    return value


def to_int01(value):
    try:
        return int(float(value))
    except Exception:
        return 0


def to_bool(value, default=False):
    if isinstance(value, bool):
        return value

    value = clean_na(value).lower()

    if value in ["true", "1", "yes", "y"]:
        return True
    if value in ["false", "0", "no", "n", "n/a"]:
        return False

    return default


def fmt_float(value, digits=4):
    try:
        if pd.isna(value):
            return "N/A"
        return f"{float(value):.{digits}f}"
    except Exception:
        return "N/A"


def fmt_article(value):
    value = clean_na(value)
    if value == "N/A":
        return "N/A"

    # يحل مشكلة ظهور 113, 0
    value = value.replace(",", ".")
    try:
        f = float(value)
        if f.is_integer():
            return str(int(f))
        return str(value)
    except Exception:
        return str(value)


def esc(value):
    return html.escape(clean_na(value))


def get_first_available(row_or_df, cols, default="N/A"):
    if isinstance(row_or_df, pd.Series):
        for c in cols:
            if c in row_or_df.index:
                value = clean_na(row_or_df.get(c))
                if value != "N/A":
                    return value
        return default

    if isinstance(row_or_df, pd.DataFrame):
        for c in cols:
            if c in row_or_df.columns:
                values = row_or_df[c].dropna().astype(str).tolist()
                values = [
                    v for v in values
                    if v.strip().lower() not in ["", "nan", "none", "<na>"]
                ]
                if values:
                    return values[0]
        return default

    return default


def badge(text, bg, fg):
    return f"""
    <span style="
        display:inline-block;
        background-color:{bg};
        color:{fg};
        font-weight:bold;
        padding:5px 9px;
        border-radius:6px;
        font-size:12px;
        line-height:1.4;
    ">{esc(text)}</span>
    """


def result_badge(value):
    if value == "Correct":
        return badge("Correct", "#E2F0D9", "#375623")
    if value == "Wrong":
        return badge("Wrong", "#FCE4D6", "#9C0006")
    return badge(value, "#F2F2F2", "#333333")


def eval_type_badge(value):
    value = str(value)

    if value == "faq_retrieval":
        return badge(value, "#FFF2CC", "#7F6000")
    if value == "legal_article_retrieval":
        return badge(value, "#E2F0D9", "#375623")
    if value == "out_of_scope":
        return badge(value, "#FCE4D6", "#9C0006")
    if value == "small_talk":
        return badge(value, "#EAF3F8", "#1F4E78")

    return badge(value, "#F2F2F2", "#333333")


def error_badge(value):
    value = str(value)

    colors = {
        "Correct": ("#E2F0D9", "#375623"),
        "legal_citation_missing_or_wrong": ("#FCE4D6", "#9C0006"),
        "no_answer_or_refusal": ("#FFF2CC", "#7F6000"),
        "other_generation_error": ("#EAF3F8", "#1F4E78"),
        "failed_to_refuse_out_of_scope": ("#F4CCCC", "#990000"),
        "retrieval_unreliable_but_answered": ("#D9EAD3", "#274E13"),
        "low_context_overlap_hallucination_proxy": ("#EADCF8", "#5B2C6F"),
    }

    bg, fg = colors.get(value, ("#F2F2F2", "#333333"))
    return badge(value, bg, fg)


def answer_box(text):
    text = esc(text)

    if ANSWER_BOX_HEIGHT_PX is None:
        height_style = ""
    else:
        height_style = f"max-height:{ANSWER_BOX_HEIGHT_PX}px; overflow-y:auto;"

    return f"""
    <div style="
        direction:rtl;
        text-align:right;
        white-space:pre-wrap;
        line-height:2;
        font-size:13px;
        padding:12px;
        background-color:#FFFFFF;
        border:1px solid #D9E2F3;
        border-radius:8px;
        {height_style}
    ">{text}</div>
    """


def question_box(text):
    return f"""
    <div style="
        direction:rtl;
        text-align:right;
        white-space:pre-wrap;
        line-height:1.9;
        font-size:14px;
        font-weight:bold;
        color:#222222;
        padding:10px;
        background-color:#FFFFFF;
        border:1px solid #D9E2F3;
        border-radius:8px;
    ">{esc(text)}</div>
    """


# ---------------------------------------------------------
# 3) Error classification
# ---------------------------------------------------------

def classify_error(row):
    eval_type = str(row.get("eval_type", "")).strip()
    answer = clean_na(row.get("answer", ""))
    answered = to_bool(row.get("answered", False))
    refusal = to_bool(row.get("refusal", False))
    retrieval_reliable = to_bool(row.get("retrieval_reliable", True), default=True)
    hallucination_proxy = to_bool(row.get("hallucination_proxy", False))
    citation_hit = row.get("citation_hit", np.nan)

    gold_article = clean_na(
        row.get("gold_article_numbers_parsed", row.get("gold_article_numbers", ""))
    )

    answer_correct = to_int01(row.get("answer_correct", 0))

    if answer_correct == 1:
        return "Correct"

    if answer == "N/A":
        return "no_answer_or_refusal"

    if eval_type == "out_of_scope" and answered and not refusal:
        return "failed_to_refuse_out_of_scope"

    if not retrieval_reliable and answered:
        return "retrieval_unreliable_but_answered"

    if eval_type == "legal_article_retrieval" and gold_article != "N/A":
        try:
            if pd.notna(citation_hit) and float(citation_hit) == 0:
                return "legal_citation_missing_or_wrong"
        except Exception:
            pass

    if hallucination_proxy:
        return "low_context_overlap_hallucination_proxy"

    if refusal and eval_type not in ["out_of_scope", "small_talk"]:
        return "no_answer_or_refusal"

    return "other_generation_error"


# ---------------------------------------------------------
# 4) Prepare comparison dataframe
# ---------------------------------------------------------

df_compare = df_generation_eval.copy()

df_compare["answer_correct_int"] = df_compare["answer_correct"].apply(to_int01)
df_compare["result_label"] = df_compare["answer_correct_int"].apply(
    lambda x: "Correct" if x == 1 else "Wrong"
)
df_compare["error_category_for_compare"] = df_compare.apply(classify_error, axis=1)

# ترتيب النماذج
available_models = list(df_compare["model_key"].dropna().unique())
preferred_order = ["allam_7b", "qwen_7b"]
models = [m for m in preferred_order if m in available_models] + [
    m for m in available_models if m not in preferred_order
]

# اختيار الأسئلة
if SHOW_ONLY_ERROR_QUESTIONS:
    selected_questions = (
        df_compare[df_compare["answer_correct_int"] == 0]
        [["question", "eval_type"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
else:
    selected_questions = (
        df_compare[["question", "eval_type"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ترتيب: الأسئلة التي يوجد فيها اختلاف بين النماذج أولًا
priority_rows = []

for _, qrow in selected_questions.iterrows():
    q = qrow["question"]
    etype = qrow["eval_type"]

    subset = df_compare[
        (df_compare["question"] == q) &
        (df_compare["eval_type"] == etype)
    ]

    results = subset["result_label"].dropna().tolist()
    unique_results = set(results)

    priority = 1 if ("Correct" in unique_results and "Wrong" in unique_results) else 0

    priority_rows.append({
        "question": q,
        "eval_type": etype,
        "priority": priority
    })

selected_questions = pd.DataFrame(priority_rows)

selected_questions = selected_questions.sort_values(
    ["priority", "eval_type"],
    ascending=[False, True]
).drop(columns=["priority"]).reset_index(drop=True)

if MAX_QUESTIONS_TO_DISPLAY is not None:
    selected_questions = selected_questions.head(MAX_QUESTIONS_TO_DISPLAY)

# ---------------------------------------------------------
# 5) Markdown explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Table 4.29 — Readable Side-by-Side Answer Comparison
</h3>

<p>
يعرض هذا الشكل كل سؤال في بطاقة مستقلة، وتظهر إجابة كل نموذج كاملة في بطاقة فرعية.
هذا التصميم أفضل من الجدول العريض لأنه يسمح بقراءة الإجابات الطويلة ومقارنة النموذجين بشكل مباشر.
</p>

</div>
"""))

# ---------------------------------------------------------
# 6) Build HTML cards
# ---------------------------------------------------------

cards_html = ""

for idx, qrow in selected_questions.iterrows():
    question = qrow["question"]
    eval_type = qrow["eval_type"]

    subset = df_compare[
        (df_compare["question"] == question) &
        (df_compare["eval_type"] == eval_type)
    ].copy()

    gold_article = get_first_available(
        subset,
        ["gold_article_numbers_parsed", "gold_article_numbers"],
        default="N/A"
    )

    gold_article = fmt_article(gold_article)

    model_cards = ""

    for model in models:
        mrow_df = subset[subset["model_key"] == model]

        model_display = {
            "allam_7b": "ALLaM-7B",
            "qwen_7b": "Qwen2.5-7B"
        }.get(model, model)

        if len(mrow_df) == 0:
            model_cards += f"""
            <div style="
                border:1px solid #D9E2F3;
                border-radius:10px;
                padding:12px;
                background-color:#FFFFFF;
            ">
                <h4 style="margin:0 0 10px 0; color:#1F4E78; text-align:center;">
                    {esc(model_display)}
                </h4>
                <p style="text-align:center;">N/A</p>
            </div>
            """
            continue

        mrow = mrow_df.iloc[0]

        result = clean_na(mrow.get("result_label", "N/A"))
        error_category = clean_na(mrow.get("error_category_for_compare", "N/A"))
        answer = clean_na(mrow.get("answer", "N/A"))

        retrieved_article = fmt_article(
            get_first_available(
                mrow,
                ["top_article_number", "retrieved_article", "top_article"],
                default="N/A"
            )
        )

        top_score = fmt_float(mrow.get("top_score", np.nan), 4)
        groundedness = fmt_float(
            mrow.get("lexical_groundedness_proxy", mrow.get("groundedness", np.nan)),
            4
        )

        total_latency = fmt_float(mrow.get("total_latency_sec", np.nan), 2)

        model_border = "#70AD47" if result == "Correct" else "#C00000"
        model_bg = "#F8FBFD" if result == "Correct" else "#FFF7F4"

        model_cards += f"""
        <div style="
            border:2px solid {model_border};
            border-radius:10px;
            padding:12px;
            background-color:{model_bg};
        ">

            <h4 style="
                margin:0 0 10px 0;
                color:#1F4E78;
                text-align:center;
                font-size:16px;
            ">
                {esc(model_display)}
            </h4>

            <div style="
                display:grid;
                grid-template-columns:repeat(2, minmax(0, 1fr));
                gap:8px;
                margin-bottom:10px;
                direction:ltr;
            ">
                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Result</div>
                    {result_badge(result)}
                </div>

                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Error Category</div>
                    {error_badge(error_category)}
                </div>

                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Retrieved Article</div>
                    <b>{esc(retrieved_article)}</b>
                </div>

                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Top Score</div>
                    <b>{esc(top_score)}</b>
                </div>

                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Groundedness</div>
                    <b>{esc(groundedness)}</b>
                </div>

                <div style="text-align:center;">
                    <div style="font-size:11px; color:#666;">Latency Sec</div>
                    <b>{esc(total_latency)}</b>
                </div>
            </div>

            <div style="
                font-weight:bold;
                color:#1F4E78;
                text-align:right;
                direction:rtl;
                margin-bottom:6px;
            ">
                إجابة النموذج:
            </div>

            {answer_box(answer)}

        </div>
        """

    cards_html += f"""
    <div style="
        border:1px solid #B4C7E7;
        border-radius:12px;
        margin:18px 0;
        overflow:hidden;
        background-color:#FFFFFF;
        box-shadow:0 1px 4px rgba(0,0,0,0.06);
    ">

        <div style="
            background-color:#1F4E78;
            color:white;
            padding:12px 14px;
            display:flex;
            justify-content:space-between;
            gap:10px;
            align-items:center;
            direction:rtl;
            flex-wrap:wrap;
        ">
            <div style="font-weight:bold;">
                Question #{idx + 1}
            </div>
            <div>
                {eval_type_badge(eval_type)}
            </div>
            <div style="font-weight:bold;">
                Gold Article: {esc(gold_article)}
            </div>
        </div>

        <div style="padding:14px; background-color:#F8FBFD;">
            <div style="
                font-weight:bold;
                color:#1F4E78;
                margin-bottom:8px;
                direction:rtl;
                text-align:right;
            ">
                السؤال:
            </div>

            {question_box(question)}

            <div style="
                display:grid;
                grid-template-columns:repeat(auto-fit, minmax(420px, 1fr));
                gap:16px;
                margin-top:14px;
                direction:ltr;
            ">
                {model_cards}
            </div>
        </div>

    </div>
    """

full_html = f"""
<div style="
    font-family:Arial, Tahoma, sans-serif;
    width:100%;
">
    {cards_html}
</div>
"""

display(HTML(full_html))

print("Readable side-by-side comparison cards generated successfully.")
print("Displayed questions:", len(selected_questions))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Table 4.29 — Readable Side-by-Side Answer Comparison
</h3>

<p>
يعرض هذا الشكل كل سؤال في بطاقة مستقلة، وتظهر إجابة كل نموذج كاملة في بطاقة فرعية.
هذا التصميم أفضل من الجدول العريض لأنه يسمح بقراءة الإجابات الطويلة ومقارنة النموذجين بشكل مباشر.
</p>

</div>


Readable side-by-side comparison cards generated successfully.
Displayed questions: 15


In [25]:
# =========================================================
# Stage 04.21 - Select Best LLM Configuration
# اختيار أفضل نموذج بطريقة أكاديمية بسيطة
# =========================================================

best_llm_table = summary_overall.copy()

best_llm_table["selection_score"] = (
    0.45 * best_llm_table["answer_accuracy"].fillna(0)
    + 0.25 * best_llm_table["avg_groundedness"].fillna(0)
    + 0.20 * best_llm_table["avg_voice_suitability"].fillna(0)
    - 0.10 * best_llm_table["hallucination_proxy_rate"].fillna(0)
)

best_llm_table = best_llm_table.sort_values("selection_score", ascending=False).reset_index(drop=True)

best_llm_config = best_llm_table.iloc[0].to_dict() if len(best_llm_table) else {}

print("Best LLM according to Stage 04 selection score:")
display(best_llm_table)
print(json.dumps(best_llm_config, ensure_ascii=False, indent=2))

Best LLM according to Stage 04 selection score:


,model_key,model_name,questions,answer_accuracy,answered_rate,refusal_rate,citation_hit_rate,avg_groundedness,hallucination_proxy_rate,avg_voice_suitability,avg_total_latency_sec,avg_generation_latency_sec,avg_output_tokens,selection_score
0,allam_7b,ALLaM-7B-Instruct-preview,45,0.7333,0.5333,0.4667,0.625,0.4238,0.0889,1.0000,1.0864,0.9115,37.8667,0.627045
1,qwen_7b,Qwen2.5-7B-Instruct,45,0.6889,0.4667,0.5333,0.625,0.3905,0.0889,0.9944,1.5452,1.3693,45.6000,0.597620


{
  "model_key": "allam_7b",
  "model_name": "ALLaM-7B-Instruct-preview",
  "questions": 45,
  "answer_accuracy": 0.7333,
  "answered_rate": 0.5333,
  "refusal_rate": 0.4667,
  "citation_hit_rate": 0.625,
  "avg_groundedness": 0.4238,
  "hallucination_proxy_rate": 0.0889,
  "avg_voice_suitability": 1.0,
  "avg_total_latency_sec": 1.0864,
  "avg_generation_latency_sec": 0.9115,
  "avg_output_tokens": 37.8667,
  "selection_score": 0.627045
}


In [26]:
# =========================================================
# Stage 04.22 - Voice Layer Demo Table
# جدول مختصر جاهز لمرحلة TTS
# =========================================================

voice_layer_demo = df_generation_eval[
    (df_generation_eval["answer_correct"].fillna(0).astype(int) == 1)
].copy()

if len(best_llm_config) and "model_key" in best_llm_config:
    voice_layer_demo = voice_layer_demo[voice_layer_demo["model_key"] == best_llm_config["model_key"]]

voice_layer_demo = voice_layer_demo[[
    "question", "answer", "model_name", "eval_type",
    "voice_suitability_score", "top_article_number", "top_source_type"
]].head(20)

print("Voice layer demo examples:")
display(voice_layer_demo)

Voice layer demo examples:


,question,answer,model_name,eval_type,voice_suitability_score,top_article_number,top_source_type
0,هلو,أهلًا بك. أنا مساعد ذكي تجريبي مخصص للإجابة عن...,ALLaM-7B-Instruct-preview,small_talk,1.0,N/A,N/A
1,وش تقدر تساعدني؟,أهلًا بك. أنا مساعد ذكي تجريبي مخصص للإجابة عن...,ALLaM-7B-Instruct-preview,small_talk,1.0,N/A,N/A
2,كم سعر البيت في الرياض؟,عذرًا، لا أستطيع تقديم إجابة موثوقة لهذا السؤا...,ALLaM-7B-Instruct-preview,out_of_scope,1.0,N/A,N/A
3,ما حكم الطلاق؟,عذرًا، لا أستطيع تقديم إجابة موثوقة لهذا السؤا...,ALLaM-7B-Instruct-preview,out_of_scope,1.0,N/A,N/A
5,ماهي حقوق العامل في الإجازة المرضیة؟,وفقًا للمادة رقم 117 من نظام العمل السعودي، يح...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,N/A,faq
7,ماهي متطلبات الاستفادة من مبادرة عقد العمل الم...,يتم توثيق عقد العمل الموثق سندًا تنفيذيًا عبر ...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,N/A,faq
9,هل يتم احتساب الفرد الذي يعمل لدى أكثر من منشأ...,يتم الاحتساب على كيان المنشأة، حيث يعرف الكيان...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,N/A,faq
10,من يدفع رسوم الإقامة ورخص العمل في النظام الجد...,وفقًا للمادة رقم 40 من نظام العمل السعودي، يتح...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,"1, 2, 3, 4, 40",faq
12,هل تمكن البوابة والانظمة التابعة لها المستخدم ...,نعم، تمكن البوابة والأنظمة التابعة لها المستخد...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,N/A,faq
13,كيف اتابع الشكوى وما هي الوثائق المطلوبة؟,وفقًا للمادة رقم 122 من نظام العمل السعودي، يت...,ALLaM-7B-Instruct-preview,faq_retrieval,1.0,N/A,faq


In [27]:
# =========================================================
# Stage 04.23 - Save Stage 04 Outputs
# حفظ مخرجات المرحلة الرابعة
# =========================================================

STAGE04_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Error-analysis subset (df_errors): cases worth manual review
# (incorrect answer, hallucination proxy, or runtime error).
# Defined here defensively because no earlier cell builds it.
# ---------------------------------------------------------
if "df_errors" not in globals():
    _err_mask = pd.Series(False, index=df_generation_eval.index)
    if "answer_correct" in df_generation_eval.columns:
        _err_mask = _err_mask | df_generation_eval["answer_correct"].fillna(0).astype(int).eq(0)
    if "hallucination_proxy" in df_generation_eval.columns:
        _err_mask = _err_mask | df_generation_eval["hallucination_proxy"].fillna(0).astype(int).eq(1)
    if "runtime_error" in df_generation_eval.columns:
        _rt = df_generation_eval["runtime_error"]
        _err_mask = _err_mask | (_rt.notna() & _rt.astype("string").str.strip().fillna("").ne(""))
    df_errors = df_generation_eval[_err_mask].copy()
    print(f"Error-analysis rows (df_errors): {len(df_errors)} of {len(df_generation_eval)}")

# Models actually evaluated (used in the Stage 04 config manifest below).
if "model_keys_to_run" not in globals():
    if "model_key" in df_generation_eval.columns:
        model_keys_to_run = list(pd.Series(df_generation_eval["model_key"]).dropna().unique())
    else:
        model_keys_to_run = [k for k, v in LOCAL_MODELS.items() if v.get("enabled", True)]

paths = {
    "generation_evaluation_detailed_csv": STAGE04_DIR / "generation_evaluation_detailed.csv",
    "generation_evaluation_detailed_xlsx": STAGE04_DIR / "generation_evaluation_detailed.xlsx",
    "llm_comparison_overall_csv": STAGE04_DIR / "llm_comparison_overall.csv",
    "llm_comparison_overall_xlsx": STAGE04_DIR / "llm_comparison_overall.xlsx",
    "llm_comparison_by_type_csv": STAGE04_DIR / "llm_comparison_by_type.csv",
    "llm_comparison_by_type_xlsx": STAGE04_DIR / "llm_comparison_by_type.xlsx",
    "generation_error_analysis_csv": STAGE04_DIR / "generation_error_analysis.csv",
    "generation_error_analysis_xlsx": STAGE04_DIR / "generation_error_analysis.xlsx",
    "voice_layer_demo_csv": STAGE04_DIR / "voice_layer_demo.csv",
    "voice_layer_demo_xlsx": STAGE04_DIR / "voice_layer_demo.xlsx",
    "best_llm_config_json": STAGE04_DIR / "best_llm_config.json",
    "stage04_config_json": STAGE04_DIR / "stage04_config.json",
    "stage04_output_manifest_csv": STAGE04_DIR / "stage04_output_manifest.csv",
    "stage04_output_manifest_xlsx": STAGE04_DIR / "stage04_output_manifest.xlsx",
}

df_generation_eval.to_csv(paths["generation_evaluation_detailed_csv"], index=False, encoding="utf-8-sig")
df_generation_eval.to_excel(paths["generation_evaluation_detailed_xlsx"], index=False)

summary_overall.to_csv(paths["llm_comparison_overall_csv"], index=False, encoding="utf-8-sig")
summary_overall.to_excel(paths["llm_comparison_overall_xlsx"], index=False)

summary_by_type.to_csv(paths["llm_comparison_by_type_csv"], index=False, encoding="utf-8-sig")
summary_by_type.to_excel(paths["llm_comparison_by_type_xlsx"], index=False)

df_errors.to_csv(paths["generation_error_analysis_csv"], index=False, encoding="utf-8-sig")
df_errors.to_excel(paths["generation_error_analysis_xlsx"], index=False)

voice_layer_demo.to_csv(paths["voice_layer_demo_csv"], index=False, encoding="utf-8-sig")
voice_layer_demo.to_excel(paths["voice_layer_demo_xlsx"], index=False)

with open(paths["best_llm_config_json"], "w", encoding="utf-8") as f:
    json.dump(best_llm_config, f, ensure_ascii=False, indent=2)

stage04_config = {
    "stage": "Stage 04 - Grounded LLM Answer Generation",
    "project_dir": str(PROJECT_DIR),
    "stage03_dir": str(STAGE03_DIR),
    "stage04_dir": str(STAGE04_DIR),
    "best_retrieval_config": {
        "chunking_strategy": BEST_CHUNKING_STRATEGY,
        "embedding_model": BEST_EMBEDDING_KEY,
        "retriever": BEST_RETRIEVER,
        "alpha": BEST_ALPHA,
        "top_k": TOP_K,
        "candidate_k": CANDIDATE_K,
        "rerank_candidate_k": RERANK_CANDIDATE_K,
        "reliability_threshold": RELIABILITY_THRESHOLD,
        "chroma_collection": collection_name,
    },
    "models_evaluated": model_keys_to_run,
    "max_eval_rows": MAX_EVAL_ROWS,
    "evaluation_note": (
        "Legal questions are evaluated using article citation correctness when gold article numbers exist. "
        "FAQ questions are evaluated using answer presence and lexical groundedness proxy, not article citation."
    )
}

with open(paths["stage04_config_json"], "w", encoding="utf-8") as f:
    json.dump(stage04_config, f, ensure_ascii=False, indent=2)

manifest = pd.DataFrame([
    {"output_name": k, "path": str(v), "exists": Path(v).exists()}
    for k, v in paths.items()
])
manifest.to_csv(paths["stage04_output_manifest_csv"], index=False, encoding="utf-8-sig")
manifest.to_excel(paths["stage04_output_manifest_xlsx"], index=False)

print("Saved Stage 04 outputs:")
display(manifest)

Error-analysis rows (df_errors): 30 of 90


Saved Stage 04 outputs:


,output_name,path,exists
0,generation_evaluation_detailed_csv,C:\Users\PC\Desktop\data collection code\saudi...,True
1,generation_evaluation_detailed_xlsx,C:\Users\PC\Desktop\data collection code\saudi...,True
2,llm_comparison_overall_csv,C:\Users\PC\Desktop\data collection code\saudi...,True
3,llm_comparison_overall_xlsx,C:\Users\PC\Desktop\data collection code\saudi...,True
4,llm_comparison_by_type_csv,C:\Users\PC\Desktop\data collection code\saudi...,True
5,llm_comparison_by_type_xlsx,C:\Users\PC\Desktop\data collection code\saudi...,True
6,generation_error_analysis_csv,C:\Users\PC\Desktop\data collection code\saudi...,True
7,generation_error_analysis_xlsx,C:\Users\PC\Desktop\data collection code\saudi...,True
8,voice_layer_demo_csv,C:\Users\PC\Desktop\data collection code\saudi...,True
9,voice_layer_demo_xlsx,C:\Users\PC\Desktop\data collection code\saudi...,True


<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">

# نهاية المرحلة الرابعة

بعد نجاح هذه المرحلة، مخرجاتك الأساسية هي:

- `generation_evaluation_detailed.xlsx`
- `llm_comparison_overall.xlsx`
- `llm_comparison_by_type.xlsx`
- `generation_error_analysis.xlsx`
- `best_llm_config.json`
- `voice_layer_demo.xlsx`

الخطوة التالية هي **Stage 05 — Text-to-Speech (TTS)**، لكن لا تبدأ بها إلا بعد مراجعة جدول الأخطاء والتأكد أن إجابات النموذج المختار مقبولة.

</div>

<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:18px;
    margin:16px 0;
    border-radius:6px;
">

<h2 style="color:#1F4E78; margin-top:0;">
المنهجية المستخدمة في تقييم نماذج اللغة في المرحلة الرابعة
</h2>

<p>
تم في هذه المرحلة تقييم نماذج اللغة الكبيرة من خلال اختبارها على مجموعة أسئلة موحدة تم إعدادها من مصادر رسمية مرتبطة بنظام العمل السعودي وخدمات وزارة الموارد البشرية.
تم تمرير نفس الأسئلة إلى كل نموذج، ثم تمت مقارنة النتائج باستخدام مجموعة من المقاييس التي تعكس دقة الإجابة، وارتباطها بالسياق المسترجع، ومدى مناسبتها للاستخدام في مساعد صوتي قانوني.
</p>

<p>
لا يعتمد التقييم في هذه المرحلة على صحة الإجابة فقط، لأن طبيعة المشروع تتطلب نموذجًا قادرًا على توليد إجابات قانونية موثوقة، ومدعومة بالسياق، وقابلة للنطق صوتيًا بشكل واضح.
لذلك تم استخدام عدة مقاييس تقييم بدل الاعتماد على مقياس واحد.
</p>

<h3 style="color:#1F4E78;">الربط الأكاديمي بالمفاهيم المستخدمة في المحاضرات</h3>

<p>
تعتمد منهجية التقييم في هذه المرحلة على مفاهيم من عدة مجالات تمت دراستها ضمن مساقات الذكاء الاصطناعي، خصوصًا
تعلم الآلة، ومعالجة اللغة الطبيعية، واسترجاع المعلومات، وتقييم أداء الأنظمة الذكية.
المقاييس المستخدمة في هذه المرحلة لا تمثل مكتبات برمجية بحد ذاتها، بل تمثل مؤشرات تقييم تم حسابها باستخدام أدوات برمجية مناسبة.
</p>

<table style="
    width:100%;
    border-collapse:collapse;
    font-family:Arial, Tahoma, sans-serif;
    font-size:14px;
    margin:14px 0;
">
    <thead>
        <tr style="background-color:#1F4E78; color:white;">
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">Metric</th>
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">المجال الأكاديمي</th>
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">الطريقة المستخدمة</th>
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">علاقتها بالمشروع</th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>Answer Accuracy</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Machine Learning Evaluation</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Reference-based / Binary Scoring</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">قياس ما إذا كانت إجابة النموذج صحيحة أو غير صحيحة حسب نوع السؤال.</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>Groundedness</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Natural Language Processing + Information Retrieval</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Context Matching / Lexical Overlap</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">قياس مدى اعتماد الإجابة على السياق المسترجع من قاعدة المعرفة.</td>
        </tr>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>Voice Suitability</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">NLP for Speech Interfaces</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Rule-based Text Quality Heuristics</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تقييم مدى مناسبة الإجابة للتحويل إلى صوت من حيث الطول والوضوح والبساطة.</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>Hallucination Proxy</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">NLP Reliability Evaluation</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Low Context Overlap Detection</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">مؤشر تقريبي لاكتشاف الإجابات التي قد لا تكون مدعومة بالسياق.</td>
        </tr>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>Latency</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">System Performance Evaluation</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Runtime Measurement</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">قياس سرعة النظام، وهي مهمة في تطبيقات المساعد الصوتي التفاعلي.</td>
        </tr>
    </tbody>
</table>

<h3 style="color:#1F4E78;">المكتبات البرمجية المستخدمة وعلاقتها بالتقييم</h3>

<p>
تم استخدام مجموعة من مكتبات بايثون لتنفيذ خط التقييم وتحليل النتائج. بعض هذه المكتبات استخدم لتنظيم البيانات، وبعضها لمعالجة النصوص، وبعضها لتشغيل نماذج اللغة والاسترجاع.
</p>

<table style="
    width:100%;
    border-collapse:collapse;
    font-family:Arial, Tahoma, sans-serif;
    font-size:14px;
    margin:14px 0;
">
    <thead>
        <tr style="background-color:#1F4E78; color:white;">
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">Library</th>
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">الاستخدام في المشروع</th>
            <th style="border:1px solid #D9E2F3; padding:9px; text-align:center;">المجال المرتبط</th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>pandas</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تنظيم بيانات التقييم، بناء الجداول، وحساب المتوسطات والنسب.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Data Analysis</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>numpy</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">العمليات الرقمية وحساب المؤشرات الكمية.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Machine Learning Utilities</td>
        </tr>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>re</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تنظيف النصوص، واستخراج أرقام المواد، ومعالجة الأنماط النصية.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">NLP Preprocessing</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>rank_bm25</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تنفيذ الاسترجاع اللفظي باستخدام BM25.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Information Retrieval</td>
        </tr>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>ChromaDB</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تخزين واسترجاع المتجهات النصية ضمن قاعدة معرفة RAG.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Vector Database / RAG</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>sentence-transformers</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تحويل الأسئلة والنصوص إلى تمثيلات رقمية دلالية Embeddings.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Semantic Search / NLP</td>
        </tr>
        <tr style="background-color:#FFFFFF;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>transformers</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تحميل وتشغيل نماذج اللغة الكبيرة المستخدمة في توليد الإجابات.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">LLM / NLP</td>
        </tr>
        <tr style="background-color:#F8FBFD;">
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;"><code>torch</code></td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:right;">تشغيل النماذج العميقة والاستفادة من المعالجة على GPU.</td>
            <td style="border:1px solid #D9E2F3; padding:8px; text-align:center;">Deep Learning</td>
        </tr>
    </tbody>
</table>

<h3 style="color:#1F4E78;">أولًا: تقييم طبقة الاسترجاع قبل توليد الإجابة</h3>

<p>
قبل تمرير السؤال إلى نموذج اللغة، يتم اختبار طبقة الاسترجاع للتأكد من أن النظام استرجع سياقًا مناسبًا من قاعدة المعرفة.
يتم ذلك من خلال فحص درجة الاسترجاع الأعلى، وتحديد ما إذا كان السياق المسترجع تجاوز عتبة الموثوقية المحددة مسبقًا.
</p>

<p>
في حال كان الاسترجاع موثوقًا، يتم تمرير السياق إلى نموذج اللغة. أما إذا كانت درجة الاسترجاع أقل من العتبة، فيتم منع النموذج من توليد إجابة غير مدعومة.
</p>

<h3 style="color:#1F4E78;">ثانيًا: تقييم صحة الإجابة</h3>

<p>
تم قياس صحة الإجابة من خلال مؤشر <code>answer_correct</code>، والذي يحدد ما إذا كانت إجابة النموذج مقبولة وفق نوع السؤال.
تختلف طريقة الحكم على صحة الإجابة حسب نوع التقييم:
</p>

<ul>
    <li>
        <b>الأسئلة القانونية:</b> يتم تقييمها بناءً على صحة الإجابة، وارتباطها بالسياق، وذكر رقم المادة الصحيح عند توفره.
    </li>
    <li>
        <b>أسئلة FAQ:</b> يتم تقييمها بناءً على مدى دعم الإجابة من السياق الرسمي المسترجع، ولا يشترط فيها وجود رقم مادة قانونية.
    </li>
    <li>
        <b>الأسئلة خارج النطاق:</b> تكون الإجابة الصحيحة هي الرفض الآمن بدل توليد إجابة غير مرتبطة بالمجال.
    </li>
    <li>
        <b>المحادثة العامة:</b> يتم تقييمها بناءً على قدرة النموذج على تقديم رد تعريفي مختصر ومناسب.
    </li>
</ul>

<h3 style="color:#1F4E78;">ثالثًا: تقييم الاستشهاد بالمادة القانونية</h3>

<p>
بالنسبة للأسئلة القانونية التي تحتوي على رقم مادة ذهبي، يتم فحص ما إذا كان النموذج ذكر رقم المادة الصحيح في الإجابة.
يساعد هذا المقياس على تقييم قدرة النموذج على إنتاج إجابات قانونية قابلة للتتبع والرجوع إلى المصدر.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:12px;
    margin:14px 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">ملاحظة منهجية:</b>
لا يتم تطبيق تقييم الاستشهاد بالمادة على جميع الأسئلة، لأن أسئلة FAQ والمحادثة العامة والأسئلة خارج النطاق لا ترتبط غالبًا برقم مادة قانونية محددة.
</div>

<h3 style="color:#1F4E78;">رابعًا: تقييم الارتباط بالسياق المسترجع</h3>

<p>
تم استخدام مؤشر تقريبي لقياس مدى ارتباط إجابة النموذج بالسياق المسترجع من قاعدة المعرفة.
يعتمد هذا المؤشر على مقارنة الكلمات والمفاهيم الموجودة في الإجابة مع النصوص المسترجعة.
</p>

<p>
كلما زاد ارتباط الإجابة بالسياق، زادت الثقة بأن النموذج اعتمد على المعرفة الرسمية المسترجعة بدل الاعتماد على معرفة عامة أو توليد غير مدعوم.
</p>

<h3 style="color:#1F4E78;">خامسًا: مؤشر الهلوسة التقريبي</h3>

<p>
تم استخدام مؤشر تقريبي لاكتشاف الحالات التي قد تكون فيها الإجابة غير مدعومة بالسياق.
لا يمثل هذا المؤشر حكمًا نهائيًا على وجود الهلوسة، لكنه يساعد في تحديد الإجابات التي تحتاج إلى مراجعة إضافية.
</p>

<p>
تظهر احتمالية الهلوسة عندما ينتج النموذج إجابة، لكن درجة ارتباطها بالسياق المسترجع تكون منخفضة.
</p>

<h3 style="color:#1F4E78;">سادسًا: تقييم الرفض الآمن</h3>

<p>
تم تقييم قدرة النموذج على رفض الأسئلة التي تقع خارج نطاق نظام العمل السعودي أو خارج نطاق خدمات وزارة الموارد البشرية.
هذا المقياس مهم لأن المساعد القانوني لا يجب أن يجيب عن أسئلة غير مدعومة أو غير مرتبطة بمجال المشروع.
</p>

<p>
الرفض الآمن يعد سلوكًا صحيحًا في الأسئلة خارج النطاق أو في الحالات التي لا يكون فيها السياق المسترجع موثوقًا بما يكفي.
</p>

<h3 style="color:#1F4E78;">سابعًا: تقييم ملاءمة الإجابة للصوت</h3>

<p>
بما أن المشروع يستهدف بناء مساعد صوتي، تم تقييم مدى مناسبة الإجابة للتحويل إلى صوت.
تعتمد هذه الدرجة على عوامل مثل طول الإجابة، ووضوحها، وقلة التعداد المعقد، وعدم احتوائها على تفاصيل طويلة يصعب نطقها أو سماعها.
</p>

<p>
الإجابة المناسبة للصوت يجب أن تكون مختصرة، مباشرة، واضحة، ومدعومة بالسياق القانوني أو الخدمي المناسب.
</p>

<h3 style="color:#1F4E78;">ثامنًا: قياس زمن الاستجابة</h3>

<p>
تم قياس زمن الاستجابة الكلي لكل نموذج، بما يشمل زمن الاسترجاع وزمن توليد الإجابة.
هذا المقياس مهم لأن المساعد الصوتي يحتاج إلى استجابة سريعة ومناسبة لتجربة المستخدم.
</p>

<h3 style="color:#1F4E78;">اختيار أفضل نموذج</h3>

<p>
بعد حساب المقاييس السابقة، تم اختيار أفضل نموذج باستخدام درجة مركبة تجمع بين عدة عوامل، مثل دقة الإجابة، والارتباط بالسياق، وملاءمة الإجابة للصوت، وتقليل مؤشر الهلوسة التقريبي.
</p>

<p>
بهذه الطريقة لا يتم اختيار النموذج بناءً على الدقة فقط، بل بناءً على مدى ملاءمته لبناء مساعد صوتي قانوني موثوق، قادر على تقديم إجابات صحيحة، مختصرة، ومدعومة بالسياق الرسمي.
</p>

<div style="
    background-color:#EAF3F8;
    border-right:4px solid #5B9BD5;
    padding:14px;
    margin:16px 0 0 0;
    border-radius:5px;
">
<b style="color:#1F4E78;">الخلاصة:</b>
تعتمد منهجية التقييم في هذه المرحلة على مقارنة النماذج باستخدام Benchmark داخلي مبني على أسئلة من مصادر رسمية.
وتجمع عملية التقييم بين صحة الإجابة، ودقة الاستشهاد، والارتباط بالسياق، والرفض الآمن، ومؤشر الهلوسة التقريبي، وملاءمة الإجابة للصوت، وزمن الاستجابة.
كما ترتبط هذه المقاييس أكاديميًا بمفاهيم تعلم الآلة، ومعالجة اللغة الطبيعية، واسترجاع المعلومات، وتحليل أداء الأنظمة الذكية.
وهذا يجعل المقارنة أكثر ملاءمة لطبيعة المشروع من الاعتماد على مقياس واحد فقط.
</div>

</div>
